# ATML PA3 · Task 2 — Vision-Language Model and Visual Grounding (SigLIP → MLP → Qwen3-0.6B)
**CS-6304 / EE-5102 · Danish Rasool (28100233)** · self-contained notebook (all code is written to disk by the cells below)

### Before running
1. **Settings → Accelerator: GPU T4 ×2**, **Internet: On**.
2. **Save Version → Save & Run All (Commit)** so the run continues with the browser closed and outputs are saved.
3. Optional: attach a dataset with the COCO 2014 / VQAv2 annotation JSONs (and `train2014`/`val2014` image folders)
   if a download host is unreachable — attached copies under `/kaggle/input` are used automatically.

### What happens (`AUTO`)
hub prefetch → data selection (seed 6304; COCO captions, annotation-derived presence/counting pairs, VQAv2
complementary pairs; IDs saved) + image download → label audit sheets → SigLIP feature cache (FP16, one shard per
GPU, timed separately; SigLIP then leaves GPU memory) → preflight (3 updates per stage type) → **training: GPU0 =
196-token main run, GPU1 = 49-token run, concurrently** (4,500 updates each, dev eval every 500 + stage ends,
resumable checkpoints every 250) → final evaluation under correct / complementary swap / mismatched / no image
(pair-bootstrap CIs) → report figures and tables → `results_task2.zip`.

Training pauses itself before the session budget (`BUDGET_HOURS`) runs out and the notebook ends normally, so
Kaggle keeps the outputs. If `status.json` says **paused**: Output → New Dataset, attach it to a new session, run
again — finished steps are skipped and training resumes exactly.


In [ ]:
# ============================ SETTINGS ============================
import os, time
os.environ.setdefault("PA3_SESSION_START", str(time.time()))   # the session budget starts now
BUDGET_HOURS = 11.25        # stop cleanly before Kaggle's 12 h limit
RUN_49_TOKENS = True        # optional 49-token experiment on GPU1 (concurrent with the main run)
RUN_CPU_TESTS = False       # run the CPU test suite first (~5 min)
CODE_DIR = "/kaggle/working/ATML-PA3"   # fixed: the %%writefile cells below write here
OUTPUT_ROOT = "/kaggle/working/pa3_task2_outputs"

In [ ]:
# ============================ WRITE THE CODE ============================
import os
for d in [
 "/kaggle/working/ATML-PA3/common/pa3common",
 "/kaggle/working/ATML-PA3/task_2",
 "/kaggle/working/ATML-PA3/task_2/configs",
 "/kaggle/working/ATML-PA3/task_2/scripts",
 "/kaggle/working/ATML-PA3/task_2/src/pa3vlm",
 "/kaggle/working/ATML-PA3/task_2/src/pa3vlm/data",
 "/kaggle/working/ATML-PA3/task_2/src/pa3vlm/eval",
 "/kaggle/working/ATML-PA3/task_2/src/pa3vlm/features",
 "/kaggle/working/ATML-PA3/task_2/src/pa3vlm/models",
 "/kaggle/working/ATML-PA3/task_2/src/pa3vlm/train",
 "/kaggle/working/ATML-PA3/task_2/tests"
]:
    os.makedirs(d, exist_ok=True)
print('writing 52 files under /kaggle/working/ATML-PA3')

In [ ]:
%%writefile /kaggle/working/ATML-PA3/common/pa3common/__init__.py
"""Shared, task-agnostic helpers for ATML PA3 (utils, Kaggle orchestration)."""


In [ ]:
%%writefile /kaggle/working/ATML-PA3/common/pa3common/utils.py
"""Shared helpers (task-agnostic): seeding, distributed helpers, JSON-lines logging, file helpers, memory stats."""
from __future__ import annotations

import datetime
import hashlib
import json
import os
import random
import shutil
import time
from contextlib import contextmanager
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.distributed as dist


# ----------------------------------------------------------------------------- seeding
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed % (2**32))
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def stable_int_seed(*parts: Any) -> int:
    """Deterministic 63-bit seed from arbitrary parts (independent of PYTHONHASHSEED)."""
    h = hashlib.sha256(":".join(str(p) for p in parts).encode()).hexdigest()
    return int(h[:15], 16)


def get_rng_state() -> dict:
    st = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        st["cuda"] = torch.cuda.get_rng_state()
    return st


def set_rng_state(st: dict) -> None:
    random.setstate(st["python"])
    np.random.set_state(st["numpy"])
    torch.set_rng_state(st["torch"])
    if "cuda" in st and torch.cuda.is_available():
        torch.cuda.set_rng_state(st["cuda"])


# ----------------------------------------------------------------------------- distributed
def dist_info() -> tuple[int, int, int]:
    """(rank, local_rank, world_size) from torchrun env vars (defaults to single process)."""
    return (
        int(os.environ.get("RANK", 0)),
        int(os.environ.get("LOCAL_RANK", 0)),
        int(os.environ.get("WORLD_SIZE", 1)),
    )


def init_distributed() -> tuple[int, int, int, torch.device]:
    rank, local_rank, world = dist_info()
    if torch.cuda.is_available():
        torch.cuda.set_device(local_rank)
        device = torch.device("cuda", local_rank)
        backend = "nccl"
    else:
        device = torch.device("cpu")
        backend = "gloo"
    if world > 1 and not dist.is_initialized():
        # generous timeout: rank 0 may write multi-GB checkpoints while the other rank waits at a barrier
        dist.init_process_group(backend=backend, timeout=datetime.timedelta(minutes=int(os.environ.get(
            "PA3_DIST_TIMEOUT_MIN", "60"))))
    return rank, local_rank, world, device


def is_dist() -> bool:
    return dist.is_available() and dist.is_initialized()


def barrier() -> None:
    if is_dist():
        dist.barrier()


def all_reduce_sum(t: torch.Tensor) -> torch.Tensor:
    if is_dist():
        dist.all_reduce(t, op=dist.ReduceOp.SUM)
    return t


def broadcast_int(value: int, device: torch.device, src: int = 0) -> int:
    """Make every rank adopt rank `src`'s decision (e.g. pause / checkpoint now)."""
    if not is_dist():
        return int(value)
    t = torch.tensor([int(value)], device=device, dtype=torch.long)
    dist.broadcast(t, src=src)
    return int(t.item())


def all_gather_object(obj: Any) -> list[Any]:
    if not is_dist():
        return [obj]
    out: list[Any] = [None] * dist.get_world_size()
    dist.all_gather_object(out, obj)
    return out


def cleanup_distributed() -> None:
    if is_dist():
        dist.barrier()
        dist.destroy_process_group()


# ----------------------------------------------------------------------------- logging / io
class JsonlLogger:
    def __init__(self, path: str | Path, enabled: bool = True, truncate_after: int | None = None):
        self.path = Path(path)
        self.enabled = enabled
        if enabled:
            self.path.parent.mkdir(parents=True, exist_ok=True)
            if truncate_after is not None and self.path.exists():
                # on resume: drop log rows written after the checkpoint we resume from
                rows = [json.loads(line) for line in self.path.read_text().splitlines() if line.strip()]
                rows = [r for r in rows if r.get("update", 0) <= truncate_after]
                self.path.write_text("".join(json.dumps(r) + "\n" for r in rows))

    def log(self, row: dict) -> None:
        if self.enabled:
            with open(self.path, "a") as fh:
                fh.write(json.dumps(row, default=_json_default) + "\n")


def _json_default(o: Any) -> Any:
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, torch.Tensor):
        return o.item() if o.numel() == 1 else o.tolist()
    if isinstance(o, Path):
        return str(o)
    if isinstance(o, (set, frozenset, tuple)):
        return list(o)
    if callable(o):  # e.g. functions nested inside lm-eval task configs (hellaswag)
        return f"<callable {getattr(o, '__module__', '')}.{getattr(o, '__qualname__', repr(o))}>"
    return str(o)  # logging must never crash a multi-hour run


def write_json(path: str | Path, obj: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, default=_json_default))
    os.replace(tmp, path)


def read_json(path: str | Path) -> Any:
    return json.loads(Path(path).read_text())


def dir_size_bytes(path: str | Path) -> int:
    p = Path(path)
    if p.is_file():
        return p.stat().st_size
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file())


def rmtree(path: str | Path) -> None:
    shutil.rmtree(path, ignore_errors=True)


def sha256_file(path: str | Path, chunk: int = 1 << 22) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        while b := fh.read(chunk):
            h.update(b)
    return h.hexdigest()


# ----------------------------------------------------------------------------- artifacts
def find_artifact(rel: str | Path, roots: list[str | Path] | None = None) -> Path | None:
    """Look for `rel` under the writable outputs root first, then under prior-session outputs.

    Prior Kaggle sessions are attached read-only under /kaggle/input/<dataset>/...; the env var
    PA3_PRIOR_ROOTS (os.pathsep-separated) lists those roots (the notebook sets it).
    """
    rel = Path(rel)
    if rel.is_absolute():
        return rel if rel.exists() else None
    cands: list[Path] = [Path(r) for r in (roots or [])]
    cands.append(Path(os.environ.get("PA3_OUTPUT_ROOT", "outputs")))
    cands += [Path(p) for p in os.environ.get("PA3_PRIOR_ROOTS", "").split(os.pathsep) if p]
    for root in cands:
        p = root / rel
        if p.exists():
            return p
    return None


def output_root() -> Path:
    return Path(os.environ.get("PA3_OUTPUT_ROOT", "outputs"))


# ----------------------------------------------------------------------------- memory / time
def host_mem_stats() -> dict:
    """Process RSS / peak RSS and machine MemAvailable in GiB (Linux /proc), for host-OOM diagnosis."""
    out = {}
    try:
        for line in open("/proc/self/status"):
            if line.startswith(("VmRSS:", "VmHWM:")):
                out["rss_gib" if line.startswith("VmRSS") else "peak_rss_gib"] = int(line.split()[1]) / 2**20
        for line in open("/proc/meminfo"):
            if line.startswith(("MemAvailable:", "MemTotal:")):
                out[line.split(":")[0].lower() + "_gib"] = int(line.split()[1]) / 2**20
    except OSError:
        pass
    return out


def gpu_mem_stats(device: torch.device) -> dict:
    if device.type != "cuda":
        return {}
    return {
        "peak_alloc_gib": torch.cuda.max_memory_allocated(device) / 2**30,
        "peak_reserved_gib": torch.cuda.max_memory_reserved(device) / 2**30,
        "total_gib": torch.cuda.get_device_properties(device).total_memory / 2**30,
    }


@contextmanager
def timer(store: dict, key: str, sync_device: torch.device | None = None):
    if sync_device is not None and sync_device.type == "cuda":
        torch.cuda.synchronize(sync_device)
    t0 = time.perf_counter()
    yield
    if sync_device is not None and sync_device.type == "cuda":
        torch.cuda.synchronize(sync_device)
    store[key] = store.get(key, 0.0) + time.perf_counter() - t0


def count_params(model: torch.nn.Module, trainable_only: bool = False) -> int:
    seen: set[int] = set()
    n = 0
    for p in model.parameters():  # .parameters() already de-duplicates tied weights
        if id(p) in seen or (trainable_only and not p.requires_grad):
            continue
        seen.add(id(p))
        n += p.numel()
    return n


In [ ]:
%%writefile /kaggle/working/ATML-PA3/common/pa3common/kaggle.py
"""Task-agnostic Kaggle helpers: subprocess streaming, GPU job queue, session deadline, disk/scratch,
prior-session discovery and retries.  Used by task orchestrators (e.g. task_1/src/pa3/orchestrate.py)."""
from __future__ import annotations

import os
import shutil
import subprocess
import sys
import time
from pathlib import Path
from typing import Callable

NOISE = ("Loading weights", "it/s]", "s/it]", "Fetching", "Downloading", "Map:", "Generating ",
         "Requesting API", "Filter:", "Resolving data files")


# ------------------------------------------------------------------ session deadline
class Deadline:
    """Wall-clock budget for one Kaggle session (hard limit 12 h; outputs are saved only if the
    notebook finishes, so every step must stop before the deadline)."""

    def __init__(self, budget_hours: float, start: float | None = None):
        self.start = start or float(os.environ.get("PA3_SESSION_START", time.time()))
        self.end = self.start + budget_hours * 3600
        os.environ["PA3_SESSION_START"] = str(self.start)
        os.environ["PA3_DEADLINE_UNIX"] = str(self.end)

    def remaining(self) -> float:
        return self.end - time.time()

    def fits(self, seconds: float, margin: float = 600) -> bool:
        return self.remaining() > seconds + margin

    def __str__(self) -> str:
        return f"{self.remaining() / 3600:.2f} h left of session budget"


class StagePaused(RuntimeError):
    """Raised when the session budget is exhausted; rerun the notebook to continue."""


# ------------------------------------------------------------------ processes
def sh(cmd: list, log: Path, env: dict | None = None, timeout: float | None = None, echo: bool = True,
       cwd: str | Path | None = None) -> int:
    log = Path(log)
    log.parent.mkdir(parents=True, exist_ok=True)
    print(f"\n$ {' '.join(map(str, cmd))}\n  (log: {log})", flush=True)
    t0 = time.time()
    with open(log, "a") as fh:
        fh.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} $ {' '.join(map(str, cmd))}\n")
        p = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             env={**os.environ, **(env or {})}, cwd=str(cwd) if cwd else None, bufsize=1)
        try:
            for line in p.stdout:
                fh.write(line)
                fh.flush()
                if echo and not any(s in line for s in NOISE):
                    print(line, end="", flush=True)
                if timeout and time.time() - t0 > timeout:
                    p.kill()
                    print(f"!! timeout after {timeout:.0f}s", flush=True)
                    return -9
            return p.wait()
        finally:
            if p.poll() is None:
                p.kill()


def n_gpus() -> int:
    try:
        import torch

        return torch.cuda.device_count()
    except Exception:
        return 0


def run_on_gpus(jobs: list[tuple[str, list, Path]], env_extra: dict | None = None,
                poll: float = 10.0, kill_at: float | None = None) -> dict[str, int]:
    """Run independent single-GPU jobs concurrently, one per visible GPU (queue). Returns rc per job.
    Jobs still running at unix time `kill_at` are killed (rc -9) so the session can end cleanly."""
    slots = list(range(max(1, n_gpus())))
    todo = list(jobs)
    running: dict[int, tuple[str, subprocess.Popen, object, Path]] = {}
    rcs: dict[str, int] = {}
    while todo or running:
        while todo and len(running) < len(slots):
            gpu = next(s for s in slots if s not in running)
            name, cmd, log = todo.pop(0)
            log = Path(log)
            log.parent.mkdir(parents=True, exist_ok=True)
            fh = open(log, "a")
            fh.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} GPU{gpu} $ {' '.join(map(str, cmd))}\n")
            fh.flush()
            print(f"[gpu-queue] {name} -> GPU{gpu} (log {log})", flush=True)
            p = subprocess.Popen(list(map(str, cmd)), stdout=fh, stderr=subprocess.STDOUT,
                                 env={**os.environ, **(env_extra or {}), "CUDA_VISIBLE_DEVICES": str(gpu)})
            running[gpu] = (name, p, fh, log)
        time.sleep(poll)
        if kill_at is not None and time.time() > kill_at:
            for gpu, (name, p, fh, log) in list(running.items()):
                print(f"[gpu-queue] {name}: session deadline reached -> killing", flush=True)
                p.kill()
            todo = []
        for gpu in list(running):
            name, p, fh, log = running[gpu]
            if p.poll() is not None:
                fh.close()
                rcs[name] = p.returncode
                tail = [l for l in log.read_text(errors="ignore").splitlines() if l.startswith("[")][-5:]
                print(f"[gpu-queue] {name} finished rc={p.returncode}\n  " + "\n  ".join(tail), flush=True)
                del running[gpu]
    return rcs


def log_has(log: Path, *needles: str) -> bool:
    t = Path(log).read_text(errors="ignore") if Path(log).exists() else ""
    return any(n in t for n in needles)


def is_oom(log: Path) -> bool:
    return log_has(log, "OutOfMemoryError", "CUDA out of memory")


def is_host_oom(log: Path) -> bool:
    """All ranks SIGKILLed without a Python traceback = the kernel's OOM killer (host RAM)."""
    return log_has(log, "Signal 9 (SIGKILL)", "exitcode: -9") and not is_oom(log)


def retry(fn: Callable, tries: int = 4, wait: float = 20.0, what: str = ""):
    for i in range(tries):
        try:
            return fn()
        except Exception as ex:  # noqa: BLE001
            if i == tries - 1:
                raise
            print(f"[retry] {what} failed ({type(ex).__name__}: {str(ex)[:200]}); retrying in {wait:.0f}s",
                  flush=True)
            time.sleep(wait * (i + 1))


def pip_install(pkgs: list[str], tries: int = 3) -> None:
    def go():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed", *pkgs],
                       check=True)
    retry(go, tries, 15, "pip install")


# ------------------------------------------------------------------ disk / paths
def scratch_root(name: str = "pa3_scratch") -> Path:
    """Large, non-persisted disk for rolling recovery checkpoints (keeps /kaggle/working < 20 GB)."""
    env = os.environ.get("PA3_SCRATCH")
    cands = [Path(env)] if env else [Path("/kaggle/temp") / name, Path("/tmp") / name]
    for c in cands:
        if _is_tmpfs(c):
            print(f"[scratch] {c} is RAM-backed (tmpfs); skipping", flush=True)
            continue
        try:
            c.mkdir(parents=True, exist_ok=True)
            probe = c / ".w"
            probe.write_text("ok")
            probe.unlink()
            os.environ["PA3_SCRATCH"] = str(c)
            return c
        except OSError:
            continue
    raise RuntimeError("no writable scratch directory")


def _is_tmpfs(path: Path) -> bool:
    """True if `path` would live on a RAM-backed filesystem (writing GB-sized checkpoints there eats RAM)."""
    try:
        best, fstype = "", ""
        target = str(Path(path).resolve())
        for line in open("/proc/mounts"):
            parts = line.split()
            mnt, fs = parts[1], parts[2]
            if (target == mnt or target.startswith(mnt.rstrip("/") + "/")) and len(mnt) > len(best):
                best, fstype = mnt, fs
        return fstype in ("tmpfs", "ramfs")
    except OSError:
        return False


def free_gib(path: str | Path) -> float:
    return shutil.disk_usage(str(path)).free / 2**30


def tree_gib(root: str | Path) -> float:
    root = Path(root)
    if not root.exists():
        return 0.0
    return sum(f.stat().st_size for f in root.rglob("*") if f.is_file()) / 2**30


def disk_report(working: str | Path = "/kaggle/working", limit_gib: float = 19.0) -> float:
    w = Path(working) if Path(working).exists() else Path.cwd()
    used = tree_gib(w)
    msg = f"[disk] {w}: {used:.2f} GiB used (persisted-output limit ~20 GiB)"
    if os.environ.get("PA3_SCRATCH"):
        s = os.environ["PA3_SCRATCH"]
        msg += f" | scratch {s}: {tree_gib(s):.2f} GiB used, {free_gib(s):.0f} GiB free"
    print(msg, flush=True)
    if used > limit_gib:
        print("!! WARNING: /kaggle/working is close to the 20 GiB limit", flush=True)
    return used


def discover_prior_roots(dirname: str, base: str = "/kaggle/input", max_depth: int = 6) -> list[str]:
    """Earlier sessions' outputs attached as inputs: every directory called `dirname` under /kaggle/input."""
    b = Path(base)
    if not b.exists():
        return []
    return [str(p) for p in sorted(b.glob(f"**/{dirname}")) if p.is_dir() and len(p.relative_to(b).parts) <= max_depth]


def copytree_merge(src: Path, dst: Path, skip: tuple[str, ...] = ()) -> None:
    for f in Path(src).rglob("*"):
        rel = f.relative_to(src)
        if any(part in skip for part in rel.parts):
            continue
        t = Path(dst) / rel
        if f.is_dir():
            t.mkdir(parents=True, exist_ok=True)
        elif not t.exists():
            t.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, t)


In [ ]:
%%writefile /kaggle/working/ATML-PA3/common/pa3common/lora.py
"""Shared LoRA module (plain LoRA, the Task 1 LoRALinear without the OPLoRA projection).

    y = W0 x + (alpha / r) * B A dropout(x)

Initialisation matches PEFT's default (`init_lora_weights=True`): A ~ U(-1/sqrt(d_in), 1/sqrt(d_in))
(= kaiming_uniform(a=sqrt 5)), B = 0, so injecting adapters leaves the model function unchanged.
A is drawn from a CPU generator seeded by (seed, "lora_A", module name): identical factors regardless
of module construction order or device.  Trainable factors are FP32; the frozen base may be FP16.
Task 1 keeps its own copy (task_1/src/pa3/models/lora.py); this one is for Task 2 onwards.
"""
from __future__ import annotations

import math

import torch
import torch.nn.functional as F
from torch import nn

from .utils import stable_int_seed


class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r: int, alpha: float, dropout: float, init_seed: int):
        super().__init__()
        assert base.bias is None, "Qwen3 attention projections have no bias"
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)
        dout, din = base.weight.shape
        self.r, self.alpha, self.scale = r, float(alpha), float(alpha) / r
        g = torch.Generator().manual_seed(init_seed)
        bound = 1.0 / math.sqrt(din)
        A = (torch.rand(r, din, generator=g, dtype=torch.float32) * 2 - 1) * bound
        dev = base.weight.device
        self.lora_A = nn.Parameter(A.to(dev))
        self.lora_B = nn.Parameter(torch.zeros(dout, r, dtype=torch.float32, device=dev))
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.base(x)
        h = self.dropout(x)
        if not torch.is_autocast_enabled(x.device.type):
            h = h.to(self.lora_A.dtype)
        out = F.linear(F.linear(h, self.lora_A), self.lora_B)
        return y + (out * self.scale).to(y.dtype)

    def extra_repr(self) -> str:
        return f"r={self.r}, alpha={self.alpha}"


def _set_submodule(root: nn.Module, name: str, new: nn.Module) -> None:
    parent_name, leaf = name.rsplit(".", 1)
    setattr(root.get_submodule(parent_name), leaf, new)


def inject_lora(model: nn.Module, r: int, alpha: float, dropout: float, seed: int,
                targets: tuple[str, ...] | list[str]) -> dict[str, LoRALinear]:
    """Wrap every nn.Linear whose leaf name is in `targets` (base weights frozen)."""
    names = [n for n, m in model.named_modules()
             if isinstance(m, nn.Linear) and n.rsplit(".", 1)[-1] in set(targets)]
    assert names, "no target modules found"
    out: dict[str, LoRALinear] = {}
    for n in names:
        mod = LoRALinear(model.get_submodule(n), r, alpha, dropout, stable_int_seed(seed, "lora_A", n))
        _set_submodule(model, n, mod)
        out[n] = mod
    return out


def lora_modules(model: nn.Module) -> dict[str, LoRALinear]:
    return {n: m for n, m in model.named_modules() if isinstance(m, LoRALinear)}


def lora_state(model: nn.Module) -> dict[str, torch.Tensor]:
    return {f"{n}.{k}": getattr(m, k).detach().float().cpu().clone()
            for n, m in lora_modules(model).items() for k in ("lora_A", "lora_B")}


def load_lora_state(model: nn.Module, state: dict[str, torch.Tensor]) -> None:
    mods = lora_modules(model)
    assert len(state) == 2 * len(mods), (len(state), len(mods))
    for n, m in mods.items():
        m.lora_A.data.copy_(state[f"{n}.lora_A"].to(m.lora_A.device))
        m.lora_B.data.copy_(state[f"{n}.lora_B"].to(m.lora_B.device))


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/configs/base.yaml
# Shared recipe (PA3 Task 2). Relative paths are under $PA3_OUTPUT_ROOT; feature/image caches under $PA3_SCRATCH.
seed: 6304
model_id: Qwen/Qwen3-0.6B
vision_id: google/siglip-base-patch16-224
data_dir: data
features_dir: features            # under scratch
run_name: vt196
pool: 1                           # 1 -> 196 visual tokens; 2 -> 49 (2x2 average of the 14x14 grid)
dtype: float16                    # frozen backbone dtype on GPU (float32 on CPU tests)
micro_bs: 8
grad_accum: 2                     # effective batch 16
stages:
  - {name: caption, updates: 2250, lr_connector: 1.0e-3, lr_lora: null, mix: null}
  - {name: instruct, updates: 1500, lr_connector: 1.0e-4, lr_lora: 2.0e-5,
     mix: {presence: 0.5, counting: 0.25, caption: 0.25}}
  - {name: vqa, updates: 750, lr_connector: 5.0e-5, lr_lora: 1.0e-5,
     mix: {vqa: 0.7, presence: 0.1, counting: 0.1, caption: 0.1}}
caption_passes: 2
warmup_frac: 0.05                 # ceil(5% of stage updates) -> 113 / 75 / 38
final_lr_frac: 0.1                # cosine to 10% of peak at the stage's last update
weight_decay: 0.01                # weights only (no biases)
betas: [0.9, 0.999]
eps: 1.0e-8
clip_norm: 1.0
init_scale: 65536.0
grad_checkpointing: true
attn_impl: sdpa
lora: {r: 16, alpha: 32, dropout: 0.05, targets: [q_proj, k_proj, v_proj, o_proj]}
caps: {question: 80, caption: 64, answer: 16}
calib_images: 1024
dev_every: 500
ckpt_every: 250
keep_resumable: 2
eval_bs: 8
max_new_tokens: 16
bootstrap: 4000
deadline_margin_min: 35           # training pauses when less than this is left in the session budget
stop_after_update: null           # tests only: pause after this global update


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/configs/data.yaml
# Data selection (seed 6304). Sizes from the PA3 manual.
seed: 6304
caption_train_images: 6000
captions_per_train_image: 3
caption_dev_images: 96
vqa_train: {"yes/no": 640, "number": 256, "other": 640}
vqa_dev: {"yes/no": 12, "number": 8, "other": 12}
vqa_test: {"yes/no": 36, "number": 24, "other": 36}
vqa_min_agree: 8
grounding_dev: {presence: 12, counting: 12}
grounding_test: {presence: 32, counting: 32}
min_area: 1024                    # 32 x 32 px (COCO small/medium boundary)
max_count: 4
pool_cap_per_category: 400        # presence entries / counting pairs per category in the train pools


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/configs/vt196.yaml
run_name: vt196
pool: 1


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/configs/vt49.yaml
run_name: vt49
pool: 2


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/__init__.py
"""PA3 Task 2: SigLIP -> MLP connector -> Qwen3-0.6B vision-language model and grounding evaluation."""


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/config.py
"""YAML config loading: configs/base.yaml + a run config + `key=value` overrides (YAML-parsed values)."""
from __future__ import annotations

from pathlib import Path

import yaml

TASK = Path(__file__).resolve().parents[2]
CONFIGS = TASK / "configs"


def _merge(a: dict, b: dict) -> dict:
    out = dict(a)
    for k, v in b.items():
        out[k] = _merge(out[k], v) if isinstance(v, dict) and isinstance(out.get(k), dict) else v
    return out


def load_config(path: str | Path | None = None, sets: list[str] | None = None, base: str | Path | None = None) -> dict:
    cfg = yaml.safe_load(Path(base or CONFIGS / "base.yaml").read_text())
    if path:
        p = Path(path)
        if not p.exists():
            p = CONFIGS / f"{path}.yaml"
        cfg = _merge(cfg, yaml.safe_load(p.read_text()) or {})
    for s in sets or []:
        k, v = s.split("=", 1)
        node = cfg
        keys = k.split(".")
        for kk in keys[:-1]:
            node = node.setdefault(kk, {})
        node[keys[-1]] = yaml.safe_load(v)
    return cfg


def load_data_config(path: str | Path | None = None, sets: list[str] | None = None) -> dict:
    return load_config(None, sets, base=path or CONFIGS / "data.yaml")


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/data/__init__.py



In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/data/bundle.py
"""Manifests + lazy, cached tokenisation of training items and evaluation suites."""
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

from .prompts import ChatFormat, Encoded, encode_caption, encode_pair

POOLS = {"presence": "presence_pool", "counting": "counting_pool", "vqa": "vqa_train"}


class Bundle:
    def __init__(self, data_dir: str | Path, fmt: ChatFormat):
        self.dir = Path(data_dir)
        self.fmt = fmt
        self._m: dict[str, list] = {}
        self._cap: dict[int, Encoded] = {}
        self._pair: dict[tuple[str, int], list[Encoded]] = {}

    def manifest(self, name: str) -> list:
        if name not in self._m:
            self._m[name] = json.loads((self.dir / f"{name}.json").read_text())
        return self._m[name]

    @property
    def n_captions(self) -> int:
        return len(self.manifest("captions_train"))

    def pool_sizes(self) -> dict[str, int]:
        return {k: len(self.manifest(v)) for k, v in POOLS.items()}

    def caption(self, i: int) -> Encoded:
        if i not in self._cap:
            self._cap[i] = encode_caption(self.fmt, self.manifest("captions_train")[i])
        return self._cap[i]

    def pair(self, pool: str, j: int) -> list[Encoded]:
        key = (pool, j)
        if key not in self._pair:
            self._pair[key] = encode_pair(self.fmt, self.manifest(POOLS[pool])[j])
        return self._pair[key]

    def item(self, it: tuple) -> Encoded:
        return self.caption(it[1]) if it[0] == "caption" else self.pair(it[0], it[1])[it[2]]

    def stage1_token_counts(self) -> Counter:
        c: Counter = Counter()
        for i in range(self.n_captions):
            e = self.caption(i)
            c.update(e.prefix)
            c.update(e.suffix)
            c.update(e.target)
        return c

    def captions_dev(self) -> list[Encoded]:
        return [encode_caption(self.fmt, r) for r in self.manifest("captions_dev")]

    def suite(self, name: str) -> tuple[list[dict], list[Encoded]]:
        """Pairs and their examples in (pair, slot) order."""
        pairs = self.manifest(name)
        encs = [e for p in pairs for e in encode_pair(self.fmt, p)]
        return pairs, encs


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/data/coco.py
"""COCO 2014 helpers: category wording table, annotation indexing, caption mention check."""
from __future__ import annotations

import re
from collections import defaultdict

# COCO name -> (presence phrase incl. article, counting plural, extra caption synonyms)
# Display names avoid ambiguity ("computer mouse", "remote control"); synonyms are used only to reject
# presence negatives whose captions mention the object (COCO annotation omissions).
CATEGORIES: dict[str, tuple[str, str, tuple[str, ...]]] = {
    "person": ("a person", "people", ("person", "people", "persons", "man", "men", "woman", "women", "boy", "boys",
               "girl", "girls", "child", "children", "kid", "kids", "player", "players", "guy", "guys", "lady",
               "ladies", "baby", "babies", "skier", "skiers", "surfer", "surfers", "skateboarder", "snowboarder",
               "rider", "crowd", "someone", "toddler", "adult", "adults", "pitcher", "batter", "catcher", "umpire")),
    "bicycle": ("a bicycle", "bicycles", ("bicycle", "bicycles", "bike", "bikes", "cyclist")),
    "car": ("a car", "cars", ("car", "cars", "taxi", "suv", "sedan")),
    "motorcycle": ("a motorcycle", "motorcycles", ("motorcycle", "motorcycles", "motorbike", "motor bike", "scooter",
                   "moped", "dirt bike")),
    "airplane": ("an airplane", "airplanes", ("airplane", "airplanes", "plane", "planes", "jet", "jets", "aircraft",
                 "airliner")),
    "bus": ("a bus", "buses", ("bus", "buses", "busses")),
    "train": ("a train", "trains", ("train", "trains", "locomotive", "tram", "trolley", "subway")),
    "truck": ("a truck", "trucks", ("truck", "trucks", "pickup", "lorry")),
    "boat": ("a boat", "boats", ("boat", "boats", "ship", "ships", "canoe", "kayak", "sailboat", "yacht", "ferry")),
    "traffic light": ("a traffic light", "traffic lights", ("traffic light", "traffic lights", "stoplight",
                      "stop light", "traffic signal")),
    "fire hydrant": ("a fire hydrant", "fire hydrants", ("hydrant", "hydrants")),
    "stop sign": ("a stop sign", "stop signs", ("stop sign", "stop signs")),
    "parking meter": ("a parking meter", "parking meters", ("parking meter", "parking meters", "meter", "meters")),
    "bench": ("a bench", "benches", ("bench", "benches")),
    "bird": ("a bird", "birds", ("bird", "birds", "duck", "ducks", "pigeon", "pigeons", "seagull", "gull", "goose",
             "geese", "parrot", "swan", "owl", "eagle")),
    "cat": ("a cat", "cats", ("cat", "cats", "kitten", "kitty", "kittens")),
    "dog": ("a dog", "dogs", ("dog", "dogs", "puppy", "puppies", "pup")),
    "horse": ("a horse", "horses", ("horse", "horses", "pony", "foal")),
    "sheep": ("a sheep", "sheep", ("sheep", "lamb", "lambs", "ram")),
    "cow": ("a cow", "cows", ("cow", "cows", "cattle", "bull", "bulls", "calf", "ox", "oxen")),
    "elephant": ("an elephant", "elephants", ("elephant", "elephants")),
    "bear": ("a bear", "bears", ("bear", "bears")),
    "zebra": ("a zebra", "zebras", ("zebra", "zebras")),
    "giraffe": ("a giraffe", "giraffes", ("giraffe", "giraffes")),
    "backpack": ("a backpack", "backpacks", ("backpack", "backpacks", "back pack", "rucksack")),
    "umbrella": ("an umbrella", "umbrellas", ("umbrella", "umbrellas", "parasol")),
    "handbag": ("a handbag", "handbags", ("handbag", "handbags", "purse", "purses", "bag", "bags")),
    "tie": ("a tie", "ties", ("tie", "ties", "necktie", "bow tie")),
    "suitcase": ("a suitcase", "suitcases", ("suitcase", "suitcases", "luggage", "baggage")),
    "frisbee": ("a frisbee", "frisbees", ("frisbee", "frisbees", "disc")),
    "skis": ("skis", "pairs of skis", ("ski", "skis", "skiing", "skier", "skiers")),
    "snowboard": ("a snowboard", "snowboards", ("snowboard", "snowboards", "snowboarding", "snowboarder")),
    "sports ball": ("a ball", "balls", ("ball", "balls", "soccer ball", "football", "baseball", "tennis ball",
                    "basketball", "volleyball")),
    "kite": ("a kite", "kites", ("kite", "kites")),
    "baseball bat": ("a baseball bat", "baseball bats", ("bat", "bats")),
    "baseball glove": ("a baseball glove", "baseball gloves", ("glove", "gloves", "mitt", "mitts")),
    "skateboard": ("a skateboard", "skateboards", ("skateboard", "skateboards", "skateboarding", "skateboarder")),
    "surfboard": ("a surfboard", "surfboards", ("surfboard", "surfboards", "surf board", "surfing", "surfer",
                  "board")),
    "tennis racket": ("a tennis racket", "tennis rackets", ("racket", "rackets", "racquet", "racquets")),
    "bottle": ("a bottle", "bottles", ("bottle", "bottles")),
    "wine glass": ("a wine glass", "wine glasses", ("wine glass", "wine glasses", "glass of wine", "wineglass")),
    "cup": ("a cup", "cups", ("cup", "cups", "mug", "mugs", "coffee")),
    "fork": ("a fork", "forks", ("fork", "forks")),
    "knife": ("a knife", "knives", ("knife", "knives")),
    "spoon": ("a spoon", "spoons", ("spoon", "spoons")),
    "bowl": ("a bowl", "bowls", ("bowl", "bowls")),
    "banana": ("a banana", "bananas", ("banana", "bananas")),
    "apple": ("an apple", "apples", ("apple", "apples")),
    "sandwich": ("a sandwich", "sandwiches", ("sandwich", "sandwiches", "burger", "hamburger", "sub")),
    "orange": ("an orange", "oranges", ("orange", "oranges", "citrus", "tangerine")),
    "broccoli": ("broccoli", "pieces of broccoli", ("broccoli",)),
    "carrot": ("a carrot", "carrots", ("carrot", "carrots")),
    "hot dog": ("a hot dog", "hot dogs", ("hot dog", "hot dogs", "hotdog", "hotdogs")),
    "pizza": ("a pizza", "pizzas", ("pizza", "pizzas")),
    "donut": ("a donut", "donuts", ("donut", "donuts", "doughnut", "doughnuts")),
    "cake": ("a cake", "cakes", ("cake", "cakes", "cupcake", "cupcakes")),
    "chair": ("a chair", "chairs", ("chair", "chairs", "seat", "seats", "stool")),
    "couch": ("a couch", "couches", ("couch", "couches", "sofa", "sofas", "loveseat")),
    "potted plant": ("a potted plant", "potted plants", ("plant", "plants", "flower", "flowers", "pot", "potted")),
    "bed": ("a bed", "beds", ("bed", "beds", "bedroom")),
    "dining table": ("a dining table", "dining tables", ("table", "tables", "desk")),
    "toilet": ("a toilet", "toilets", ("toilet", "toilets", "urinal")),
    "tv": ("a TV", "TVs", ("tv", "tvs", "television", "televisions", "monitor", "screen")),
    "laptop": ("a laptop", "laptops", ("laptop", "laptops", "computer", "computers", "notebook")),
    "mouse": ("a computer mouse", "computer mice", ("mouse", "mice")),
    "remote": ("a remote control", "remote controls", ("remote", "remotes", "controller", "controllers")),
    "keyboard": ("a keyboard", "keyboards", ("keyboard", "keyboards")),
    "cell phone": ("a cell phone", "cell phones", ("phone", "phones", "cellphone", "cellphones", "smartphone",
                   "mobile")),
    "microwave": ("a microwave", "microwaves", ("microwave", "microwaves")),
    "oven": ("an oven", "ovens", ("oven", "ovens", "stove", "stoves")),
    "toaster": ("a toaster", "toasters", ("toaster", "toasters")),
    "sink": ("a sink", "sinks", ("sink", "sinks")),
    "refrigerator": ("a refrigerator", "refrigerators", ("refrigerator", "refrigerators", "fridge", "fridges")),
    "book": ("a book", "books", ("book", "books", "bookshelf", "novel")),
    "clock": ("a clock", "clocks", ("clock", "clocks")),
    "vase": ("a vase", "vases", ("vase", "vases")),
    "scissors": ("scissors", "pairs of scissors", ("scissor", "scissors")),
    "teddy bear": ("a teddy bear", "teddy bears", ("teddy", "teddies", "stuffed animal", "stuffed bear",
                   "stuffed animals")),
    "hair drier": ("a hair dryer", "hair dryers", ("hair dryer", "hair drier", "blow dryer", "hairdryer")),
    "toothbrush": ("a toothbrush", "toothbrushes", ("toothbrush", "toothbrushes", "tooth brush")),
}
# Categories whose instance counts are ambiguous (pairs, bunches, uncountable, stacks): no counting questions.
NO_COUNTING = {"skis", "banana", "broccoli", "carrot", "book", "scissors"}


def presence_question(cat: str) -> str:
    return f"Does this image contain {CATEGORIES[cat][0]}?"


def counting_question(cat: str) -> str:
    return f"How many {CATEGORIES[cat][1]} can you see?"


def _patterns(cat: str) -> re.Pattern:
    words = set(CATEGORIES[cat][2]) | {cat}
    alt = "|".join(sorted((re.escape(w) for w in words), key=len, reverse=True))
    return re.compile(rf"\b(?:{alt})\b", re.IGNORECASE)


_PAT_CACHE: dict[str, re.Pattern] = {}


def captions_mention(cat: str, captions: list[str]) -> bool:
    if cat not in _PAT_CACHE:
        _PAT_CACHE[cat] = _patterns(cat)
    pat = _PAT_CACHE[cat]
    return any(pat.search(c) for c in captions)


def index_captions(caps_json: dict) -> dict[int, list[dict]]:
    """image_id -> caption annotations sorted by annotation id."""
    out: dict[int, list[dict]] = defaultdict(list)
    for a in caps_json["annotations"]:
        out[a["image_id"]].append({"id": a["id"], "caption": a["caption"]})
    for v in out.values():
        v.sort(key=lambda r: r["id"])
    return dict(out)


def index_instances(inst_json: dict) -> tuple[dict[int, str], dict[int, dict[str, dict]]]:
    """Returns (category_id -> name, image_id -> {name: {n, n_crowd, min_area, max_area}}).

    Images without any instance annotation map to {} (all COCO images are listed)."""
    cats = {c["id"]: c["name"] for c in inst_json["categories"]}
    assert set(cats.values()) == set(CATEGORIES), "unexpected COCO category set"
    stats: dict[int, dict[str, dict]] = {im["id"]: {} for im in inst_json["images"]}
    for a in inst_json["annotations"]:
        s = stats.setdefault(a["image_id"], {}).setdefault(
            cats[a["category_id"]], {"n": 0, "n_crowd": 0, "min_area": float("inf"), "max_area": 0.0})
        if a.get("iscrowd", 0):
            s["n_crowd"] += 1
        else:
            s["n"] += 1
            s["min_area"] = min(s["min_area"], float(a["area"]))
            s["max_area"] = max(s["max_area"], float(a["area"]))
    return cats, stats


def presence_positive_ok(st: dict[str, dict], cat: str, min_area: float) -> bool:
    s = st.get(cat)
    return bool(s) and s["n"] >= 1 and s["max_area"] >= min_area


def presence_negative_ok(st: dict[str, dict], cat: str, captions: list[str]) -> bool:
    return cat not in st and len(st) > 0 and not captions_mention(cat, captions)


def count_ok(st: dict[str, dict], cat: str, min_area: float, max_count: int) -> int | None:
    """Unambiguous count in 1..max_count, else None."""
    if cat in NO_COUNTING:
        return None
    s = st.get(cat)
    if not s or s["n_crowd"] > 0 or not (1 <= s["n"] <= max_count) or s["min_area"] < min_area:
        return None
    return s["n"]


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/data/collate.py
"""Batch construction: token ids with visual slots, attention mask, positions, labels.

Row layout: prefix | n_vis visual slots | suffix | target   (the visual slots are dropped for no-image rows).
- attention_mask: 1 on real tokens (text and visual), 0 on padding; the model applies causal masking.
- position_ids  : cumsum(attention_mask) - 1  -> contiguous 1-D RoPE positions over real tokens, for either
                  padding side; rebuilt automatically when the visual slots are removed.
- labels        : token id at target positions (answer/caption content + <|im_end|>), -100 elsewhere
                  (prefix, visual slots, suffix/prompt, padding).  Labels are *unshifted*; the loss shifts.
- term          : True at the termination-token label positions (excluded from answer NLL).
"""
from __future__ import annotations

from dataclasses import dataclass

import torch

from .prompts import Encoded


@dataclass
class Batch:
    ids: torch.Tensor
    vis: torch.Tensor
    attn: torch.Tensor
    pos: torch.Tensor
    labels: torch.Tensor
    term: torch.Tensor
    feats: torch.Tensor | None     # [n_rows_with_image, n_vis, 768], rows in batch order
    images: list

    def to(self, device: torch.device) -> "Batch":
        mv = {k: getattr(self, k).to(device, non_blocking=True) for k in ("ids", "vis", "attn", "pos", "labels", "term")}
        f = self.feats.to(device, non_blocking=True) if self.feats is not None else None
        return Batch(**mv, feats=f, images=self.images)


def collate(encs: list[Encoded], images: list[str | None], get_feats, n_vis: int, pad_id: int,
            side: str = "right", with_target: bool = True) -> Batch:
    assert len(encs) == len(images) and side in ("left", "right")
    rows = []
    for e, img in zip(encs, images):
        nv = n_vis if img is not None else 0
        tgt = e.target if with_target else []
        ids = list(e.prefix) + [pad_id] * nv + list(e.suffix) + list(tgt)
        vis = [False] * len(e.prefix) + [True] * nv + [False] * (len(e.suffix) + len(tgt))
        lab = [-100] * (len(ids) - len(tgt)) + list(tgt)
        term = [False] * (len(ids) - len(tgt)) + [False] * (len(tgt) - e.n_term) + [True] * min(e.n_term, len(tgt))
        rows.append((ids, vis, lab, term))
    L = max(len(r[0]) for r in rows)
    B = len(rows)
    ids = torch.full((B, L), pad_id, dtype=torch.long)
    vis = torch.zeros((B, L), dtype=torch.bool)
    attn = torch.zeros((B, L), dtype=torch.long)
    lab = torch.full((B, L), -100, dtype=torch.long)
    term = torch.zeros((B, L), dtype=torch.bool)
    for b, (i, v, l, t) in enumerate(rows):
        n = len(i)
        sl = slice(0, n) if side == "right" else slice(L - n, L)
        ids[b, sl] = torch.tensor(i)
        vis[b, sl] = torch.tensor(v)
        attn[b, sl] = 1
        lab[b, sl] = torch.tensor(l)
        term[b, sl] = torch.tensor(t)
    pos = (attn.cumsum(-1) - 1).clamp_min(0)
    keys = [k for k in images if k is not None]
    feats = get_feats(keys) if keys else None
    if feats is not None:
        assert feats.shape[:2] == (len(keys), n_vis), (feats.shape, n_vis)
    return Batch(ids, vis, attn, pos, lab, term, feats, list(images))


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/data/download.py
"""COCO 2014 / VQAv2 assets: use copies attached under /kaggle/input when present, else download
from the official hosts (with retries).  Only the images that are actually selected are fetched."""
from __future__ import annotations

import io
import os
import re
import shutil
import time
import urllib.request
import zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

COCO_ANN_ZIP = "http://images.cocodataset.org/annotations/annotations_trainval2014.zip"
VQA_BASE = "https://cvmlp.s3.amazonaws.com/vqa/mscoco/vqa"
FILES = {
    # name of the json -> zip url that contains it
    "captions_train2014.json": COCO_ANN_ZIP,
    "captions_val2014.json": COCO_ANN_ZIP,
    "instances_train2014.json": COCO_ANN_ZIP,
    "instances_val2014.json": COCO_ANN_ZIP,
    "v2_OpenEnded_mscoco_train2014_questions.json": f"{VQA_BASE}/v2_Questions_Train_mscoco.zip",
    "v2_OpenEnded_mscoco_val2014_questions.json": f"{VQA_BASE}/v2_Questions_Val_mscoco.zip",
    "v2_mscoco_train2014_annotations.json": f"{VQA_BASE}/v2_Annotations_Train_mscoco.zip",
    "v2_mscoco_val2014_annotations.json": f"{VQA_BASE}/v2_Annotations_Val_mscoco.zip",
    "v2_mscoco_train2014_complementary_pairs.json": f"{VQA_BASE}/v2_Complementary_Pairs_Train_mscoco.zip",
    "v2_mscoco_val2014_complementary_pairs.json": f"{VQA_BASE}/v2_Complementary_Pairs_Val_mscoco.zip",
}
IMG_URL = "http://images.cocodataset.org/{split}/COCO_{split}_{id:012d}.jpg"
INPUT_ROOTS = [p for p in os.environ.get("PA3_INPUT_ROOTS", "/kaggle/input").split(os.pathsep) if p]
UA = {"User-Agent": "Mozilla/5.0 (pa3-task2 data prep)"}


def _http_get(url: str, tries: int = 5, timeout: float = 120) -> bytes:
    last = None
    for i in range(tries):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=timeout) as r:
                return r.read()
        except Exception as ex:  # noqa: BLE001
            last = ex
            time.sleep(min(60, 5 * 2 ** i))
    raise RuntimeError(f"download failed after {tries} tries: {url} ({last})")


def _download_to(url: str, dst: Path, tries: int = 5) -> Path:
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_suffix(dst.suffix + ".part")
    last = None
    for i in range(tries):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=300) as r, open(tmp, "wb") as fh:
                shutil.copyfileobj(r, fh, 1 << 22)
            os.replace(tmp, dst)
            return dst
        except Exception as ex:  # noqa: BLE001
            last = ex
            print(f"[download] {url}: {ex}; retry {i + 1}/{tries}", flush=True)
            time.sleep(min(60, 5 * 2 ** i))
    raise RuntimeError(f"download failed: {url} ({last})")


def find_attached(name: str) -> Path | None:
    for root in INPUT_ROOTS:
        r = Path(root)
        if r.exists():
            for p in r.rglob(name):
                if p.is_file():
                    return p
    return None


def ensure_annotation(name: str, raw_dir: Path) -> Path:
    """Path to a json (attached copy, cached copy, or extracted from the official zip)."""
    local = raw_dir / name
    if local.exists():
        return local
    att = find_attached(name)
    if att is not None:
        print(f"[data] using attached {att}", flush=True)
        return att
    url = FILES[name]
    zpath = raw_dir / "zips" / url.rsplit("/", 1)[1]
    if not zpath.exists():
        print(f"[data] downloading {url}", flush=True)
        _download_to(url, zpath)
    with zipfile.ZipFile(zpath) as z:
        member = next(m for m in z.namelist() if m.endswith(name))
        local.parent.mkdir(parents=True, exist_ok=True)
        with z.open(member) as src, open(local, "wb") as dst:
            shutil.copyfileobj(src, dst, 1 << 22)
    return local


# ------------------------------------------------------------------------------------------- images
_ID = re.compile(r"(\d{6,12})\.jpe?g$", re.IGNORECASE)


def attached_image_index(split: str) -> dict[int, Path]:
    """image_id -> path for any attached directory named like the split (train2014 / val2014)."""
    idx: dict[int, Path] = {}
    for root in INPUT_ROOTS:
        r = Path(root)
        if not r.exists():
            continue
        for d in r.rglob(split):
            if d.is_dir():
                for f in d.iterdir():
                    m = _ID.search(f.name)
                    if m:
                        idx.setdefault(int(m.group(1)), f)
    return idx


def image_path(img_dir: Path, key: str) -> Path:
    split, i = key.split(":")
    return img_dir / split / f"COCO_{split}_{int(i):012d}.jpg"


def ensure_images(keys: list[str], img_dir: Path, workers: int = 32) -> dict:
    """Make every image in `keys` ("train2014:123") available under img_dir (copy/link attached or download)."""
    from PIL import Image

    t0 = time.time()
    att = {s: attached_image_index(s) for s in ("train2014", "val2014")}
    todo, n_att = [], 0
    for k in keys:
        dst = image_path(img_dir, k)
        if dst.exists():
            continue
        split, i = k.split(":")
        src = att[split].get(int(i))
        dst.parent.mkdir(parents=True, exist_ok=True)
        if src is not None:
            try:
                os.symlink(src, dst)
            except OSError:
                shutil.copy2(src, dst)
            n_att += 1
        else:
            todo.append(k)

    def fetch(k: str) -> str | None:
        split, i = k.split(":")
        dst = image_path(img_dir, k)
        try:
            data = _http_get(IMG_URL.format(split=split, id=int(i)))
            Image.open(io.BytesIO(data)).verify()
            tmp = dst.with_suffix(".part")
            tmp.write_bytes(data)
            os.replace(tmp, dst)
            return None
        except Exception as ex:  # noqa: BLE001
            return f"{k}: {ex}"

    errors = []
    if todo:
        print(f"[data] downloading {len(todo)} images ({n_att} attached)", flush=True)
        with ThreadPoolExecutor(workers) as ex:
            for j, err in enumerate(ex.map(fetch, todo)):
                if err:
                    errors.append(err)
                if (j + 1) % 1000 == 0:
                    print(f"[data]   {j + 1}/{len(todo)} images", flush=True)
    if errors:
        raise RuntimeError(f"{len(errors)} images failed, e.g. {errors[:3]}")
    return {"n_images": len(keys), "attached": n_att, "downloaded": len(todo), "seconds": time.time() - t0}


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/data/grounding.py
"""Annotation-derived grounding questions (presence / counting) from COCO instances.

Presence pair: the same question on an image with the object (target "yes") and one without ("no").
Counting pair: the same question on two images with different unambiguous counts in 1..4 (digit targets).
Slot order inside every pair is randomised (seeded), so slot 0 is not always "yes" / the smaller count.
"""
from __future__ import annotations

from collections import defaultdict
from itertools import cycle

import numpy as np

from .coco import CATEGORIES, count_ok, counting_question, presence_negative_ok, presence_positive_ok, \
    presence_question

COUNT_COMBOS = [(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]


def _pair(kind: str, cat: str, split: str, ex: list[tuple[int, str]], rng: np.random.Generator) -> dict:
    if rng.random() < 0.5:
        ex = ex[::-1]
    q = presence_question(cat) if kind == "presence" else counting_question(cat)
    return {"kind": kind, "category": cat, "question": q,
            "examples": [{"image": f"{split}:{i}", "image_id": i, "target": t, "canonical": t, "answers": None}
                         for i, t in ex]}


def _draw_negative(cat: str, universe: list[int], stats: dict, caps: dict, rng: np.random.Generator,
                   excluded: set[int], tries: int = 2000) -> int | None:
    for _ in range(tries):
        i = universe[int(rng.integers(len(universe)))]
        if i in excluded:
            continue
        if presence_negative_ok(stats[i], cat, [c["caption"] for c in caps.get(i, [])]):
            return i
    return None


def train_pools(universe: list[int], stats: dict, caps: dict, cfg: dict, seed: int, split: str = "train2014"
                ) -> tuple[list[dict], list[dict], dict]:
    universe = sorted(universe)
    cap = cfg["pool_cap_per_category"]
    pres, cnt, report = [], [], {"presence_per_cat": {}, "counting_per_cat": {}}
    for ci, cat in enumerate(sorted(CATEGORIES)):
        rng = np.random.default_rng([seed, 11, ci])
        pos = [i for i in universe if presence_positive_ok(stats[i], cat, cfg["min_area"])]
        rng.shuffle(pos)
        n = 0
        for p in pos[:cap]:
            neg = _draw_negative(cat, universe, stats, caps, rng, {p})
            if neg is not None:
                pres.append(_pair("presence", cat, split, [(p, "yes"), (neg, "no")], rng))
                n += 1
        report["presence_per_cat"][cat] = n
        by = defaultdict(list)
        for i in universe:
            k = count_ok(stats[i], cat, cfg["min_area"], cfg["max_count"])
            if k is not None:
                by[k].append(i)
        ks = sorted(by)
        n = 0
        if len(ks) >= 2:
            imgs = [i for k in ks for i in by[k]]
            rng.shuffle(imgs)
            kof = {i: k for k in ks for i in by[k]}
            for a in imgs:
                if n >= cap:
                    break
                others = [k for k in ks if k != kof[a]]
                kb = others[int(rng.integers(len(others)))]
                b = by[kb][int(rng.integers(len(by[kb])))]
                cnt.append(_pair("counting", cat, split, [(a, str(kof[a])), (b, str(kb))], rng))
                n += 1
        report["counting_per_cat"][cat] = n
    report.update(n_presence=len(pres), n_counting=len(cnt))
    return pres, cnt, report


def eval_pairs(candidates: list[int], stats: dict, caps: dict, cfg: dict, n_presence: int, n_counting: int,
               used: set[int], seed: int, tag: int, split: str = "val2014") -> list[dict]:
    """Image-disjoint eval pairs (every image at most once; `used` is updated in place)."""
    rng = np.random.default_rng([seed, 21, tag])
    cands = sorted(candidates)
    cats = sorted(CATEGORIES)
    rng.shuffle(cats)
    out: list[dict] = []
    # presence: categories round-robin
    pos_lists = {}
    for cat in cats:
        lst = [i for i in cands if presence_positive_ok(stats[i], cat, cfg["min_area"])]
        rng.shuffle(lst)
        pos_lists[cat] = lst
    it = cycle(cats)
    guard = 0
    while sum(p["kind"] == "presence" for p in out) < n_presence:
        guard += 1
        assert guard < 100 * (n_presence + 1), "not enough presence candidates"
        cat = next(it)
        p = next((i for i in pos_lists[cat] if i not in used), None)
        if p is None:
            continue
        neg = _draw_negative(cat, cands, stats, caps, rng, used | {p})
        if neg is None:
            continue
        used.update((p, neg))
        out.append(_pair("presence", cat, split, [(p, "yes"), (neg, "no")], rng))
    # counting: count combinations round-robin, categories rotated for diversity
    by = {cat: defaultdict(list) for cat in cats}
    for i in cands:
        for cat in cats:
            k = count_ok(stats[i], cat, cfg["min_area"], cfg["max_count"])
            if k is not None:
                by[cat][k].append(i)
    for cat in cats:
        for k in by[cat]:
            rng.shuffle(by[cat][k])
    cat_order = list(cats)
    ci = 0
    combos = cycle(COUNT_COMBOS)
    guard = 0
    while sum(p["kind"] == "counting" for p in out) < n_counting:
        guard += 1
        assert guard < 100 * (n_counting + 1), "not enough counting candidates"
        ka, kb = next(combos)
        for j in range(len(cat_order)):
            cat = cat_order[(ci + j) % len(cat_order)]
            a = next((i for i in by[cat].get(ka, []) if i not in used), None)
            b = next((i for i in by[cat].get(kb, []) if i not in used and i != a), None)
            if a is not None and b is not None:
                used.update((a, b))
                out.append(_pair("counting", cat, split, [(a, str(ka)), (b, str(kb))], rng))
                ci = (ci + j + 1) % len(cat_order)
                break
    return out


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/data/normalize.py
"""VQA answer normalisation and local soft-consensus scoring.

Port of the official VQA evaluation code (vqaEval.py: processPunctuation + processDigitArticle,
contractions, number words -> digits, articles removed).  Applied identically to training targets,
references and predictions, for both suites.  The official quirk `periodStrip.sub("", t, re.UNICODE)`
(re.UNICODE passed as `count`=32) is irrelevant for short answers; we strip all such periods.
"""
from __future__ import annotations

import re

CONTRACTIONS = {
    "aint": "ain't", "arent": "aren't", "cant": "can't", "couldve": "could've", "couldnt": "couldn't",
    "couldn'tve": "couldn't've", "couldnt've": "couldn't've", "didnt": "didn't", "doesnt": "doesn't",
    "dont": "don't", "hadnt": "hadn't", "hadnt've": "hadn't've", "hadn'tve": "hadn't've", "hasnt": "hasn't",
    "havent": "haven't", "hed": "he'd", "hed've": "he'd've", "he'dve": "he'd've", "hes": "he's", "howd": "how'd",
    "howll": "how'll", "hows": "how's", "Id've": "I'd've", "I'dve": "I'd've", "Im": "I'm", "Ive": "I've",
    "isnt": "isn't", "itd": "it'd", "itd've": "it'd've", "it'dve": "it'd've", "itll": "it'll", "let's": "let's",
    "maam": "ma'am", "mightnt": "mightn't", "mightnt've": "mightn't've", "mightn'tve": "mightn't've",
    "mightve": "might've", "mustnt": "mustn't", "mustve": "must've", "neednt": "needn't", "notve": "not've",
    "oclock": "o'clock", "oughtnt": "oughtn't", "ow's'at": "'ow's'at", "'ows'at": "'ow's'at", "'ow'sat": "'ow's'at",
    "shant": "shan't", "shed've": "she'd've", "she'dve": "she'd've", "she's": "she's", "shouldve": "should've",
    "shouldnt": "shouldn't", "shouldnt've": "shouldn't've", "shouldn'tve": "shouldn't've", "somebody'd": "somebodyd",
    "somebodyd've": "somebody'd've", "somebody'dve": "somebody'd've", "somebodyll": "somebody'll",
    "somebodys": "somebody's", "someoned": "someone'd", "someoned've": "someone'd've", "someone'dve": "someone'd've",
    "someonell": "someone'll", "someones": "someone's", "somethingd": "something'd",
    "somethingd've": "something'd've", "something'dve": "something'd've", "somethingll": "something'll",
    "thats": "that's", "thered": "there'd", "thered've": "there'd've", "there'dve": "there'd've",
    "therere": "there're", "theres": "there's", "theyd": "they'd", "theyd've": "they'd've", "they'dve": "they'd've",
    "theyll": "they'll", "theyre": "they're", "theyve": "they've", "twas": "'twas", "wasnt": "wasn't",
    "wed've": "we'd've", "we'dve": "we'd've", "weve": "we've", "werent": "weren't", "whatll": "what'll",
    "whatre": "what're", "whats": "what's", "whatve": "what've", "whens": "when's", "whered": "where'd",
    "wheres": "where's", "whereve": "where've", "whod": "who'd", "whod've": "who'd've", "who'dve": "who'd've",
    "wholl": "who'll", "whos": "who's", "whove": "who've", "whyll": "why'll", "whyre": "why're", "whys": "why's",
    "wont": "won't", "wouldve": "would've", "wouldnt": "wouldn't", "wouldnt've": "wouldn't've",
    "wouldn'tve": "wouldn't've", "yall": "y'all", "yall'll": "y'all'll", "y'allll": "y'all'll",
    "yall'd've": "y'all'd've", "y'alld've": "y'all'd've", "y'all'dve": "y'all'd've", "youd": "you'd",
    "youd've": "you'd've", "you'dve": "you'd've", "youll": "you'll", "youre": "you're", "youve": "you've",
}
MANUAL_MAP = {"none": "0", "zero": "0", "one": "1", "two": "2", "three": "3", "four": "4", "five": "5",
              "six": "6", "seven": "7", "eight": "8", "nine": "9", "ten": "10"}
ARTICLES = {"a", "an", "the"}
PERIOD_STRIP = re.compile(r"(?!<=\d)(\.)(?!\d)")   # (sic) as in the official code
COMMA_STRIP = re.compile(r"(\d)(,)(\d)")
PUNCT = [";", r"/", "[", "]", '"', "{", "}", "(", ")", "=", "+", "\\", "_", "-", ">", "<", "@", "`", ",", "?", "!"]


def _punctuation(t: str) -> str:
    out = t
    for p in PUNCT:
        if (p + " " in t or " " + p in t) or (re.search(COMMA_STRIP, t) is not None):
            out = out.replace(p, "")
        else:
            out = out.replace(p, " ")
    return PERIOD_STRIP.sub("", out)


def _digit_article(t: str) -> str:
    words = []
    for w in t.lower().split():
        w = MANUAL_MAP.get(w, w)
        if w not in ARTICLES:
            words.append(w)
    return " ".join(CONTRACTIONS.get(w, w) for w in words)


def normalize_answer(a: str | None) -> str:
    if a is None:
        return ""
    t = a.replace("\n", " ").replace("\t", " ").strip()
    return _digit_article(_punctuation(t))


def soft_score(pred_norm: str, refs_norm: list[str]) -> float:
    """Local VQA accuracy: mean over the 10 leave-one-out subsets of min(1, #matches among the other 9 / 3)."""
    n = len(refs_norm)
    assert n > 0
    total = 0.0
    matches = sum(r == pred_norm for r in refs_norm)
    for r in refs_norm:
        others = matches - (r == pred_norm)
        total += min(1.0, others / 3.0)
    return total / n


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/data/prompts.py
"""Prompt construction: Qwen3 chat template with a visual slot.

Each example is rendered with `apply_chat_template([{"role": "user", "content": SENTINEL + "\n" + text}],
add_generation_prompt=True, enable_thinking=False)` and split at the sentinel:

    prefix  = "<|im_start|>user\n"                        (token ids, identical for all examples)
    [visual] = 196 (or 49) connector outputs                (inserted between prefix and suffix)
    suffix  = "\n{text}<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
    target  = content tokens (capped) + <|im_end|>          (supervised)

Prefix and suffix are tokenised separately (nothing can merge across the image).  The no-image control
uses exactly the same prefix + suffix ids with the visual rows removed.
"""
from __future__ import annotations

from dataclasses import dataclass, field

CAPTION_PROMPT = "Write a short caption for this image."
ANSWER_INSTRUCTION = "Answer the question using a single word or number."
SENTINEL = "⟦IMAGE⟧"


def user_text(question: str | None) -> str:
    return CAPTION_PROMPT if question is None else f"{question}\n{ANSWER_INSTRUCTION}"


@dataclass
class Encoded:
    ex_id: str
    image: str | None              # feature key, e.g. "train2014:123"
    prefix: list[int]
    suffix: list[int]
    target: list[int]              # content tokens + <|im_end|> (empty for prompts only)
    n_term: int = 1                # trailing termination tokens inside `target` (excluded from answer NLL)
    meta: dict = field(default_factory=dict)


class ChatFormat:
    def __init__(self, tok, caps: dict):
        self.tok = tok
        self.caps = caps
        self.im_end = tok.convert_tokens_to_ids("<|im_end|>")
        self.eos_ids = sorted({self.im_end, tok.convert_tokens_to_ids("<|endoftext|>")})
        self.pad_id = tok.pad_token_id if tok.pad_token_id is not None else tok.convert_tokens_to_ids("<|endoftext|>")
        pre, _ = self._render("x")
        self.prefix = tok(pre, add_special_tokens=False).input_ids
        self.stats = {"question_truncated": 0, "target_truncated": 0}
        self._suffix_cache: dict[str, list[int]] = {}

    def _render(self, text: str) -> tuple[str, str]:
        s = self.tok.apply_chat_template([{"role": "user", "content": SENTINEL + "\n" + text}], tokenize=False,
                                         add_generation_prompt=True, enable_thinking=False)
        assert s.count(SENTINEL) == 1, "chat template altered the sentinel"
        pre, post = s.split(SENTINEL)
        return pre, post

    def _cap(self, text: str, cap: int, key: str) -> tuple[list[int], str]:
        ids = self.tok(text, add_special_tokens=False).input_ids
        if len(ids) > cap:
            self.stats[key] += 1
            ids = ids[:cap]
            text = self.tok.decode(ids)
        return ids, text

    def encode(self, ex_id: str, image: str | None, question: str | None, target: str | None,
               meta: dict | None = None) -> Encoded:
        if question is not None:
            _, question = self._cap(question, self.caps["question"], "question_truncated")
        text = user_text(question)
        suffix = self._suffix_cache.get(text)
        if suffix is None:
            pre, post = self._render(text)
            assert self.tok(pre, add_special_tokens=False).input_ids == self.prefix, "prefix must not vary"
            suffix = self._suffix_cache[text] = self.tok(post, add_special_tokens=False).input_ids
        prefix = self.prefix
        tgt: list[int] = []
        if target is not None:
            cap = self.caps["caption"] if question is None else self.caps["answer"]
            tgt, _ = self._cap(target, cap, "target_truncated")
            tgt = tgt + [self.im_end]
        return Encoded(ex_id, image, prefix, suffix, tgt, 1, meta or {})


def encode_caption(fmt: ChatFormat, row: dict) -> Encoded:
    return fmt.encode(row["ex_id"], row["image"], None, row["caption"], {"kind": "caption"})


def encode_pair(fmt: ChatFormat, pair: dict) -> list[Encoded]:
    out = []
    for e in pair["examples"]:
        out.append(fmt.encode(e["ex_id"], e["image"], pair["question"], e["target"],
                              {"kind": pair["kind"], "pair_id": pair["pair_id"], "suite": pair["suite"]}))
    return out


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/data/sampler.py
"""Curriculum sampler (stateless: every batch is a pure function of (seed, stage, update)).

Stage 1: `passes` seeded permutations of the caption examples; pass p uses default_rng([seed, 1, p]);
         an update takes 16 consecutive examples of one pass (18,000 / 16 = 1,125 updates per pass).
Stages 2/3: an update = 8 draws, a microbatch = 4 draws; draw k of stage s uses default_rng([seed, s, k]):
         pick a component with the stage probabilities, then a uniform pair from that pool (both examples,
         kept together) or two uniform caption examples (caption replay).
Resuming only needs the global update counter, so there is no sampler RNG state to drift.
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np

# item = ("caption", index) or (pool_name, pair_index, slot)
Item = tuple


@dataclass
class StageSpec:
    index: int          # 1, 2, 3
    updates: int
    mix: dict | None    # None for stage 1


class CurriculumSampler:
    def __init__(self, seed: int, n_captions: int, pool_sizes: dict[str, int], micro_bs: int, grad_accum: int,
                 passes: int = 2):
        assert micro_bs % 2 == 0, "pairs must fit in a microbatch"
        self.seed, self.n_cap, self.pool_sizes = seed, n_captions, pool_sizes
        self.mb, self.acc, self.passes = micro_bs, grad_accum, passes
        self.eff = micro_bs * grad_accum
        self._perm: dict[int, np.ndarray] = {}

    def stage1_updates(self) -> int:
        assert self.n_cap % self.eff == 0, "caption count must be a multiple of the effective batch"
        return self.passes * self.n_cap // self.eff

    def _perm_for(self, p: int) -> np.ndarray:
        if p not in self._perm:
            self._perm[p] = np.random.default_rng([self.seed, 1, p]).permutation(self.n_cap)
        return self._perm[p]

    def draw(self, stage: int, k: int, mix: dict) -> tuple[str, list[Item]]:
        rng = np.random.default_rng([self.seed, stage, k])
        names = sorted(mix)
        probs = np.array([mix[n] for n in names], dtype=float)
        comp = names[int(rng.choice(len(names), p=probs / probs.sum()))]
        if comp == "caption":
            idx = rng.integers(self.n_cap, size=2)
            return comp, [("caption", int(idx[0])), ("caption", int(idx[1]))]
        j = int(rng.integers(self.pool_sizes[comp]))
        return comp, [(comp, j, 0), (comp, j, 1)]

    def update(self, spec: StageSpec, u: int) -> tuple[list[list[Item]], dict]:
        """Microbatches (lists of items) for local update u (0-based) of a stage, plus per-component example counts."""
        assert 0 <= u < spec.updates
        counts: dict[str, int] = {}
        if spec.mix is None:
            per_pass = self.n_cap // self.eff
            p, off = divmod(u, per_pass)
            idx = self._perm_for(p)[off * self.eff:(off + 1) * self.eff]
            items = [("caption", int(i)) for i in idx]
            counts["caption"] = self.eff
        else:
            dpu = self.eff // 2
            items = []
            for d in range(dpu):
                comp, its = self.draw(spec.index, u * dpu + d, spec.mix)
                counts[comp] = counts.get(comp, 0) + 2
                items += its
        mbs = [items[m * self.mb:(m + 1) * self.mb] for m in range(self.acc)]
        return mbs, counts


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/data/splits.py
"""Seeded data selection (seed 6304) -> manifests under $PA3_OUTPUT_ROOT/<data_dir>.

    python -m pa3vlm.data.splits [--data-config configs/data.yaml] [--no-images]

Order: evaluation sets first from val2014 (VQA test, VQA dev, grounding test, grounding dev, caption dev),
each image used at most once across all of them; then training sets from train2014 (captions, VQA pairs,
grounding pools built on caption + VQA-train images).  Disjointness is asserted and recorded.

Unified pair record:
  {"pair_id", "suite": "grounding"|"vqa", "kind", "question", "category"?, "examples": [
      {"ex_id", "image": "val2014:123", "image_id", "target", "canonical", "answers": [10]|None, "qid"?}, x2]}
Caption record: {"ex_id", "image", "image_id", "ann_id", "caption"}
"""
from __future__ import annotations

import argparse
import json
import os
import time
from collections import Counter
from pathlib import Path

import numpy as np

from pa3common.utils import output_root, write_json

from ..config import load_data_config
from .coco import index_captions, index_instances
from .download import ensure_annotation, ensure_images
from .grounding import eval_pairs, train_pools
from .vqa import candidate_pairs, load_vqa


def _load(name: str, raw: Path) -> dict:
    p = ensure_annotation(name, raw)
    print(f"[data] loading {p}", flush=True)
    return json.loads(p.read_text())


def _dedupe_pairs(pairs: list) -> list:
    seen, out = set(), []
    for p in pairs:
        k = frozenset(p)
        if len(k) == 2 and k not in seen:
            seen.add(k)
            out.append(list(p))
    return out


def _take_vqa(cands: list[dict], quotas: dict, rng: np.random.Generator, used: set[int] | None) -> list[dict]:
    """Per answer-type quota; if `used` is given every image is used at most once (eval sets)."""
    out = []
    for kind in sorted(quotas):
        pool = [c for c in cands if c["kind"] == kind]
        order = rng.permutation(len(pool))
        got = 0
        for j in order:
            c = pool[int(j)]
            ids = [e["image_id"] for e in c["examples"]]
            if used is not None and any(i in used for i in ids):
                continue
            if used is not None:
                used.update(ids)
            out.append(c)
            got += 1
            if got == quotas[kind]:
                break
        assert got == quotas[kind], f"only {got}/{quotas[kind]} VQA pairs of type {kind}"
    return out


def _finalize(pairs: list[dict], name: str, suite: str) -> list[dict]:
    for j, p in enumerate(pairs):
        p["pair_id"] = f"{name}-{j:04d}"
        p["suite"] = suite
        for s, e in enumerate(p["examples"]):
            e["ex_id"] = f"{p['pair_id']}/{s}"
    return pairs


def select_all(raw: Path, cfg: dict) -> tuple[dict, dict]:
    seed = cfg["seed"]
    rep: dict = {"seed": seed, "config": cfg}
    # ---------------------------------------------------------------- val2014: evaluation sets
    caps_val = index_captions(_load("captions_val2014.json", raw))
    _, stats_val = index_instances(_load("instances_val2014.json", raw))
    qv, av = load_vqa(_load("v2_OpenEnded_mscoco_val2014_questions.json", raw),
                      _load("v2_mscoco_val2014_annotations.json", raw))
    pv = _dedupe_pairs(_load("v2_mscoco_val2014_complementary_pairs.json", raw))
    cv = candidate_pairs(pv, qv, av, "val2014", cfg["vqa_min_agree"])
    rep["vqa_val_candidates"] = dict(Counter(c["kind"] for c in cv), pairs_in_file=len(pv))
    del qv, av
    used: set[int] = set()
    sets: dict[str, list] = {}
    sets["vqa_test"] = _finalize(_take_vqa(cv, cfg["vqa_test"], np.random.default_rng([seed, 1]), used), "vqa_test", "vqa")
    sets["vqa_dev"] = _finalize(_take_vqa(cv, cfg["vqa_dev"], np.random.default_rng([seed, 2]), used), "vqa_dev", "vqa")
    val_ids = sorted(stats_val)
    g = cfg["grounding_test"]
    sets["grounding_test"] = _finalize(eval_pairs(val_ids, stats_val, caps_val, cfg, g["presence"], g["counting"],
                                                  used, seed, tag=1), "grounding_test", "grounding")
    g = cfg["grounding_dev"]
    sets["grounding_dev"] = _finalize(eval_pairs(val_ids, stats_val, caps_val, cfg, g["presence"], g["counting"],
                                                 used, seed, tag=2), "grounding_dev", "grounding")
    rng = np.random.default_rng([seed, 3])
    free = [i for i in val_ids if i not in used and len(caps_val.get(i, [])) >= 1]
    dev_imgs = sorted(int(i) for i in rng.choice(free, cfg["caption_dev_images"], replace=False))
    used.update(dev_imgs)
    sets["captions_dev"] = []
    for j, i in enumerate(dev_imgs):
        c = caps_val[i][int(rng.integers(len(caps_val[i])))]
        sets["captions_dev"].append({"ex_id": f"cap_dev-{j:03d}", "image": f"val2014:{i}", "image_id": i,
                                     "ann_id": c["id"], "caption": " ".join(c["caption"].split())})
    del caps_val, stats_val
    # ---------------------------------------------------------------- train2014: training sets
    caps_tr = index_captions(_load("captions_train2014.json", raw))
    rng = np.random.default_rng([seed, 4])
    elig = sorted(i for i, v in caps_tr.items() if len(v) >= cfg["captions_per_train_image"])
    tr_imgs = sorted(int(i) for i in rng.choice(elig, cfg["caption_train_images"], replace=False))
    sets["captions_train"] = []
    for i in tr_imgs:
        for k in sorted(rng.choice(len(caps_tr[i]), cfg["captions_per_train_image"], replace=False)):
            c = caps_tr[i][int(k)]
            sets["captions_train"].append({"ex_id": f"cap_train-{len(sets['captions_train']):05d}",
                                           "image": f"train2014:{i}", "image_id": i, "ann_id": c["id"],
                                           "caption": " ".join(c["caption"].split())})
    qt, at = load_vqa(_load("v2_OpenEnded_mscoco_train2014_questions.json", raw),
                      _load("v2_mscoco_train2014_annotations.json", raw))
    pt = _dedupe_pairs(_load("v2_mscoco_train2014_complementary_pairs.json", raw))
    ct = candidate_pairs(pt, qt, at, "train2014", cfg["vqa_min_agree"])
    rep["vqa_train_candidates"] = dict(Counter(c["kind"] for c in ct), pairs_in_file=len(pt))
    del qt, at
    sets["vqa_train"] = _finalize(_take_vqa(ct, cfg["vqa_train"], np.random.default_rng([seed, 5]), None),
                                  "vqa_train", "vqa")
    _, stats_tr = index_instances(_load("instances_train2014.json", raw))
    universe = sorted(set(tr_imgs) | {e["image_id"] for p in sets["vqa_train"] for e in p["examples"]})
    pres, cnt, prep = train_pools(universe, stats_tr, caps_tr, cfg, seed)
    sets["presence_pool"] = _finalize(pres, "presence_pool", "grounding")
    sets["counting_pool"] = _finalize(cnt, "counting_pool", "grounding")
    rep["grounding_pools"] = prep
    rep["grounding_universe_images"] = len(universe)
    return sets, rep


def image_keys(sets: dict) -> dict[str, list[str]]:
    keys: dict[str, set[str]] = {"train": set(), "val": set()}
    for name, rows in sets.items():
        for r in rows:
            exs = r["examples"] if "examples" in r else [r]
            for e in exs:
                keys["train" if e["image"].startswith("train2014") else "val"].add(e["image"])
    return {k: sorted(v, key=lambda s: (s.split(":")[0], int(s.split(":")[1]))) for k, v in keys.items()}


def check_isolation(sets: dict) -> dict:
    """Eval sets: val2014 only, mutually image-disjoint, each image once; training: train2014 only."""
    def imgs(name):
        rows = sets[name]
        return [e["image"] for r in rows for e in (r["examples"] if "examples" in r else [r])]

    ev = ["vqa_test", "vqa_dev", "grounding_test", "grounding_dev", "captions_dev"]
    tr = ["captions_train", "vqa_train", "presence_pool", "counting_pool"]
    out = {}
    allev: list[str] = []
    for n in ev:
        im = imgs(n)
        assert all(i.startswith("val2014:") for i in im), n
        assert len(im) == len(set(im)), f"{n}: an image is used twice"
        allev += im
        out[n] = len(im)
    assert len(allev) == len(set(allev)), "evaluation sets share images"
    trimgs = {i for n in tr for i in imgs(n)}
    assert all(i.startswith("train2014:") for i in trimgs)
    assert not (trimgs & set(allev))
    for n in ev + ["vqa_train", "presence_pool", "counting_pool"]:
        for r in sets[n]:
            if "examples" in r:
                a, b = r["examples"]
                assert a["image"] != b["image"] and a["canonical"] != b["canonical"], r["pair_id"]
    out.update(eval_images_total=len(allev), train_images_total=len(trimgs), ok=True)
    return out


def write_manifests(sets: dict, rep: dict, data_dir: Path) -> None:
    data_dir.mkdir(parents=True, exist_ok=True)
    for name, rows in sets.items():
        write_json(data_dir / f"{name}.json", rows)
    write_json(data_dir / "images.json", image_keys(sets))
    ids = {}
    for name, rows in sets.items():
        if rows and "examples" in rows[0]:
            ids[name] = {"pair_ids": [r["pair_id"] for r in rows],
                         "source_question_pairs": [r.get("source_pair") for r in rows],
                         "images": [[e["image"] for e in r["examples"]] for r in rows]}
        else:
            ids[name] = {"ex_ids": [r["ex_id"] for r in rows], "ann_ids": [r["ann_id"] for r in rows],
                         "images": [r["image"] for r in rows]}
    write_json(data_dir / "selected_ids.json", ids)
    rep["isolation"] = check_isolation(sets)
    rep["sizes"] = {n: len(r) for n, r in sets.items()}
    rep["kinds"] = {n: dict(Counter(p["kind"] for p in r)) for n, r in sets.items() if r and "kind" in r[0]}
    write_json(data_dir / "selection_report.json", rep)


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-config", default=None)
    ap.add_argument("--set", nargs="*", default=[])
    ap.add_argument("--no-images", action="store_true")
    a = ap.parse_args()
    cfg = load_data_config(a.data_config, a.set)
    scratch = Path(os.environ.get("PA3_SCRATCH", "/tmp/pa3_scratch"))
    data_dir = output_root() / "data"
    t0 = time.time()
    if not (data_dir / "selection_report.json").exists():
        sets, rep = select_all(scratch / "raw", cfg)
        rep["selection_seconds"] = time.time() - t0
        write_manifests(sets, rep, data_dir)
        print(f"[data] manifests written ({time.time() - t0:.0f}s): {rep['sizes']}", flush=True)
    if not a.no_images:
        keys = json.loads((data_dir / "images.json").read_text())
        st = ensure_images(keys["train"] + keys["val"], scratch / "images")
        write_json(data_dir / "image_fetch.json", st)
        print(f"[data] images ready: {st}", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/data/vqa.py
"""VQAv2 complementary-pair filtering.

Kept pairs: both questions exist, different images, same question text and answer_type, each image's
10 normalised answers have a consensus answer given by >= min_agree annotators, and the two consensus
answers differ.  Training target = most frequent *original* answer string inside the consensus group
(ties: lexicographically smallest); all ten raw answers are kept for scoring.
"""
from __future__ import annotations

from collections import Counter

from .normalize import normalize_answer


def consensus(raw_answers: list[str], min_agree: int) -> tuple[str, str] | None:
    """(canonical normalised answer, original target string) or None."""
    assert len(raw_answers) == 10, len(raw_answers)
    norm = [normalize_answer(a) for a in raw_answers]
    canon, n = Counter(norm).most_common(1)[0]
    if n < min_agree or canon == "":
        return None
    raw = Counter(a.strip() for a, z in zip(raw_answers, norm) if z == canon)
    target = sorted(raw.items(), key=lambda kv: (-kv[1], kv[0]))[0][0]
    return canon, target


def candidate_pairs(pairs: list[list[int]], questions: dict, annotations: dict, split: str,
                    min_agree: int) -> list[dict]:
    """questions: qid -> {image_id, question}; annotations: qid -> {answers: [10 raw], answer_type}."""
    out = []
    for q1, q2 in pairs:
        if q1 not in questions or q2 not in questions or q1 not in annotations or q2 not in annotations:
            continue
        a, b = questions[q1], questions[q2]
        if a["image_id"] == b["image_id"]:
            continue
        if " ".join(a["question"].split()).lower() != " ".join(b["question"].split()).lower():
            continue
        ta, tb = annotations[q1]["answer_type"], annotations[q2]["answer_type"]
        if ta != tb:
            continue
        ca = consensus(annotations[q1]["answers"], min_agree)
        cb = consensus(annotations[q2]["answers"], min_agree)
        if ca is None or cb is None or ca[0] == cb[0]:
            continue
        exs = []
        for qid, q, (canon, target) in ((q1, a, ca), (q2, b, cb)):
            exs.append({"qid": qid, "image": f"{split}:{q['image_id']}", "image_id": q["image_id"],
                        "target": target, "canonical": canon, "answers": list(annotations[qid]["answers"])})
        out.append({"kind": ta, "question": " ".join(a["question"].split()), "examples": exs,
                    "source_pair": [q1, q2]})
    return out


def load_vqa(questions_json: dict, annotations_json: dict) -> tuple[dict, dict]:
    qs = {q["question_id"]: {"image_id": q["image_id"], "question": q["question"]}
          for q in questions_json["questions"]}
    anns = {a["question_id"]: {"answers": [x["answer"] for x in a["answers"]], "answer_type": a["answer_type"]}
            for a in annotations_json["annotations"]}
    return qs, anns


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/eval/__init__.py



In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/eval/bootstrap.py
"""Pair bootstrap (complete complementary pairs are the resampling unit).

Within a suite (and question-type subset) every metric and every condition uses the *same* resampled pair
indices, so condition differences (e.g. correct - swap) are paired.  Percentile 95% intervals.
"""
from __future__ import annotations

import numpy as np


def resample_indices(n_pairs: int, n_boot: int, seed_parts: list[int]) -> np.ndarray:
    return np.random.default_rng(seed_parts).integers(0, n_pairs, size=(n_boot, n_pairs))


def ci(values: np.ndarray, idx: np.ndarray, alpha: float = 0.05) -> tuple[float, float, float]:
    point = float(values.mean())
    boots = values[idx].mean(axis=1)
    lo, hi = np.quantile(boots, [alpha / 2, 1 - alpha / 2])
    return point, float(lo), float(hi)


def diff_ci(a: np.ndarray, b: np.ndarray, idx: np.ndarray, alpha: float = 0.05) -> tuple[float, float, float]:
    """CI of mean(a) - mean(b) with a, b aligned on the same pairs."""
    assert a.shape == b.shape
    return ci(a - b, idx, alpha)


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/eval/conditions.py
"""Image-control conditions.  Questions and original references stay fixed; only the image input changes.

correct    : the example's own image
swap       : the partner image of the same complementary pair (different reference answer)
mismatched : the image in the same slot of another pair of the same suite (seeded derangement of pairs,
             fixed before generation and reused everywhere; never either image of the original pair;
             every image is used exactly once)
none       : visual tokens omitted (masks and positions rebuilt; text unchanged)
"""
from __future__ import annotations

import numpy as np

CONDITIONS = ("correct", "swap", "mismatched", "none")


def derangement(n: int, seed_parts: list[int]) -> list[int]:
    """Sattolo's algorithm: a uniformly random cyclic permutation, hence no fixed points (n >= 2)."""
    assert n >= 2
    rng = np.random.default_rng(seed_parts)
    p = list(range(n))
    for i in range(n - 1, 0, -1):
        j = int(rng.integers(i))
        p[i], p[j] = p[j], p[i]
    assert all(p[i] != i for i in range(n))
    return p


def mismatch_mapping(pairs: list[dict], seed: int, tag: int) -> dict[str, str]:
    """ex_id -> image key of the mismatched control."""
    perm = derangement(len(pairs), [seed, 31, tag])
    out = {}
    for i, p in enumerate(pairs):
        other = pairs[perm[i]]
        own = {e["image"] for e in p["examples"]}
        for s, e in enumerate(p["examples"]):
            img = other["examples"][s]["image"]
            assert img not in own
            out[e["ex_id"]] = img
    assert len(set(out.values())) == len(out), "each image is used once"
    return out


def images_for(pairs: list[dict], cond: str, mismatch: dict[str, str] | None = None) -> list[str | None]:
    imgs: list[str | None] = []
    for p in pairs:
        a, b = p["examples"]
        for e, partner in ((a, b), (b, a)):
            if cond == "correct":
                imgs.append(e["image"])
            elif cond == "swap":
                imgs.append(partner["image"])
            elif cond == "mismatched":
                imgs.append(mismatch[e["ex_id"]])
            elif cond == "none":
                imgs.append(None)
            else:
                raise ValueError(cond)
    return imgs


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/eval/dev.py
"""Shared evaluation runners and the development evaluation.

Development metrics (logged before training, every 500 updates and at stage ends):
- caption reference loss on the 96 development captions (training objective: content + <|im_end|>),
  with the correct image and (extra) without the image;
- reference-answer NLL (termination token excluded; token-mean per example, then mean over examples) on
  both development suites under correct / complementary-swap / no-image inputs, and the image-swap
  penalty  dNLL = NLL(y | q, I_partner) - NLL(y | q, I_correct)  (positive = the correct image helps);
- (extra) greedy accuracy with the correct image;
- the median per-token norm of the scaled connector output on development images (drift monitor).
"""
from __future__ import annotations

import contextlib

import numpy as np
import torch

from ..data.collate import collate
from ..data.normalize import normalize_answer
from ..data.prompts import Encoded
from ..models.vlm import VLM, per_example_mean
from .conditions import images_for
from .generate import timed_greedy
from .metrics import score


def amp_ctx(device: torch.device):
    return torch.autocast("cuda", dtype=torch.float16) if device.type == "cuda" else contextlib.nullcontext()


@torch.no_grad()
def nll_examples(vlm: VLM, encs: list[Encoded], images: list, get_feats, n_vis: int, pad_id: int, bs: int,
                 device: torch.device, exclude_term: bool = True) -> np.ndarray:
    out = []
    for i in range(0, len(encs), bs):
        b = collate(encs[i:i + bs], images[i:i + bs], get_feats, n_vis, pad_id, "right", True).to(device)
        with amp_ctx(device):
            nll, rows, term = vlm.supervised_nll(b)
        keep = ~term if exclude_term else None
        out.append(per_example_mean(nll, rows, len(encs[i:i + bs]), keep).cpu())
    return torch.cat(out).numpy().astype(np.float64)


@torch.no_grad()
def generate_examples(vlm: VLM, encs: list[Encoded], images: list, get_feats, n_vis: int, fmt, bs: int,
                      max_new: int, device: torch.device) -> list[dict]:
    res = []
    for i in range(0, len(encs), bs):
        chunk = encs[i:i + bs]
        b = collate(chunk, images[i:i + bs], get_feats, n_vis, fmt.pad_id, "left", False).to(device)
        with amp_ctx(device):
            seqs, stopped, secs = timed_greedy(vlm, b, max_new, fmt.eos_ids, fmt.pad_id, device)
        for s, st in zip(seqs, stopped):
            raw = fmt.tok.decode(s, skip_special_tokens=True)
            res.append({"pred_raw": raw, "pred_norm": normalize_answer(raw), "stopped": bool(st),
                        "n_new_tokens": len(s) + int(st), "gen_time": secs / len(chunk)})
    return res


def dev_eval(vlm: VLM, bundle, get_feats, n_vis: int, fmt, device: torch.device, bs: int, max_new: int,
             update: int, stage: int) -> dict:
    was_training = vlm.training
    vlm.eval()
    row: dict = {"update": update, "stage": stage}
    caps = bundle.captions_dev()
    imgs = [e.image for e in caps]
    row["caption_loss"] = float(nll_examples(vlm, caps, imgs, get_feats, n_vis, fmt.pad_id, bs, device, False).mean())
    row["caption_loss_noimage"] = float(nll_examples(vlm, caps, [None] * len(caps), get_feats, n_vis, fmt.pad_id, bs,
                                                     device, False).mean())
    for name, suite in (("grounding_dev", "grounding"), ("vqa_dev", "vqa")):
        pairs, encs = bundle.suite(name)
        kinds = np.array([p["kind"] for p in pairs for _ in p["examples"]])
        exs = [e for p in pairs for e in p["examples"]]
        nl = {}
        for cond in ("correct", "swap", "none"):
            nl[cond] = nll_examples(vlm, encs, images_for(pairs, cond), get_feats, n_vis, fmt.pad_id, bs, device)
        gens = generate_examples(vlm, encs, images_for(pairs, "correct"), get_feats, n_vis, fmt, bs, max_new, device)
        acc = np.array([score(suite, g["pred_norm"], e) for g, e in zip(gens, exs)])
        for sub in ["all"] + sorted(set(kinds)):
            m = np.ones(len(kinds), bool) if sub == "all" else kinds == sub
            key = f"{suite}/{sub}"
            for cond, v in nl.items():
                row[f"{key}/nll_{cond}"] = float(v[m].mean())
            row[f"{key}/delta_nll_swap"] = float((nl["swap"] - nl["correct"])[m].mean())
            row[f"{key}/delta_nll_none"] = float((nl["none"] - nl["correct"])[m].mean())
            row[f"{key}/acc_greedy"] = float(acc[m].mean())
    with torch.no_grad(), amp_ctx(device):
        f = get_feats(sorted(set(imgs))[:32]).to(device)
        row["connector_norm_median"] = float(vlm.connector(f).float().norm(dim=-1).median())
    if was_training:
        vlm.train()
    return row


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/eval/final.py
"""Final test evaluation of a finished run under four image conditions, plus summary tables with pair-bootstrap CIs.

    python -m pa3vlm.eval.final --config vt196

Greedy decoding, thinking disabled (template), 16 new tokens, batch 8.  Questions and original references
are fixed across conditions; the mismatched mapping is fixed (seeded) before generation and saved.
Per-example records (raw generation, normalised answer, score, reference NLL, time) are written per
condition; finished conditions are skipped on rerun.
"""
from __future__ import annotations

import argparse
import json
import time
from pathlib import Path

import numpy as np
import torch

from pa3common.lora import load_lora_state
from pa3common.utils import gpu_mem_stats, output_root, read_json, write_json

from ..config import load_config
from ..models.build import build, device_dtype
from ..train.loop import attach_lora
from .bootstrap import ci, diff_ci, resample_indices
from .conditions import CONDITIONS, images_for, mismatch_mapping
from .dev import generate_examples, nll_examples
from .metrics import METRICS, pair_matrix, score

SUITES = (("grounding_test", "grounding", 1), ("vqa_test", "vqa", 2))


def load_run_weights(vlm, path: Path, cfg: dict) -> dict:
    w = torch.load(path, map_location="cpu", weights_only=False)
    if w["lora"]:
        attach_lora(vlm, cfg)
        load_lora_state(vlm.lm, w["lora"])
    vlm.connector.load_state_dict(w["connector"])
    return w["meta"]


def run_conditions(cfg: dict, weights: str = "final.pt") -> Path:
    device, dtype = device_dtype(cfg)
    run = output_root() / "runs" / cfg["run_name"]
    out = run / "final_eval"
    out.mkdir(parents=True, exist_ok=True)
    tok, fmt, bundle, store, vlm = build(cfg, device, dtype, grad_ckpt=False)
    meta = load_run_weights(vlm, run / "weights" / weights, cfg)
    vlm.to(device).eval()
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    for name, suite, tag in SUITES:
        pairs, encs = bundle.suite(name)
        mm_path = output_root() / "eval_mappings" / f"{name}_mismatched.json"
        mm = mismatch_mapping(pairs, cfg["seed"], tag)
        if mm_path.exists():
            assert read_json(mm_path) == mm, "mismatched mapping changed"
        else:
            write_json(mm_path, mm)
        exs = [(p, s, e) for p in pairs for s, e in enumerate(p["examples"])]
        for cond in CONDITIONS:
            f = out / f"{name}__{cond}.jsonl"
            if f.exists():
                continue
            t0 = time.time()
            imgs = images_for(pairs, cond, mm)
            gens = generate_examples(vlm, encs, imgs, store.get, store.n_vis, fmt, cfg["eval_bs"],
                                     cfg["max_new_tokens"], device)
            nll = nll_examples(vlm, encs, imgs, store.get, store.n_vis, fmt.pad_id, cfg["eval_bs"], device)
            recs = []
            for (p, s, e), gen, img, nl in zip(exs, gens, imgs, nll):
                partner = p["examples"][1 - s]
                recs.append({"suite": suite, "condition": cond, "pair_id": p["pair_id"], "slot": s, "ex_id": e["ex_id"],
                             "kind": p["kind"], "question": p["question"], "image_used": img, "own_image": e["image"],
                             "partner_image": partner["image"], "target": e["target"], "canonical": e["canonical"],
                             "partner_canonical": partner["canonical"], "answers": e["answers"], **gen,
                             "score": score(suite, gen["pred_norm"], e), "nll": float(nl)})
            tmp = f.with_suffix(".tmp")
            tmp.write_text("".join(json.dumps(r) + "\n" for r in recs))
            tmp.replace(f)
            print(f"[final] {cfg['run_name']} {name} {cond}: acc={np.mean([r['score'] for r in recs]):.3f} "
                  f"({time.time() - t0:.0f}s)", flush=True)
    write_json(out / "eval_resources.json", {"memory": gpu_mem_stats(device), "weights": weights, "meta": meta,
                                             "gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else "cpu"})
    return out


def _fmt(p, lo, hi, pct=True):
    k = 100 if pct else 1
    return f"{p * k:.1f} [{lo * k:.1f}, {hi * k:.1f}]" if pct else f"{p:.3f} [{lo:.3f}, {hi:.3f}]"


def summarize(run_dir: Path, seed: int, n_boot: int) -> dict:
    out = run_dir / "final_eval"
    recs = [json.loads(l) for f in sorted(out.glob("*__*.jsonl")) for l in f.read_text().splitlines() if l.strip()]
    pm = pair_matrix(recs)
    summary: dict = {"n_boot": n_boot, "suites": {}}
    md = []
    for name, suite, tag in SUITES:
        if (suite, "correct") not in pm:
            continue
        base = pm[(suite, "correct")]
        kinds = sorted(set(base["kinds"]))
        S: dict = {}
        for ki, sub in enumerate(["all"] + kinds):
            m = np.ones(len(base["kinds"]), bool) if sub == "all" else np.array(base["kinds"]) == sub
            idx = resample_indices(int(m.sum()), n_boot, [seed, 41, tag, ki])
            T: dict = {"n_pairs": int(m.sum()), "conditions": {}, "advantages": {}}
            for cond in CONDITIONS:
                if (suite, cond) not in pm:
                    continue
                M = pm[(suite, cond)]
                assert M["pair_ids"] == base["pair_ids"], "pairs must match across conditions"
                T["conditions"][cond] = {mt: ci(M[mt][m], idx) for mt in METRICS}
            for cond in ("swap", "mismatched", "none"):
                if (suite, cond) not in pm:
                    continue
                M = pm[(suite, cond)]
                T["advantages"][cond] = {"acc": diff_ci(base["acc"][m], M["acc"][m], idx),
                                         "pair_acc": diff_ci(base["pair_acc"][m], M["pair_acc"][m], idx),
                                         "nll": diff_ci(M["nll"][m], base["nll"][m], idx)}
            S[sub] = T
        summary["suites"][suite] = S
        # ---------- markdown
        md.append(f"### {suite} ({S['all']['n_pairs']} pairs, {2 * S['all']['n_pairs']} questions) — "
                  "mean [95% pair-bootstrap CI]\n")
        md.append("| condition | accuracy % | pair acc % | identical % | empty % | follows partner % | unsupported % | "
                  "ref NLL | ms/answer |")
        md.append("|---|---|---|---|---|---|---|---|---|")
        for cond, v in S["all"]["conditions"].items():
            md.append(f"| {cond} | {_fmt(*v['acc'])} | {_fmt(*v['pair_acc'])} | {_fmt(*v['identical'])} | "
                      f"{_fmt(*v['empty'])} | {_fmt(*v['follows_partner'])} | {_fmt(*v['unsupported'])} | "
                      f"{_fmt(*v['nll'], pct=False)} | {v['gen_time'][0] * 1000:.1f} |")
        md.append("\n**By question type (accuracy %)**\n")
        md.append("| type | pairs | " + " | ".join(CONDITIONS) + " |")
        md.append("|---|---|" + "---|" * len(CONDITIONS))
        for sub in ["all"] + kinds:
            cells = [_fmt(*S[sub]["conditions"][c]["acc"]) if c in S[sub]["conditions"] else "—" for c in CONDITIONS]
            md.append(f"| {sub} | {S[sub]['n_pairs']} | " + " | ".join(cells) + " |")
        md.append("\n**Correct-image advantage** (accuracy / pair accuracy: correct − control, pp; "
                  "NLL: control − correct)\n")
        md.append("| type | control | Δ accuracy | Δ pair acc | Δ NLL |")
        md.append("|---|---|---|---|---|")
        for sub in ["all"] + kinds:
            for cond, v in S[sub]["advantages"].items():
                md.append(f"| {sub} | {cond} | {_fmt(*v['acc'])} | {_fmt(*v['pair_acc'])} | "
                          f"{_fmt(*v['nll'], pct=False)} |")
        md.append("")
    write_json(out / "summary.json", summary)
    (out / "summary.md").write_text("\n".join(md))
    return summary


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--set", nargs="*", default=[])
    ap.add_argument("--weights", default="final.pt")
    a = ap.parse_args()
    cfg = load_config(a.config, a.set)
    out = run_conditions(cfg, a.weights)
    summarize(out.parent, cfg["seed"], cfg["bootstrap"])
    print(f"[final] {cfg['run_name']}: summary written", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/eval/generate.py
"""Greedy decoding with a KV cache and explicit position ids (left-padded batches, inputs_embeds prefill).

Written by hand rather than via HF `generate` so that positions for left-padded embedding inputs are
under our control (cumsum(mask)-1, continued by +1 per step); checked against HF generate in the tests.
"""
from __future__ import annotations

import time

import torch

from ..data.collate import Batch
from ..models.vlm import VLM


@torch.no_grad()
def greedy(vlm: VLM, b: Batch, max_new: int, eos_ids: list[int], pad_id: int) -> tuple[list[list[int]], list[bool]]:
    """Returns new token ids per row (termination token excluded) and whether the row stopped by itself."""
    body, head = vlm.body, vlm.lm.lm_head
    out = body(inputs_embeds=vlm.embed(b), attention_mask=b.attn, position_ids=b.pos, use_cache=True)
    past = out.past_key_values
    nxt = head(out.last_hidden_state[:, -1]).float().argmax(-1)
    B = nxt.shape[0]
    eos = torch.tensor(eos_ids, device=nxt.device)
    done = torch.zeros(B, dtype=torch.bool, device=nxt.device)
    stopped = torch.zeros(B, dtype=torch.bool, device=nxt.device)
    toks: list[torch.Tensor] = []
    attn, pos = b.attn, b.pos[:, -1:]
    for step in range(max_new):
        is_eos = torch.isin(nxt, eos)
        stopped |= is_eos & ~done
        toks.append(torch.where(done | is_eos, torch.full_like(nxt, -1), nxt))
        done |= is_eos
        if bool(done.all()) or step == max_new - 1:
            break
        attn = torch.cat([attn, torch.ones((B, 1), dtype=attn.dtype, device=attn.device)], dim=1)
        pos = pos + 1
        out = body(input_ids=nxt.unsqueeze(1), attention_mask=attn, position_ids=pos, past_key_values=past,
                   use_cache=True)
        past = out.past_key_values
        nxt = head(out.last_hidden_state[:, -1]).float().argmax(-1)
    T = torch.stack(toks, 1).tolist() if toks else [[] for _ in range(B)]
    seqs = []
    for row in T:
        r = []
        for t in row:
            if t == -1:
                break
            r.append(t)
        seqs.append(r)
    return seqs, stopped.tolist()


def timed_greedy(vlm: VLM, b: Batch, max_new: int, eos_ids: list[int], pad_id: int, device: torch.device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    t0 = time.perf_counter()
    seqs, stopped = greedy(vlm, b, max_new, eos_ids, pad_id)
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    return seqs, stopped, time.perf_counter() - t0


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/eval/metrics.py
"""Per-example scoring and per-(suite, condition, question type) summaries.

grounding: normalised exact match against the canonical answer ("yes"/"no"/digit).
vqa      : local soft-consensus accuracy against all ten (normalised) human answers.
Pair accuracy: both predictions equal their original canonical answers.  Identical-answer rate: both
predictions of a pair are the same normalised string.  Empty outputs stay in every denominator.
Under swaps we also record whether a prediction equals the *partner's* canonical answer (follows the new
image) or neither canonical answer (unsupported).
"""
from __future__ import annotations

from collections import defaultdict

import numpy as np

from ..data.normalize import normalize_answer, soft_score


def score(suite: str, pred_norm: str, ex: dict) -> float:
    if suite == "vqa":
        return soft_score(pred_norm, [normalize_answer(a) for a in ex["answers"]])
    return float(pred_norm == ex["canonical"])


def pair_table(records: list[dict]) -> dict[tuple, dict]:
    """Group example records (one condition) by pair -> dict with both slots."""
    out: dict[tuple, dict] = defaultdict(dict)
    for r in records:
        out[(r["suite"], r["condition"], r["pair_id"])][r["slot"]] = r
    for k, v in out.items():
        assert set(v) == {0, 1}, k
    return out


def pair_values(pair: dict) -> dict[str, float]:
    a, b = pair[0], pair[1]
    return {
        "acc": (a["score"] + b["score"]) / 2,
        "pair_acc": float(a["pred_norm"] == a["canonical"] and b["pred_norm"] == b["canonical"]),
        "identical": float(a["pred_norm"] == b["pred_norm"]),
        "empty": (float(a["pred_norm"] == "") + float(b["pred_norm"] == "")) / 2,
        "follows_partner": (float(a["pred_norm"] == a["partner_canonical"]) +
                            float(b["pred_norm"] == b["partner_canonical"])) / 2,
        "unsupported": sum(float(r["pred_norm"] not in (r["canonical"], r["partner_canonical"])) for r in (a, b)) / 2,
        "nll": (a["nll"] + b["nll"]) / 2,
        "gen_time": (a["gen_time"] + b["gen_time"]) / 2,
        "hit_cap": (float(not a["stopped"]) + float(not b["stopped"])) / 2,
    }


METRICS = ("acc", "pair_acc", "identical", "empty", "follows_partner", "unsupported", "nll", "gen_time", "hit_cap")


def pair_matrix(records: list[dict]) -> dict:
    """{(suite, condition): {"pair_ids": [...], "kinds": [...], metric: np.array[P]}} with pairs sorted by id."""
    pt = pair_table(records)
    groups: dict[tuple, list] = defaultdict(list)
    for (suite, cond, pid), pr in pt.items():
        groups[(suite, cond)].append((pid, pr))
    out = {}
    for k, lst in groups.items():
        lst.sort(key=lambda x: x[0])
        vals = [pair_values(pr) for _, pr in lst]
        out[k] = {"pair_ids": [p for p, _ in lst], "kinds": [pr[0]["kind"] for _, pr in lst],
                  **{m: np.array([v[m] for v in vals], dtype=np.float64) for m in METRICS}}
    return out


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/eval/qualitative.py
"""Qualitative complementary-pair examples with a selection rule fixed before looking at outputs.

Rule: for each suite, classify every test pair (main run) into buckets using the correct and swap
conditions, then pick up to K pairs per bucket with a seeded RNG (seed 6304) from the bucket's pairs
sorted by pair id:
  grounded   : both answers right with the correct images AND both follow the partner's answer when swapped
  same_answer: identical answer for both images of the pair (correct condition)
  unsupported: at least one swapped prediction matches neither the own nor the partner reference
A pair may fall into several buckets; each pair is shown at most once (first bucket in this order).
"""
from __future__ import annotations

import json
import textwrap
from collections import defaultdict
from pathlib import Path

import numpy as np

BUCKETS = ("grounded", "same_answer", "unsupported")


def load_records(final_dir: Path) -> list[dict]:
    return [json.loads(l) for f in sorted(final_dir.glob("*__*.jsonl")) for l in f.read_text().splitlines() if l.strip()]


def by_pair(recs: list[dict]) -> dict:
    out: dict = defaultdict(lambda: defaultdict(dict))
    for r in recs:
        out[(r["suite"], r["pair_id"])][r["condition"]][r["slot"]] = r
    return out


def buckets(pr: dict) -> list[str]:
    c, s = pr.get("correct", {}), pr.get("swap", {})
    out = []
    if c and s and all(c[i]["pred_norm"] == c[i]["canonical"] for i in (0, 1)) and \
            all(s[i]["pred_norm"] == s[i]["partner_canonical"] for i in (0, 1)):
        out.append("grounded")
    if c and c[0]["pred_norm"] == c[1]["pred_norm"]:
        out.append("same_answer")
    if s and any(s[i]["pred_norm"] not in (s[i]["canonical"], s[i]["partner_canonical"]) for i in (0, 1)):
        out.append("unsupported")
    return out


def select(recs: list[dict], seed: int, k: int = 2) -> list[dict]:
    bp = by_pair(recs)
    chosen, seen = [], set()
    for si, suite in enumerate(sorted({s for s, _ in bp})):
        for bi, b in enumerate(BUCKETS):
            cands = sorted(pid for (s, pid), pr in bp.items() if s == suite and b in buckets(pr) and (s, pid) not in seen)
            rng = np.random.default_rng([seed, 51, si, bi])
            pick = sorted(rng.choice(cands, size=min(k, len(cands)), replace=False).tolist()) if cands else []
            for pid in pick:
                seen.add((suite, pid))
                chosen.append({"suite": suite, "pair_id": pid, "bucket": b, "conditions": {
                    c: {str(sl): {kk: r[kk] for kk in ("question", "image_used", "own_image", "target", "canonical",
                                                       "pred_raw", "pred_norm", "score", "answers")}
                        for sl, r in v.items()} for c, v in bp[(suite, pid)].items()}})
    return chosen


def render(chosen: list[dict], img_dir: Path, out_png: Path) -> None:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from PIL import Image

    from ..data.download import image_path

    if not chosen:
        return
    n = len(chosen)
    fig, axes = plt.subplots(n, 2, figsize=(11, 4.3 * n))
    axes = np.atleast_2d(axes)
    for r, ex in enumerate(chosen):
        cond = ex["conditions"]
        for s in (0, 1):
            ax = axes[r, s]
            c = cond["correct"][str(s)]
            try:
                ax.imshow(Image.open(image_path(img_dir, c["own_image"])).convert("RGB"))
            except OSError:
                ax.text(0.5, 0.5, "image unavailable", ha="center")
            ax.axis("off")
            refs = c["canonical"] if c["answers"] is None else ", ".join(sorted(set(c["answers"])))
            lines = [f"[{ex['bucket']}] {ex['pair_id']} slot {s}", f"Q: {c['question']}", f"ref: {refs}"]
            for cn in ("correct", "swap", "mismatched", "none"):
                if cn in cond:
                    lines.append(f"{cn}: {cond[cn][str(s)]['pred_raw']!r}")
            ax.set_title("\n".join(textwrap.shorten(x, 70) for x in lines), fontsize=8, loc="left")
    fig.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=90)
    plt.close(fig)


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/features/__init__.py



In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/features/siglip_cache.py
"""Frozen SigLIP feature cache (run once, before training; the process exits so SigLIP leaves the GPU).

    python -m pa3vlm.features.siglip_cache --shard 0 --nshards 2     # one process per GPU

Features = the 196 final patch tokens in 14x14 row-major order with SigLIP's final LayerNorm applied
exactly once: we compute post_layernorm(hidden_states[-1]) explicitly and check it equals HF's
last_hidden_state (which is already post-LN), so the cache is correct whichever way HF behaves.
Encoder runs in FP32; features are stored as FP16 in a .npy memmap [N, 196, 768].
Preprocessing is deterministic PIL code driven by the checkpoint's preprocessor_config.json
(RGB, resize to 224x224 with its resample filter, rescale 1/255, normalise with its mean/std).
"""
from __future__ import annotations

import argparse
import json
import os
import time
from pathlib import Path

import numpy as np
import torch

from pa3common.utils import gpu_mem_stats, output_root, write_json

from ..data.download import image_path


def load_preproc(vision_id: str) -> dict:
    if Path(vision_id).is_dir():
        cfg = json.loads((Path(vision_id) / "preprocessor_config.json").read_text())
    else:
        from huggingface_hub import hf_hub_download

        cfg = json.loads(Path(hf_hub_download(vision_id, "preprocessor_config.json")).read_text())
    size = cfg.get("size", {"height": 224, "width": 224})
    return {"height": int(size.get("height", 224)), "width": int(size.get("width", 224)),
            "resample": int(cfg.get("resample", 3)), "rescale": float(cfg.get("rescale_factor", 1 / 255)),
            "mean": [float(x) for x in cfg.get("image_mean", [0.5] * 3)],
            "std": [float(x) for x in cfg.get("image_std", [0.5] * 3)],
            "do_resize": cfg.get("do_resize", True), "do_normalize": cfg.get("do_normalize", True)}


def preprocess(path: Path, pp: dict) -> np.ndarray:
    from PIL import Image

    with Image.open(path) as im:
        im = im.convert("RGB")
        if pp["do_resize"]:
            im = im.resize((pp["width"], pp["height"]), resample=pp["resample"])
        x = np.asarray(im, dtype=np.float32) * pp["rescale"]
    if pp["do_normalize"]:
        x = (x - np.array(pp["mean"], np.float32)) / np.array(pp["std"], np.float32)
    return x.transpose(2, 0, 1).copy()


class _DS(torch.utils.data.Dataset):
    def __init__(self, keys, img_dir, pp):
        self.keys, self.img_dir, self.pp = keys, img_dir, pp

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, i):
        return torch.from_numpy(preprocess(image_path(self.img_dir, self.keys[i]), self.pp))


@torch.no_grad()
def encode(model, pix: torch.Tensor, check: bool = False) -> tuple[torch.Tensor, float | None]:
    out = model(pixel_values=pix, output_hidden_states=True)
    vm = model.vision_model if hasattr(model, "vision_model") else model
    feats = vm.post_layernorm(out.hidden_states[-1])
    diff = (feats - out.last_hidden_state).abs().max().item() if check else None
    return feats, diff


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--shard", type=int, default=0)
    ap.add_argument("--nshards", type=int, default=1)
    ap.add_argument("--vision-id", default="google/siglip-base-patch16-224")
    ap.add_argument("--batch", type=int, default=64)
    ap.add_argument("--workers", type=int, default=2)
    a = ap.parse_args()
    from transformers import SiglipVisionModel

    scratch = Path(os.environ.get("PA3_SCRATCH", "/tmp/pa3_scratch"))
    feat_dir = scratch / "features"
    feat_dir.mkdir(parents=True, exist_ok=True)
    keys_all = json.loads((output_root() / "data" / "images.json").read_text())
    keys_all = keys_all["train"] + keys_all["val"]
    keys = keys_all[a.shard::a.nshards]
    done = feat_dir / f"shard{a.shard}of{a.nshards}.json"
    if done.exists() and json.loads(done.read_text()).get("keys") == keys:
        print(f"[features] shard {a.shard} already cached", flush=True)
        return
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    t0 = time.time()
    pp = load_preproc(a.vision_id)
    model, info = SiglipVisionModel.from_pretrained(a.vision_id, dtype=torch.float32, output_loading_info=True)
    assert not info["missing_keys"], f"SigLIP vision weights missing from checkpoint: {sorted(info['missing_keys'])[:5]}"
    model = model.to(dev).eval()
    for p in model.parameters():
        p.requires_grad_(False)
    grid = model.config.image_size // model.config.patch_size
    n_tok, dim = grid * grid, model.config.hidden_size
    assert (n_tok, dim) == (196, 768), (n_tok, dim)
    arr = np.lib.format.open_memmap(feat_dir / f"shard{a.shard}of{a.nshards}.npy", mode="w+", dtype=np.float16,
                                    shape=(len(keys), n_tok, dim))
    dl = torch.utils.data.DataLoader(_DS(keys, scratch / "images", pp), batch_size=a.batch, shuffle=False,
                                     num_workers=a.workers, pin_memory=dev.type == "cuda")
    j, max_abs, ln_diff, n_nonfinite = 0, 0.0, None, 0
    for b, pix in enumerate(dl):
        feats, d = encode(model, pix.to(dev), check=(b == 0))
        if d is not None:
            ln_diff = d
            assert d < 1e-3, f"post_layernorm(hidden_states[-1]) != last_hidden_state (max diff {d})"
        f16 = feats.half()
        n_nonfinite += int((~torch.isfinite(f16)).sum())
        max_abs = max(max_abs, feats.abs().max().item())
        arr[j:j + len(f16)] = f16.cpu().numpy()
        j += len(f16)
        if b % 20 == 0:
            print(f"[features] shard {a.shard}: {j}/{len(keys)} images", flush=True)
    arr.flush()
    assert j == len(keys) and n_nonfinite == 0, (j, n_nonfinite)
    secs = time.time() - t0
    write_json(done, {"keys": keys, "n": len(keys), "shape": [len(keys), n_tok, dim], "dtype": "float16"})
    write_json(output_root() / "features" / f"cache_shard{a.shard}.json", {
        "n_images": len(keys), "seconds": secs, "images_per_s": len(keys) / max(secs, 1e-9),
        "max_abs_feature": max_abs, "ln_check_max_diff": ln_diff, "nonfinite_fp16": n_nonfinite,
        "preprocessing": pp, "encoder_dtype": "float32", "store_dtype": "float16",
        "gpu": torch.cuda.get_device_name(dev) if dev.type == "cuda" else "cpu", "memory": gpu_mem_stats(dev),
        "bytes": int(arr.nbytes)})
    print(f"[features] shard {a.shard} done: {len(keys)} images in {secs:.0f}s", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/features/store.py
"""Read-only access to the cached SigLIP features (+ optional 2x2 average pooling for 49 tokens)."""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import torch


def pool2x2(x: torch.Tensor) -> torch.Tensor:
    """[N, 196, D] (14x14 row-major) -> [N, 49, D]: mean of each spatial 2x2 block, row-major 7x7."""
    n, t, d = x.shape
    g = int(round(t ** 0.5))
    assert g * g == t and g % 2 == 0
    return x.view(n, g // 2, 2, g // 2, 2, d).mean(dim=(2, 4)).reshape(n, (g // 2) ** 2, d)


class FeatureStore:
    def __init__(self, feat_dir: str | Path, pool: int = 1):
        feat_dir = Path(feat_dir)
        self.pool = pool
        self.arrays, self.index = [], {}
        metas = sorted(feat_dir.glob("shard*of*.json"))
        assert metas, f"no feature shards in {feat_dir}"
        for s, m in enumerate(metas):
            meta = json.loads(m.read_text())
            self.arrays.append(np.load(m.with_suffix(".npy"), mmap_mode="r"))
            for r, k in enumerate(meta["keys"]):
                self.index[k] = (s, r)
        self.n_vis = 196 // (pool * pool)

    def __contains__(self, key: str) -> bool:
        return key in self.index

    def get(self, keys: list[str]) -> torch.Tensor:
        """FP32 tensor [len(keys), n_vis, 768]."""
        x = torch.from_numpy(np.stack([self.arrays[s][r] for s, r in (self.index[k] for k in keys)])).float()
        return pool2x2(x) if self.pool == 2 else x


class RandomFeatures:
    """Deterministic stand-in features keyed by image id (CPU tests only)."""

    def __init__(self, pool: int = 1, dim: int = 768):
        self.pool, self.dim = pool, dim
        self.n_vis = 196 // (pool * pool)

    def __contains__(self, key: str) -> bool:
        return True

    def get(self, keys: list[str]) -> torch.Tensor:
        out = []
        for k in keys:
            g = torch.Generator().manual_seed(_kseed(k))
            out.append(torch.randn(196, self.dim, generator=g))
        x = torch.stack(out)
        return pool2x2(x) if self.pool == 2 else x


def _kseed(k: str) -> int:
    import hashlib

    return int(hashlib.sha256(k.encode()).hexdigest()[:8], 16)


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/models/__init__.py



In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/models/build.py
"""Assemble tokenizer, prompt format, data bundle, feature source and the VLM from a run config."""
from __future__ import annotations

import os
from pathlib import Path

import torch

from pa3common.utils import output_root

from ..data.bundle import Bundle
from ..data.prompts import ChatFormat
from ..features.store import FeatureStore, RandomFeatures
from .connector import make_connector
from .vlm import VLM, load_lm, load_tokenizer


def device_dtype(cfg: dict) -> tuple[torch.device, torch.dtype]:
    if torch.cuda.is_available():
        return torch.device("cuda", 0), getattr(torch, cfg["dtype"])
    return torch.device("cpu"), torch.float32


def feature_source(cfg: dict):
    if cfg.get("features") == "random":
        return RandomFeatures(cfg["pool"])
    scratch = Path(os.environ.get("PA3_SCRATCH", "/tmp/pa3_scratch"))
    return FeatureStore(scratch / cfg["features_dir"], cfg["pool"])


def build(cfg: dict, device: torch.device, dtype: torch.dtype, grad_ckpt: bool | None = None):
    tok = load_tokenizer(cfg["model_id"])
    fmt = ChatFormat(tok, cfg["caps"])
    bundle = Bundle(output_root() / cfg["data_dir"], fmt)
    store = feature_source(cfg)
    lm = load_lm(cfg["model_id"], dtype, cfg["attn_impl"], cfg["grad_checkpointing"] if grad_ckpt is None else grad_ckpt)
    connector = make_connector(cfg["seed"], d_out=lm.config.hidden_size)
    vlm = VLM(lm, connector).to(device)
    vlm.connector.float()
    return tok, fmt, bundle, store, vlm


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/models/calibrate.py
"""Connector output-magnitude calibration (training data only).

Rule: s = median L2 norm of Qwen input-embedding rows over the text tokens that occur in the Stage-1
training sequences (prefix + prompt + caption + <|im_end|>, frequency weighted)
      / median per-token L2 norm of the *initialised* connector output on `n` seeded training images.
s is stored as the connector's fixed `scale` buffer (not trained).
"""
from __future__ import annotations

from collections import Counter

import numpy as np
import torch


def weighted_median(values: np.ndarray, weights: np.ndarray) -> float:
    o = np.argsort(values)
    v, w = values[o], weights[o]
    c = np.cumsum(w)
    return float(v[np.searchsorted(c, 0.5 * c[-1])])


@torch.no_grad()
def calibrate(connector, embed_weight: torch.Tensor, token_counts: Counter, get_feats, image_keys: list[str],
              n_images: int, seed: int, device: torch.device, chunk: int = 64) -> dict:
    ids = np.array(sorted(token_counts), dtype=np.int64)
    cnt = np.array([token_counts[i] for i in ids], dtype=np.float64)
    tnorm = embed_weight[torch.from_numpy(ids).to(embed_weight.device)].float().norm(dim=-1).cpu().numpy()
    text_med = weighted_median(tnorm, cnt)
    rng = np.random.default_rng([seed, 7])
    keys = sorted(rng.choice(sorted(set(image_keys)), size=min(n_images, len(set(image_keys))), replace=False).tolist())
    norms = []
    for i in range(0, len(keys), chunk):
        f = get_feats(keys[i:i + chunk]).to(device, torch.float32)
        norms.append(connector.raw(f).float().norm(dim=-1).flatten().cpu())
    vn = torch.cat(norms).numpy()
    vis_med = float(np.median(vn))
    s = text_med / vis_med
    connector.scale.fill_(s)
    return {"rule": "s = weighted median ||E[t]|| over Stage-1 training text tokens / median per-token "
                    "||g_phi(F)|| of the initialised connector on seeded training images",
            "scale": s, "text_median_norm": text_med, "visual_median_norm_before": vis_med,
            "visual_median_norm_after": vis_med * s, "n_text_tokens": int(cnt.sum()), "n_unique_text_tokens": len(ids),
            "text_norm_quantiles": {q: weighted_quantile(tnorm, cnt, q) for q in (0.1, 0.5, 0.9)},
            "visual_norm_quantiles_before": {q: float(np.quantile(vn, q)) for q in (0.1, 0.5, 0.9)},
            "n_images": len(keys), "n_visual_tokens": int(vn.size), "image_keys": keys}


def weighted_quantile(values: np.ndarray, weights: np.ndarray, q: float) -> float:
    o = np.argsort(values)
    v, w = values[o], weights[o]
    c = np.cumsum(w)
    return float(v[np.searchsorted(c, q * c[-1])])


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/models/connector.py
"""Connector g_phi: Linear 768 -> 1024, GELU, Linear 1024 -> 1024 (trainable throughout), followed by a fixed,
non-trainable calibration multiplier s (buffer `scale`, set from training data before Stage 1)."""
from __future__ import annotations

import torch
from torch import nn

from pa3common.utils import stable_int_seed


class Connector(nn.Module):
    def __init__(self, d_in: int = 768, d_hidden: int = 1024, d_out: int = 1024):
        super().__init__()
        self.fc1 = nn.Linear(d_in, d_hidden)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(d_hidden, d_out)
        self.register_buffer("scale", torch.ones((), dtype=torch.float32))

    def raw(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(self.act(self.fc1(x)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.raw(x) * self.scale


def make_connector(seed: int, d_out: int = 1024) -> Connector:
    """PyTorch default Linear init drawn from a dedicated seeded generator (independent of global RNG use)."""
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(stable_int_seed(seed, "connector"))
        return Connector(d_out=d_out)


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/models/vlm.py
"""SigLIP-features -> connector -> Qwen3 wrapper.

Visual embeddings are spliced into the frozen input embeddings with an out-of-place masked_scatter, so
autograd reaches the connector although every Qwen parameter has requires_grad=False (the LM forward is
NOT under no_grad).  The LM head is applied only at supervised positions (shifted labels != -100).
"""
from __future__ import annotations

import torch
import torch.nn.functional as F
from torch import nn

from ..data.collate import Batch
from .connector import Connector


def load_tokenizer(model_id: str):
    from transformers import AutoTokenizer

    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token_id is None:
        tok.pad_token = "<|endoftext|>"
    return tok


def load_lm(model_id: str, dtype: torch.dtype, attn_impl: str = "sdpa", grad_ckpt: bool = True) -> nn.Module:
    from transformers import AutoModelForCausalLM

    lm = AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype, attn_implementation=attn_impl)
    lm.config.use_cache = False
    for p in lm.parameters():
        p.requires_grad_(False)
    if grad_ckpt:
        lm.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    return lm


class VLM(nn.Module):
    def __init__(self, lm: nn.Module, connector: Connector):
        super().__init__()
        self.lm = lm
        self.connector = connector

    @property
    def body(self) -> nn.Module:
        return self.lm.model

    def embed(self, b: Batch) -> torch.Tensor:
        emb = self.lm.get_input_embeddings()(b.ids)
        if b.feats is None:
            assert not bool(b.vis.any())
            return emb
        v = self.connector(b.feats.to(self.connector.fc1.weight.device))
        assert v.shape[0] * v.shape[1] == int(b.vis.sum()), "visual slot count mismatch"
        return emb.masked_scatter(b.vis.unsqueeze(-1).expand_as(emb), v.to(emb.dtype))

    def hidden(self, b: Batch) -> torch.Tensor:
        out = self.body(inputs_embeds=self.embed(b), attention_mask=b.attn, position_ids=b.pos, use_cache=False)
        return out.last_hidden_state

    def supervised_nll(self, b: Batch) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """(token NLL fp32, row index, is-termination) for every supervised token (next-token prediction)."""
        h = self.hidden(b)
        lab = b.labels[:, 1:]
        sel = lab != -100
        logits = self.lm.lm_head(h[:, :-1][sel]).float()
        nll = F.cross_entropy(logits, lab[sel], reduction="none")
        rows = sel.nonzero(as_tuple=True)[0]
        return nll, rows, b.term[:, 1:][sel]


def per_example_mean(nll: torch.Tensor, rows: torch.Tensor, n_rows: int, keep: torch.Tensor | None = None
                     ) -> torch.Tensor:
    """Mean token NLL per example (rows without kept tokens -> nan)."""
    if keep is not None:
        nll, rows = nll[keep], rows[keep]
    s = torch.zeros(n_rows, dtype=nll.dtype, device=nll.device).index_add_(0, rows, nll)
    c = torch.zeros(n_rows, dtype=nll.dtype, device=nll.device).index_add_(0, rows, torch.ones_like(nll))
    return s / c


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/orchestrate.py
"""Kaggle pipeline for Task 2 (deadline-aware, resumable across sessions).

Order: [optional CPU tests] -> hub prefetch -> data selection + image download -> label audit ->
SigLIP feature cache (two shards, one per GPU; separate processes, timed separately) -> preflight (3 updates
per stage type for both runs: s/update, memory, FP16 sanity) -> training (GPU0: vt196, GPU1: vt49,
concurrently) -> final evaluation (one run per GPU) -> report assets -> results zip.

Each step is skipped when its outputs already exist (including outputs of an earlier session attached under
/kaggle/input, which are merged into the output root first).  Training pauses itself with a resumable
checkpoint before the session budget runs out; the notebook then ends normally so Kaggle keeps the outputs.
"""
from __future__ import annotations

import json
import os
import shutil
import sys
import time
import zipfile
from pathlib import Path

from pa3common.kaggle import Deadline, copytree_merge, discover_prior_roots, disk_report, n_gpus, run_on_gpus, \
    scratch_root, sh
from pa3common.utils import write_json

TASK = Path(__file__).resolve().parents[2]
COMMON = TASK.parent / "common"
PY = sys.executable
OUT_NAME = "pa3_task2_outputs"
STATE: dict = {}


def _env() -> dict:
    return {"PYTHONPATH": os.pathsep.join([str(TASK / "src"), str(COMMON)]), "TOKENIZERS_PARALLELISM": "false"}


def setup_env(output_root: str, budget_hours: float = 11.25) -> dict:
    R = Path(output_root)
    R.mkdir(parents=True, exist_ok=True)
    os.environ["PA3_OUTPUT_ROOT"] = str(R)
    os.environ.update(_env())
    dl = Deadline(budget_hours)
    scratch = scratch_root("pa3_task2_scratch")
    prior = [p for p in discover_prior_roots(OUT_NAME) if Path(p).resolve() != R.resolve()]
    for p in prior:
        print(f"[setup] merging earlier-session outputs from {p}", flush=True)
        copytree_merge(Path(p), R, skip=("report_assets",))
    os.environ["PA3_PRIOR_ROOTS"] = os.pathsep.join(prior)
    STATE.update(deadline=dl, root=R, scratch=scratch, logs=R / "logs", steps={})
    info = {"output_root": str(R), "scratch": str(scratch), "gpus": n_gpus(), "prior_roots": prior, "budget": str(dl)}
    print(json.dumps(info, indent=1), flush=True)
    return info


def _status(step: str, state: str, **kw) -> None:
    STATE["steps"][step] = {"state": state, "time": time.strftime("%H:%M:%S"), **kw}
    write_json(STATE["root"] / "status.json", {"steps": STATE["steps"], "deadline": str(STATE["deadline"])})


def _run(step: str, cmd: list, tries: int = 2, fatal: bool = True) -> bool:
    t0 = time.time()
    for i in range(tries):
        rc = sh(cmd, STATE["logs"] / f"{step}.log", env=_env(), cwd=TASK)
        if rc == 0:
            _status(step, "done", seconds=round(time.time() - t0))
            return True
        print(f"!! {step} failed (rc={rc}), attempt {i + 1}/{tries}", flush=True)
    _status(step, "failed", seconds=round(time.time() - t0))
    if fatal:
        raise RuntimeError(f"{step} failed — see {STATE['logs'] / (step + '.log')}")
    return False


def _features_done() -> bool:
    d = STATE["scratch"] / "features"
    keys = json.loads((STATE["root"] / "data" / "images.json").read_text())
    need = set(keys["train"]) | set(keys["val"])
    have: set = set()
    for m in d.glob("shard*of2.json"):
        have |= set(json.loads(m.read_text())["keys"])
    return need <= have


def _run_status(run: str) -> str:
    p = STATE["root"] / "runs" / run / "status.json"
    return json.loads(p.read_text())["state"] if p.exists() else "todo"


def stage_auto(run_cpu_tests: bool = False, run_49: bool = True) -> dict:
    R, dl = STATE["root"], STATE["deadline"]
    runs = ["vt196"] + (["vt49"] if run_49 else [])
    if run_cpu_tests:
        _run("cpu_tests", [PY, "-m", "pytest", "-q", "-x", str(TASK / "tests")], tries=1)
    _run("prefetch", [PY, "-m", "pa3vlm.prefetch"], tries=3)
    _run("data", [PY, "-m", "pa3vlm.data.splits"], tries=3)
    if not (R / "data" / "label_audit" / "summary.json").exists():
        _run("label_audit", [PY, str(TASK / "scripts" / "label_audit.py")], tries=1, fatal=False)
    disk_report()
    if not _features_done():
        t0 = time.time()
        jobs = [(f"features{i}", [PY, "-m", "pa3vlm.features.siglip_cache", "--shard", str(i), "--nshards", "2"],
                 STATE["logs"] / f"features{i}.log") for i in range(2)]
        rcs = run_on_gpus(jobs, env_extra=_env())
        assert all(v == 0 for v in rcs.values()) and _features_done(), f"feature caching failed: {rcs}"
        write_json(R / "features" / "cache_wall.json", {"wall_seconds": time.time() - t0, "n_gpus": n_gpus()})
        _status("features", "done", seconds=round(time.time() - t0))
    todo = [r for r in runs if _run_status(r) != "done"]
    pf = [(f"preflight_{r}", [PY, "-m", "pa3vlm.train.loop", "--config", r, "--preflight", "3"],
           STATE["logs"] / f"preflight_{r}.log") for r in todo
          if not (R / "preflight" / r / "preflight.json").exists()]
    if pf:
        rcs = run_on_gpus(pf, env_extra=_env())
        _status("preflight", "done" if all(v == 0 for v in rcs.values()) else "failed", rcs=rcs)
        assert all(v == 0 for v in rcs.values()), f"preflight failed: {rcs} (see logs/preflight_*.log)"
    for attempt in range(3):
        todo = [r for r in runs if _run_status(r) != "done"]
        if not todo:
            break
        jobs = [(f"train_{r}", [PY, "-m", "pa3vlm.train.loop", "--config", r], STATE["logs"] / f"train_{r}.log")
                for r in todo]
        rcs = run_on_gpus(jobs, env_extra=_env(), kill_at=dl.end - 300)
        _status("train", "running", attempt=attempt + 1, rcs=rcs)
        if any(v == 3 for v in rcs.values()) or not dl.fits(1800, margin=0):
            break
    states = {r: _run_status(r) for r in runs}
    _status("train", "done" if all(v == "done" for v in states.values()) else "paused", runs=states)
    ev = [(f"final_{r}", [PY, "-m", "pa3vlm.eval.final", "--config", r], STATE["logs"] / f"final_{r}.log")
          for r in runs if states[r] == "done" and not (R / "runs" / r / "final_eval" / "summary.json").exists()]
    if ev and dl.fits(900, margin=120):
        rcs = run_on_gpus(ev, env_extra=_env(), kill_at=dl.end - 240)
        _status("final_eval", "done" if all(v == 0 for v in rcs.values()) else "failed", rcs=rcs)
    if dl.fits(120, margin=60):
        _run("report", [PY, str(TASK / "scripts" / "make_report_assets.py")], tries=1, fatal=False)
    zip_results()
    final = {"runs": states, "main_run_done": states.get("vt196") == "done",
             "state": "done" if all((R / "runs" / r / "final_eval" / "summary.json").exists() for r in runs)
             else "paused"}
    _status("pipeline", final["state"], **final)
    return final


def zip_results() -> Path:
    """results_task2.zip = everything in the output root except resumable checkpoints and logs of the zip itself."""
    R = STATE["root"]
    z = R / "results_task2.zip"
    with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(R.rglob("*")):
            if f.is_file() and f != z and "checkpoints" not in f.parts and f.suffix != ".tmp":
                zf.write(f, f.relative_to(R))
    print(f"[zip] {z} ({z.stat().st_size / 2 ** 20:.1f} MiB)", flush=True)
    return z


def clean_scratch() -> None:
    shutil.rmtree(STATE["scratch"], ignore_errors=True)


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/prefetch.py
"""Download hub checkpoints once (with retries) before worker processes start."""
from __future__ import annotations

from pa3common.kaggle import retry


def main() -> None:
    from huggingface_hub import snapshot_download

    for rid, pats in (("Qwen/Qwen3-0.6B", ["*.json", "*.safetensors", "*.txt", "merges.txt", "vocab.json"]),
                      ("google/siglip-base-patch16-224", ["*.json", "*.safetensors", "*.model", "*.txt"])):
        p = retry(lambda: snapshot_download(rid, allow_patterns=pats), 4, 20, f"snapshot {rid}")
        print(f"[prefetch] {rid} -> {p}", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/train/__init__.py



In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/train/checkpoint.py
"""Resumable checkpoints (connector + LoRA factors + AdamW + GradScaler + RNG + progress) and stage weights.

Only trainable state is stored (the frozen backbone is reloaded from the hub), so a checkpoint is
~80 MB.  The newest `keep` resumable checkpoints are kept; every save/deletion is logged with its size.
"""
from __future__ import annotations

import os
import time
from pathlib import Path

import torch

from pa3common.lora import lora_state
from pa3common.utils import JsonlLogger, get_rng_state


def save_resumable(ck_dir: Path, update: int, stage: int, vlm, opt, scaler, extra: dict, keep: int,
                   log: JsonlLogger) -> Path:
    ck_dir.mkdir(parents=True, exist_ok=True)
    path = ck_dir / f"resume_{update:05d}.pt"
    tmp = path.with_suffix(".tmp")
    torch.save({"update": update, "stage": stage, "connector": {k: v.detach().cpu() for k, v in
                                                                 vlm.connector.state_dict().items()},
                "lora": lora_state(vlm.lm), "optimizer": opt.state_dict(), "scaler": scaler.state_dict(),
                "rng": get_rng_state(), "extra": extra, "time": time.time()}, tmp)
    os.replace(tmp, path)
    log.log({"update": update, "event": "save", "path": path.name, "bytes": path.stat().st_size})
    olds = sorted(ck_dir.glob("resume_*.pt"))[:-keep] if keep > 0 else []
    for o in olds:
        log.log({"update": update, "event": "delete", "path": o.name, "bytes": o.stat().st_size})
        o.unlink()
    return path


def latest_resumable(ck_dir: Path) -> Path | None:
    c = sorted(ck_dir.glob("resume_*.pt")) if ck_dir.exists() else []
    return c[-1] if c else None


def save_weights(path: Path, vlm, meta: dict) -> int:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    torch.save({"connector": {k: v.detach().cpu() for k, v in vlm.connector.state_dict().items()},
                "lora": lora_state(vlm.lm), "meta": meta}, tmp)
    os.replace(tmp, path)
    return path.stat().st_size


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/train/loop.py
"""Three-stage curriculum training of one VLM on one GPU (microbatch 8 x accumulation 2 = 16).

    python -m pa3vlm.train.loop --config vt196 [--set key=value ...] [--preflight N]

Stage 1 (caption alignment): connector only.  Stages 2-3: connector + LoRA (q/k/v/o, r16, alpha 32, dropout
0.05), attached with B = 0 at the stage boundary.  Fresh AdamW per stage (FP32 states), FP16 autocast with
dynamic loss scaling (unscale -> clip 1.0 -> step); a skipped (inf/nan) update is logged, AdamW's step count
does not advance, the schedule does.  Loss = mean over the 16 examples of each example's token-mean CE.

Exit codes: 0 finished, 3 paused (session budget / stop_after_update) with a resumable checkpoint saved.
"""
from __future__ import annotations

import argparse
import math
import os
import sys
import time
from pathlib import Path

import torch

from pa3common.lora import inject_lora, load_lora_state, lora_modules
from pa3common.utils import JsonlLogger, count_params, gpu_mem_stats, output_root, read_json, seed_everything, \
    set_rng_state, write_json

from ..config import load_config
from ..data.collate import collate
from ..data.sampler import CurriculumSampler, StageSpec
from ..eval.dev import amp_ctx, dev_eval
from ..models.build import build, device_dtype
from ..models.calibrate import calibrate
from ..models.vlm import per_example_mean
from .checkpoint import latest_resumable, save_resumable, save_weights
from .schedule import lr_at


def stage_bounds(cfg: dict) -> list[tuple[int, int]]:
    out, s = [], 0
    for st in cfg["stages"]:
        out.append((s, s + st["updates"]))
        s += st["updates"]
    return out


def dev_points(cfg: dict) -> list[int]:
    total = stage_bounds(cfg)[-1][1]
    pts = set(range(0, total + 1, cfg["dev_every"])) | {e for _, e in stage_bounds(cfg)}
    return sorted(pts)


def make_optimizer(vlm, cfg: dict, st: dict) -> torch.optim.Optimizer:
    c = vlm.connector
    groups = [
        {"name": "connector_w", "params": [c.fc1.weight, c.fc2.weight], "weight_decay": cfg["weight_decay"],
         "peak": st["lr_connector"]},
        {"name": "connector_b", "params": [c.fc1.bias, c.fc2.bias], "weight_decay": 0.0, "peak": st["lr_connector"]},
    ]
    if st["lr_lora"] is not None:
        lp = [p for m in lora_modules(vlm.lm).values() for p in (m.lora_A, m.lora_B)]
        assert lp, "LoRA must be attached before Stage 2"
        groups.append({"name": "lora", "params": lp, "weight_decay": cfg["weight_decay"], "peak": st["lr_lora"]})
    for g in groups:
        g["lr"] = 0.0
    trainable = {id(p) for p in vlm.parameters() if p.requires_grad}
    in_opt = {id(p) for g in groups for p in g["params"]}
    assert trainable == in_opt, "optimizer must cover exactly the trainable parameters"
    return torch.optim.AdamW(groups, betas=tuple(cfg["betas"]), eps=cfg["eps"])


def attach_lora(vlm, cfg: dict) -> None:
    lc = cfg["lora"]
    inject_lora(vlm.lm, lc["r"], lc["alpha"], lc["dropout"], cfg["seed"], tuple(lc["targets"]))


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--set", nargs="*", default=[])
    ap.add_argument("--preflight", type=int, default=0, help="run N updates of each stage type, save nothing")
    a = ap.parse_args(argv)
    cfg = load_config(a.config, a.set)
    device, dtype = device_dtype(cfg)
    seed_everything(cfg["seed"])
    out = output_root() / "runs" / cfg["run_name"] if not a.preflight else output_root() / "preflight" / cfg["run_name"]
    out.mkdir(parents=True, exist_ok=True)
    t_start = time.time()
    tok, fmt, bundle, store, vlm = build(cfg, device, dtype)
    n_vis = store.n_vis
    get_feats = store.get
    bounds = stage_bounds(cfg)
    total = bounds[-1][1]
    sampler = CurriculumSampler(cfg["seed"], bundle.n_captions, bundle.pool_sizes(), cfg["micro_bs"],
                                cfg["grad_accum"], cfg["caption_passes"])
    assert sampler.stage1_updates() == cfg["stages"][0]["updates"], "Stage 1 must be exactly the caption passes"
    eff = cfg["micro_bs"] * cfg["grad_accum"]
    if not a.preflight:
        write_json(out / "config.json", cfg)

    # ------------------------------------------------------------------ calibration (training data only)
    cal_path = out / "calibration.json"
    if cal_path.exists():
        vlm.connector.scale.fill_(read_json(cal_path)["scale"])
    else:
        t0 = time.time()
        rep = calibrate(vlm.connector, vlm.lm.get_input_embeddings().weight, bundle.stage1_token_counts(), get_feats,
                        [r["image"] for r in bundle.manifest("captions_train")], cfg["calib_images"], cfg["seed"],
                        device)
        rep["seconds"] = time.time() - t0
        rep["n_visual_tokens_per_image"] = n_vis
        write_json(cal_path, rep)
    print(f"[train] calibration scale s = {float(vlm.connector.scale):.6g}", flush=True)

    # ------------------------------------------------------------------ resume
    ck_dir = out / "checkpoints"
    ck = latest_resumable(ck_dir) if not a.preflight else None
    g = 0
    state = None
    if ck is not None:
        state = torch.load(ck, map_location="cpu", weights_only=False)
        g = state["update"]
        if state["lora"]:
            attach_lora(vlm, cfg)
            load_lora_state(vlm.lm, state["lora"])
        vlm.connector.load_state_dict(state["connector"])
        set_rng_state(state["rng"])
        print(f"[train] resumed from {ck.name} (update {g})", flush=True)
    vlm.to(device)
    train_log = JsonlLogger(out / "train_log.jsonl", not a.preflight, truncate_after=g)
    dev_log = JsonlLogger(out / "dev_metrics.jsonl", not a.preflight, truncate_after=g)
    ck_log = JsonlLogger(out / "checkpoints.jsonl", not a.preflight)
    ev_log = JsonlLogger(out / "events.jsonl", not a.preflight)
    ev_log.log({"update": g, "event": "start", "resumed_from": ck.name if ck else None, "device": str(device),
                "gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else "cpu",
                "setup_seconds": time.time() - t_start})
    scaler = torch.amp.GradScaler("cuda", init_scale=cfg["init_scale"], enabled=device.type == "cuda")
    if state is not None:
        scaler.load_state_dict(state["scaler"])
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    dpts = set(dev_points(cfg)) if not a.preflight else set()
    done_dev = {r["update"] for r in _read_jsonl(out / "dev_metrics.jsonl")}
    deadline = float(os.environ.get("PA3_DEADLINE_UNIX", "inf"))
    margin = cfg["deadline_margin_min"] * 60
    stop_after = cfg.get("stop_after_update")
    timing = {"train_s": 0.0, "dev_s": 0.0, "ckpt_s": 0.0}

    def run_dev(update: int, stage: int) -> None:
        if update in done_dev or update not in dpts:
            return
        t0 = time.time()
        row = dev_eval(vlm, bundle, get_feats, n_vis, fmt, device, cfg["eval_bs"], cfg["max_new_tokens"], update, stage)
        row["seconds"] = time.time() - t0
        timing["dev_s"] += row["seconds"]
        dev_log.log(row)
        done_dev.add(update)
        print(f"[dev] u={update} cap={row['caption_loss']:.3f} "
              f"g_dNLL={row['grounding/all/delta_nll_swap']:+.3f} v_dNLL={row['vqa/all/delta_nll_swap']:+.3f} "
              f"g_acc={row['grounding/all/acc_greedy']:.3f} v_acc={row['vqa/all/acc_greedy']:.3f}", flush=True)

    run_dev(0, 0)
    vlm.train()
    preflight_rows = []
    for si, st in enumerate(cfg["stages"]):
        start, end = bounds[si]
        if a.preflight:
            start, end = g, g + a.preflight
        if g >= end:
            continue
        if st["lr_lora"] is not None and not lora_modules(vlm.lm):
            attach_lora(vlm, cfg)
            vlm.to(device)
            ev_log.log({"update": g, "event": "lora_attached", "n_lora_params": count_params(vlm, True) -
                        count_params(vlm.connector, True)})
        n_train = count_params(vlm, True)
        ev_log.log({"update": g, "event": "stage_start", "stage": si + 1, "name": st["name"],
                    "trainable_params": n_train, "connector_params": count_params(vlm.connector, True),
                    "lora_params": n_train - count_params(vlm.connector, True),
                    "frozen_params": count_params(vlm) - n_train})
        opt = make_optimizer(vlm, cfg, st)
        if state is not None and state["stage"] == si and state["update"] < end:
            opt.load_state_dict(state["optimizer"])
        state = None
        spec = StageSpec(si + 1, st["updates"], st["mix"])
        T = st["updates"]
        while g < end:
            u = g - start
            t0 = time.time()
            for grp in opt.param_groups:
                grp["lr"] = lr_at(min(u + 1, T), T, grp["peak"], cfg["warmup_frac"], cfg["final_lr_frac"])
            mbs, counts = sampler.update(spec, u)
            loss_sum, n_tok = 0.0, 0
            for mb in mbs:
                encs = [bundle.item(it) for it in mb]
                b = collate(encs, [e.image for e in encs], get_feats, n_vis, fmt.pad_id, "right", True).to(device)
                with amp_ctx(device):
                    nll, rows, _ = vlm.supervised_nll(b)
                ex = per_example_mean(nll, rows, len(encs))
                loss = ex.sum() / eff
                scaler.scale(loss).backward()
                loss_sum += float(ex.detach().sum())
                n_tok += int(rows.numel())
            scaler.unscale_(opt)
            params = [p for grp in opt.param_groups for p in grp["params"]]
            gn = float(torch.nn.utils.clip_grad_norm_(params, cfg["clip_norm"]))
            scale_before = scaler.get_scale()
            scaler.step(opt)
            scaler.update()
            skipped = scaler.is_enabled() and scaler.get_scale() < scale_before
            opt.zero_grad(set_to_none=True)
            g += 1
            dt = time.time() - t0
            timing["train_s"] += dt
            row = {"update": g, "stage": si + 1, "loss": loss_sum / eff, "grad_norm": gn,
                   "grad_norm_finite": math.isfinite(gn), "skipped": bool(skipped), "loss_scale": scaler.get_scale(),
                   "lr": {grp["name"]: grp["lr"] for grp in opt.param_groups}, "mix": counts,
                   "supervised_tokens": n_tok, "seconds": dt}
            if device.type == "cuda":
                row["peak_mem_gib"] = torch.cuda.max_memory_allocated(device) / 2 ** 30
            if a.preflight:
                preflight_rows.append(row)
                print(f"[preflight] stage {si + 1} update {g - start}: loss {row['loss']:.3f} {dt:.2f}s", flush=True)
                continue
            train_log.log(row)
            if g % 50 == 0 or g == end:
                print(f"[train] u={g} stage={si + 1} loss={row['loss']:.4f} gn={gn:.3f} "
                      f"lr={list(row['lr'].values())} {dt:.2f}s/upd skipped={skipped}", flush=True)
            just_saved = False
            if g % cfg["ckpt_every"] == 0 or g == end:
                t1 = time.time()
                save_resumable(ck_dir, g, si, vlm, opt, scaler, {"timing": timing}, cfg["keep_resumable"], ck_log)
                timing["ckpt_s"] += time.time() - t1
                just_saved = True
            if g == end:
                sz = save_weights(out / "weights" / f"stage{si + 1}.pt", vlm, {"update": g, "stage": si + 1})
                ck_log.log({"update": g, "event": "stage_weights", "stage": si + 1, "bytes": sz})
            run_dev(g, si + 1)
            pause = time.time() > deadline - margin or (stop_after is not None and g >= stop_after)
            if pause and g < total:
                if not just_saved:
                    save_resumable(ck_dir, g, si, vlm, opt, scaler, {"timing": timing}, cfg["keep_resumable"], ck_log)
                ev_log.log({"update": g, "event": "paused", "timing": timing, "memory": gpu_mem_stats(device)})
                write_json(out / "status.json", {"state": "paused", "update": g, "total": total})
                print(f"[train] paused at update {g} (resumable)", flush=True)
                return 3
    if a.preflight:
        write_json(out / "preflight.json", {"rows": preflight_rows, "memory": gpu_mem_stats(device),
                                            "s_per_update": {f"stage{r['stage']}": r["seconds"] for r in preflight_rows}})
        return 0
    save_weights(out / "weights" / "final.pt", vlm, {"update": g})
    res = {"update": g, "timing": timing, "memory": gpu_mem_stats(device), "wall_s_this_session": time.time() - t_start}
    ev_log.log({"update": g, "event": "finished", **res})
    write_json(out / "status.json", {"state": "done", "update": g, "total": total})
    print(f"[train] finished {g} updates", flush=True)
    return 0


def _read_jsonl(path: Path) -> list[dict]:
    import json

    if not path.exists():
        return []
    return [json.loads(l) for l in path.read_text().splitlines() if l.strip()]


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/src/pa3vlm/train/schedule.py
"""Per-stage learning-rate schedule: linear warmup over w = ceil(warmup_frac * T) updates starting at peak/w
(update 1), then cosine decay reaching exactly final_frac * peak at the stage's last update T."""
from __future__ import annotations

import math


def warmup_updates(T: int, frac: float) -> int:
    return max(1, math.ceil(frac * T - 1e-9))


def lr_at(t: int, T: int, peak: float, warmup_frac: float = 0.05, final_frac: float = 0.1) -> float:
    """t = 1..T (1-based update index within the stage)."""
    assert 1 <= t <= T
    w = warmup_updates(T, warmup_frac)
    if t <= w:
        return peak * t / w
    prog = (t - w) / max(1, T - w)
    return peak * (final_frac + (1 - final_frac) * 0.5 * (1 + math.cos(math.pi * prog)))


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/scripts/build_notebook.py
"""Generate task_2/notebooks/pa3_task2_kaggle.ipynb — a self-contained Kaggle notebook.

Every source file of task_2/ (configs, src, scripts, tests, README) and the shared common/pa3common modules is
embedded as a `%%writefile` cell, followed by an integrity check (sha256 of each written file), dependency
installation and the deadline-aware AUTO pipeline.  No git clone is needed.
"""
from __future__ import annotations

import hashlib
import json
from pathlib import Path

TASK = Path(__file__).resolve().parents[1]
REPO = TASK.parent
CODE_DIR = "/kaggle/working/ATML-PA3"
COMMON_FILES = ["__init__.py", "utils.py", "kaggle.py", "lora.py"]

INTRO = r"""# ATML PA3 · Task 2 — Vision-Language Model and Visual Grounding (SigLIP → MLP → Qwen3-0.6B)
**CS-6304 / EE-5102 · Danish Rasool (28100233)** · self-contained notebook (all code is written to disk by the cells below)

### Before running
1. **Settings → Accelerator: GPU T4 ×2**, **Internet: On**.
2. **Save Version → Save & Run All (Commit)** so the run continues with the browser closed and outputs are saved.
3. Optional: attach a dataset with the COCO 2014 / VQAv2 annotation JSONs (and `train2014`/`val2014` image folders)
   if a download host is unreachable — attached copies under `/kaggle/input` are used automatically.

### What happens (`AUTO`)
hub prefetch → data selection (seed 6304; COCO captions, annotation-derived presence/counting pairs, VQAv2
complementary pairs; IDs saved) + image download → label audit sheets → SigLIP feature cache (FP16, one shard per
GPU, timed separately; SigLIP then leaves GPU memory) → preflight (3 updates per stage type) → **training: GPU0 =
196-token main run, GPU1 = 49-token run, concurrently** (4,500 updates each, dev eval every 500 + stage ends,
resumable checkpoints every 250) → final evaluation under correct / complementary swap / mismatched / no image
(pair-bootstrap CIs) → report figures and tables → `results_task2.zip`.

Training pauses itself before the session budget (`BUDGET_HOURS`) runs out and the notebook ends normally, so
Kaggle keeps the outputs. If `status.json` says **paused**: Output → New Dataset, attach it to a new session, run
again — finished steps are skipped and training resumes exactly.
"""

SETTINGS = f'''# ============================ SETTINGS ============================
import os, time
os.environ.setdefault("PA3_SESSION_START", str(time.time()))   # the session budget starts now
BUDGET_HOURS = 11.25        # stop cleanly before Kaggle's 12 h limit
RUN_49_TOKENS = True        # optional 49-token experiment on GPU1 (concurrent with the main run)
RUN_CPU_TESTS = False       # run the CPU test suite first (~5 min)
CODE_DIR = "{CODE_DIR}"   # fixed: the %%writefile cells below write here
OUTPUT_ROOT = "/kaggle/working/pa3_task2_outputs"'''

INSTALL = '''# ============================ DEPENDENCIES ============================
import sys, subprocess
sys.path[:0] = [f"{CODE_DIR}/common", f"{CODE_DIR}/task_2/src"]
from pa3common.kaggle import pip_install
import torch
torch_before = torch.__version__
pip_install(["transformers==5.21.0", "safetensors", "matplotlib", "pytest", "pyyaml"])
out = subprocess.run([sys.executable, "-c", "import torch, transformers; print(torch.__version__, transformers.__version__)"],
                     capture_output=True, text=True)
print("kernel torch:", torch_before, "| subprocess torch/transformers:", out.stdout.strip(), out.stderr[-400:])
assert out.stdout.split()[0] == torch_before, "pip changed torch — restart the kernel and rerun"
gpus = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print("GPUs:", gpus)
if len(gpus) < 2:
    print("!! WARNING: fewer than 2 GPUs visible — the two runs will execute one after the other")'''

RUN = '''# ============================ RUN ============================
os.chdir(f"{CODE_DIR}/task_2")
from pa3vlm import orchestrate as O
info = O.setup_env(OUTPUT_ROOT, budget_hours=BUDGET_HOURS)
result = O.stage_auto(run_cpu_tests=RUN_CPU_TESTS, run_49=RUN_49_TOKENS)
print(result)'''

SHOW = '''# ============================ RESULTS ============================
import json
from pathlib import Path
from IPython.display import Markdown, Image, display
R = Path(OUTPUT_ROOT)
if (R / "status.json").exists():
    print(json.dumps(json.loads((R / "status.json").read_text()), indent=1))
T, F = R / "report_assets/tables", R / "report_assets/figures"
for t in sorted(T.glob("*.md")) if T.exists() else []:
    display(Markdown(f"**{t.name}**\\n\\n" + t.read_text()))
for f in sorted(F.glob("*.png")) if F.exists() else []:
    display(Image(filename=str(f)))'''

_N = [0]


def _cell(kind: str, src: str) -> dict:
    _N[0] += 1
    c = {"cell_type": kind, "id": f"cell-{_N[0]:03d}", "metadata": {}, "source": src.splitlines(keepends=True)}
    if kind == "code":
        c.update({"execution_count": None, "outputs": []})
    return c


def files() -> list[tuple[str, Path]]:
    out = [(f"common/pa3common/{f}", REPO / "common" / "pa3common" / f) for f in COMMON_FILES]
    for sub in ("configs", "src", "scripts", "tests"):
        for p in sorted((TASK / sub).rglob("*")):
            if p.is_file() and "__pycache__" not in p.parts and p.suffix in (".py", ".yaml"):
                out.append((f"task_2/{p.relative_to(TASK).as_posix()}", p))
    out.append(("task_2/README.md", TASK / "README.md"))
    return out


def build() -> dict:
    fl = files()
    dirs = sorted({str(Path(CODE_DIR) / Path(rel).parent) for rel, _ in fl})
    mk = "# ============================ WRITE THE CODE ============================\n" \
         "import os\nfor d in " + json.dumps(dirs, indent=1) + ":\n    os.makedirs(d, exist_ok=True)\n" \
         f"print('writing {len(fl)} files under {CODE_DIR}')"
    cells = [_cell("markdown", INTRO), _cell("code", SETTINGS), _cell("code", mk)]
    sums = {}
    for rel, p in fl:
        txt = p.read_text()
        assert "\r" not in txt
        if not txt.endswith("\n"):
            txt += "\n"
        sums[rel] = hashlib.sha256(txt.rstrip("\n").encode()).hexdigest()   # robust to trailing-newline handling
        cells.append(_cell("code", f"%%writefile {CODE_DIR}/{rel}\n{txt}"))
    check = ("# ============================ INTEGRITY CHECK ============================\n"
             "import hashlib, json\n"
             f"EXPECTED = json.loads({json.dumps(json.dumps(sums))})\n"
             "bad = [r for r, h in EXPECTED.items()\n"
             f"       if hashlib.sha256(open('{CODE_DIR}/' + r, 'rb').read().rstrip(b'\\n')).hexdigest() != h]\n"
             "assert not bad, f'files differ from the embedded sources: {bad}'\n"
             "print(f'{len(EXPECTED)} files written and verified')")
    cells += [_cell("code", check), _cell("code", INSTALL), _cell("code", RUN), _cell("code", SHOW)]
    return {"cells": cells,
            "metadata": {"kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
                         "language_info": {"name": "python"},
                         "kaggle": {"accelerator": "nvidiaTeslaT4", "isInternetEnabled": True, "isGpuEnabled": True}},
            "nbformat": 4, "nbformat_minor": 5}


if __name__ == "__main__":
    out = TASK / "notebooks" / "pa3_task2_kaggle.ipynb"
    out.parent.mkdir(exist_ok=True)
    nb = build()
    out.write_text(json.dumps(nb, indent=1, ensure_ascii=False) + "\n")
    print(f"wrote {out} ({len(nb['cells'])} cells, {len(files())} embedded files)")


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/scripts/label_audit.py
"""Label audit for the evaluation grounding pairs, made before any model is trained (independent of predictions).

    python scripts/label_audit.py

Writes data/label_audit/page_XX.png (each image with its question, target and the annotated boxes of the
asked category) and data/label_audit/summary.json (automated checks: per-pair categories, counts, instance
areas, crowd flags, caption mentions for negatives).
"""
from __future__ import annotations

import json
import os
import sys
import textwrap
from collections import defaultdict
from pathlib import Path

TASK = Path(__file__).resolve().parents[1]
sys.path[:0] = [str(TASK / "src"), str(TASK.parent / "common")]

from pa3common.utils import output_root, write_json  # noqa: E402
from pa3vlm.data.coco import captions_mention, index_captions  # noqa: E402
from pa3vlm.data.download import ensure_annotation, image_path  # noqa: E402


def main(per_page: int = 12) -> None:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.patches as patches
    import matplotlib.pyplot as plt
    from PIL import Image

    R = output_root()
    scratch = Path(os.environ.get("PA3_SCRATCH", "/tmp/pa3_scratch"))
    out = R / "data" / "label_audit"
    out.mkdir(parents=True, exist_ok=True)
    inst = json.loads(ensure_annotation("instances_val2014.json", scratch / "raw").read_text())
    caps = index_captions(json.loads(ensure_annotation("captions_val2014.json", scratch / "raw").read_text()))
    cname = {c["id"]: c["name"] for c in inst["categories"]}
    boxes = defaultdict(list)
    for a in inst["annotations"]:
        boxes[(a["image_id"], cname[a["category_id"]])].append((a["bbox"], a.get("iscrowd", 0), a["area"]))
    del inst
    pairs = []
    for name in ("grounding_dev", "grounding_test"):
        pairs += json.loads((R / "data" / f"{name}.json").read_text())
    summary = []
    for p in pairs:
        for e in p["examples"]:
            b = boxes.get((e["image_id"], p["category"]), [])
            summary.append({"pair_id": p["pair_id"], "kind": p["kind"], "category": p["category"], "image": e["image"],
                            "target": e["target"], "n_noncrowd": sum(1 for _, c, _ in b if not c),
                            "n_crowd": sum(1 for _, c, _ in b if c), "areas": sorted(round(x) for _, _, x in b),
                            "caption_mentions_category": captions_mention(p["category"],
                                                                          [c["caption"] for c in caps.get(e["image_id"], [])])})
    flags = [s for s in summary if (s["target"] == "no" and (s["n_noncrowd"] or s["caption_mentions_category"]))
             or (s["kind"] == "counting" and str(s["n_noncrowd"]) != s["target"])]
    write_json(out / "summary.json", {"n_pairs": len(pairs), "flags": flags, "examples": summary})
    for pg in range(0, len(pairs), per_page):
        chunk = pairs[pg:pg + per_page]
        rows = (len(chunk) + 1) // 2
        fig, axes = plt.subplots(rows, 4, figsize=(16, 3.6 * rows))
        axes = axes.reshape(rows, 4)
        for i, p in enumerate(chunk):
            for s, e in enumerate(p["examples"]):
                ax = axes[i // 2, (i % 2) * 2 + s]
                try:
                    ax.imshow(Image.open(image_path(scratch / "images", e["image"])).convert("RGB"))
                except OSError:
                    ax.text(0.5, 0.5, "missing", ha="center")
                for (x, y, w, h), crowd, _ in boxes.get((e["image_id"], p["category"]), []):
                    ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, lw=1.5, ec="red" if crowd else "lime"))
                ax.set_title(textwrap.shorten(f"{p['pair_id']}/{s} {p['question']} -> {e['target']}", 60), fontsize=7)
                ax.axis("off")
        for j in range(len(chunk), rows * 2):
            for s in (0, 1):
                axes[j // 2, (j % 2) * 2 + s].axis("off")
        fig.tight_layout()
        fig.savefig(out / f"page_{pg // per_page:02d}.png", dpi=70)
        plt.close(fig)
    print(f"[audit] {len(pairs)} pairs, {len(flags)} automated flags -> {out}", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/scripts/make_report_assets.py
"""Report figures/tables for Task 2 from $PA3_OUTPUT_ROOT (runs/vt196, runs/vt49, data, features).

    python scripts/make_report_assets.py

Writes report_assets/{figures,tables}/: learning curves with stage boundaries, final-result tables (copied
from each run's summary.md + a 196-vs-49 comparison), resources (feature caching separated from training),
the qualitative example sheet and its selection rule.
"""
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import numpy as np

TASK = Path(__file__).resolve().parents[1]
sys.path[:0] = [str(TASK / "src"), str(TASK.parent / "common")]

from pa3common.utils import output_root, write_json  # noqa: E402
from pa3vlm.eval import qualitative as Q  # noqa: E402

RUNS = ("vt196", "vt49")


def jl(p: Path) -> list[dict]:
    return [json.loads(l) for l in p.read_text().splitlines() if l.strip()] if p.exists() else []


def curves(run: str, R: Path, fig_dir: Path) -> None:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    tr, dv = jl(R / "runs" / run / "train_log.jsonl"), jl(R / "runs" / run / "dev_metrics.jsonl")
    if not tr:
        return
    cfg = json.loads((R / "runs" / run / "config.json").read_text())
    ends = np.cumsum([s["updates"] for s in cfg["stages"]])
    u = np.array([r["update"] for r in tr])
    loss = np.array([r["loss"] for r in tr])
    k = 25
    sm = np.convolve(loss, np.ones(k) / k, mode="valid") if len(loss) >= k else loss
    du = [r["update"] for r in dv]
    fig, ax = plt.subplots(2, 3, figsize=(16, 8.5))
    ax = ax.ravel()
    ax[0].plot(u, loss, alpha=0.25, lw=0.6, label="per update")
    ax[0].plot(u[len(u) - len(sm):], sm, lw=1.5, label=f"{k}-update mean")
    ax[0].set_title("training loss (example-weighted CE)")
    ax[1].plot(du, [r["caption_loss"] for r in dv], "o-", label="correct image")
    ax[1].plot(du, [r["caption_loss_noimage"] for r in dv], "s--", label="no image")
    ax[1].set_title("dev caption reference loss")
    for i, suite in ((2, "grounding"), (3, "vqa")):
        for cond, st in (("correct", "o-"), ("swap", "s--"), ("none", "^:")):
            ax[i].plot(du, [r[f"{suite}/all/nll_{cond}"] for r in dv], st, label=cond)
        ax[i].set_title(f"dev {suite}: reference-answer NLL")
    for suite, st in (("grounding", "o-"), ("vqa", "s-")):
        ax[4].plot(du, [r[f"{suite}/all/delta_nll_swap"] for r in dv], st, label=f"{suite} ΔNLL swap")
        ax[5].plot(du, [r[f"{suite}/all/acc_greedy"] for r in dv], st, label=f"{suite} greedy acc (correct img)")
    ax[4].axhline(0, color="k", lw=0.5)
    ax[4].set_title("image-swap penalty ΔNLL = NLL(partner) − NLL(correct)")
    ax[5].set_title("dev greedy accuracy")
    for a in ax:
        for e in ends[:-1]:
            a.axvline(e, color="grey", ls=":", lw=1)
        a.set_xlabel("update")
        a.legend(fontsize=7)
    fig.suptitle(f"{run}: learning curves (dotted lines = stage boundaries {', '.join(map(str, ends[:-1]))})")
    fig.tight_layout()
    fig.savefig(fig_dir / f"{run}_learning_curves.png", dpi=110)
    plt.close(fig)


def _n(x) -> str:
    return f"{x:,}" if isinstance(x, int) else "?"


def resources(R: Path) -> tuple[dict, str]:
    res: dict = {"feature_cache": [json.loads(p.read_text()) for p in sorted((R / "features").glob("cache_shard*.json"))]}
    for f in res["feature_cache"]:
        f.pop("preprocessing", None)
    md = ["| run | stage | trainable params | connector | LoRA | train time (h) | s/update | peak GPU mem (GiB) |",
          "|---|---|---|---|---|---|---|---|"]
    for run in RUNS:
        tr, ev = jl(R / "runs" / run / "train_log.jsonl"), jl(R / "runs" / run / "events.jsonl")
        if not tr:
            continue
        starts = {e["stage"]: e for e in ev if e.get("event") == "stage_start"}
        dv = jl(R / "runs" / run / "dev_metrics.jsonl")
        rr = {"stages": {}, "dev_eval_s": sum(r.get("seconds", 0) for r in dv)}
        for s in sorted({r["stage"] for r in tr}):
            rows = [r for r in tr if r["stage"] == s]
            secs = sum(r["seconds"] for r in rows)
            mem = max((r.get("peak_mem_gib", 0) for r in rows), default=0)
            st = starts.get(s, {})
            rr["stages"][s] = {"updates": len(rows), "train_s": secs, "peak_mem_gib": mem,
                               "trainable": st.get("trainable_params"), "skipped": sum(r["skipped"] for r in rows)}
            md.append(f"| {run} | {s} | {_n(st.get('trainable_params'))} | {_n(st.get('connector_params'))} | "
                      f"{_n(st.get('lora_params'))} | {secs / 3600:.2f} | {secs / max(1, len(rows)):.2f} | {mem:.2f} |")
        fe = R / "runs" / run / "final_eval"
        if (fe / "eval_resources.json").exists():
            rr["final_eval"] = json.loads((fe / "eval_resources.json").read_text())
            gt = [r["gen_time"] for f in fe.glob("*__*.jsonl") for r in jl(f)]
            rr["final_eval"]["gen_ms_per_answer_mean"] = 1000 * float(np.mean(gt)) if gt else None
        cal = R / "runs" / run / "calibration.json"
        if cal.exists():
            c = json.loads(cal.read_text())
            rr["calibration"] = {k: c[k] for k in ("scale", "text_median_norm", "visual_median_norm_before")}
        res[run] = rr
    fc = res["feature_cache"]
    if fc:
        md.append(f"\nFeature caching (separate from training): {sum(f['n_images'] for f in fc)} images, "
                  f"max shard wall time {max(f['seconds'] for f in fc) / 60:.1f} min on {len(fc)} GPU(s), "
                  f"{sum(f['bytes'] for f in fc) / 2 ** 30:.2f} GiB FP16, peak GPU mem "
                  f"{max(f['memory'].get('peak_alloc_gib', 0) for f in fc):.2f} GiB.")
    for run in RUNS:
        if run in res:
            md.append(f"\n{run}: dev evaluation total {res[run]['dev_eval_s'] / 60:.1f} min; "
                      f"calibration {res[run].get('calibration')}; final eval {res[run].get('final_eval', {}).get('gen_ms_per_answer_mean')} ms/answer")
    return res, "\n".join(md)


def comparison(R: Path) -> str:
    rows = ["| suite | metric | condition | vt196 | vt49 |", "|---|---|---|---|---|"]
    S = {r: json.loads((R / "runs" / r / "final_eval" / "summary.json").read_text())
         for r in RUNS if (R / "runs" / r / "final_eval" / "summary.json").exists()}
    if len(S) < 2:
        return ""
    for suite in ("grounding", "vqa"):
        for cond in ("correct", "swap", "mismatched", "none"):
            for m in ("acc", "pair_acc"):
                cells = []
                for r in RUNS:
                    v = S[r]["suites"][suite]["all"]["conditions"][cond][m]
                    cells.append(f"{100 * v[0]:.1f} [{100 * v[1]:.1f}, {100 * v[2]:.1f}]")
                rows.append(f"| {suite} | {m} | {cond} | " + " | ".join(cells) + " |")
    return "\n".join(rows)


def main() -> None:
    R = output_root()
    A = R / "report_assets"
    (A / "figures").mkdir(parents=True, exist_ok=True)
    (A / "tables").mkdir(parents=True, exist_ok=True)
    for run in RUNS:
        curves(run, R, A / "figures")
        sm = R / "runs" / run / "final_eval" / "summary.md"
        if sm.exists():
            (A / "tables" / f"{run}_final_results.md").write_text(f"## {run} final test results\n\n" + sm.read_text())
    res, md = resources(R)
    write_json(A / "tables" / "resources.json", res)
    (A / "tables" / "resources.md").write_text(md)
    cmp = comparison(R)
    if cmp:
        (A / "tables" / "vt196_vs_vt49.md").write_text("## 196 vs 49 visual tokens (accuracy %, 95% CI)\n\n" + cmp)
    fe = R / "runs" / "vt196" / "final_eval"
    if fe.exists():
        chosen = Q.select(Q.load_records(fe), 6304)
        write_json(A / "tables" / "qualitative_examples.json", {"rule": Q.__doc__, "examples": chosen})
        img_dir = Path(os.environ.get("PA3_SCRATCH", "/tmp/pa3_scratch")) / "images"
        for suite in ("grounding", "vqa"):
            Q.render([c for c in chosen if c["suite"] == suite], img_dir, A / "figures" / f"qualitative_{suite}.png")
    print(f"[report] assets in {A}", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/tests/conftest.py
"""Fixtures: a tiny random Qwen3 (28 layers, real module names, real vocab size) + the Qwen3 tokenizer, and a
synthetic COCO/VQAv2 universe for the data pipeline and an end-to-end CPU training run.

Tokenizer source: $PA3_TEST_TOKENIZER (dir with tokenizer files) or the hub id Qwen/Qwen3-0.6B (on Kaggle).
"""
from __future__ import annotations

import json
import os
import random
import shutil
import sys
from pathlib import Path

import pytest
import torch

ROOT = Path(__file__).resolve().parents[1]
COMMON = ROOT.parent / "common"
sys.path[:0] = [str(ROOT / "src"), str(COMMON)]

TINY = dict(vocab_size=151936, hidden_size=64, intermediate_size=128, num_hidden_layers=28,
            num_attention_heads=4, num_key_value_heads=2, head_dim=16, max_position_embeddings=4096,
            tie_word_embeddings=True, rms_norm_eps=1e-6, rope_theta=1_000_000.0)
# Stage budgets for the CPU run: 16 train images x 3 captions = 48 = 3 updates/pass -> stage 1 = 6 updates
SMALL_DATA = ["caption_train_images=16", "caption_dev_images=6",
              'vqa_train={"yes/no": 4, "number": 2, "other": 4}', 'vqa_dev={"yes/no": 2, "number": 1, "other": 2}',
              'vqa_test={"yes/no": 3, "number": 2, "other": 3}', "grounding_dev={presence: 2, counting: 2}",
              "grounding_test={presence: 3, counting: 3}", "pool_cap_per_category=20"]


def _tokenizer_src() -> str:
    return os.environ.get("PA3_TEST_TOKENIZER", "Qwen/Qwen3-0.6B")


@pytest.fixture(scope="session")
def tiny_dir(tmp_path_factory) -> Path:
    from transformers import AutoTokenizer, Qwen3Config, Qwen3ForCausalLM

    d = tmp_path_factory.mktemp("tiny_qwen3")
    torch.manual_seed(0)
    m = Qwen3ForCausalLM(Qwen3Config(**TINY))
    with torch.no_grad():
        for p in m.parameters():
            if p.ndim >= 2:
                p.normal_(0, 0.05)
    m.to(torch.bfloat16).save_pretrained(d)
    AutoTokenizer.from_pretrained(_tokenizer_src()).save_pretrained(d)
    return d


@pytest.fixture(scope="session")
def tok(tiny_dir):
    from pa3vlm.models.vlm import load_tokenizer

    return load_tokenizer(str(tiny_dir))


# ----------------------------------------------------------------------------- synthetic COCO / VQAv2
CATS = ["dog", "cat", "car", "chair", "cup", "bottle", "person", "bench"]


def _coco(split: str, ids: list[int], rnd: random.Random) -> tuple[dict, dict]:
    from pa3vlm.data.coco import CATEGORIES

    names = sorted(CATEGORIES)
    cid = {n: i + 1 for i, n in enumerate(names)}
    images = [{"id": i, "file_name": f"COCO_{split}_{i:012d}.jpg", "width": 640, "height": 480} for i in ids]
    anns, caps, aid = [], [], 1
    for i in ids:
        present = rnd.sample(CATS, rnd.randint(1, 3))
        words = []
        for c in present:
            n = rnd.randint(1, 4)
            for _ in range(n):
                area = rnd.choice([5000.0, 8000.0, 12000.0] + ([300.0] if rnd.random() < 0.1 else []))
                anns.append({"id": aid, "image_id": i, "category_id": cid[c], "area": area,
                             "bbox": [10, 10, 50, 50], "iscrowd": 1 if rnd.random() < 0.03 else 0})
                aid += 1
            words.append(f"{n} {c}")
        for k in range(5):
            caps.append({"id": i * 10 + k, "image_id": i, "caption": f" A photo  showing {' and '.join(words)} ({k})."})
    inst = {"images": images, "annotations": anns, "categories": [{"id": cid[n], "name": n} for n in names]}
    return inst, {"images": images, "annotations": caps}


def _vqa(split: str, ids: list[int], rnd: random.Random, n_pairs: int, qid0: int) -> tuple[dict, dict, list]:
    qs, anns, pairs = [], [], []
    templates = {"yes/no": ("Is the {} visible?", ["yes", "no"]), "number": ("How many {} are there?", ["2", "3", "four"]),
                 "other": ("What color is the {}?", ["red", "blue", "green"])}
    q = qid0
    for j in range(n_pairs):
        kind = ["yes/no", "number", "other"][j % 3]
        t, vals = templates[kind]
        text = t.format(rnd.choice(CATS))
        a, b = rnd.sample(ids, 2)
        va, vb = rnd.sample(vals, 2)
        bad = j % 7 == 6                      # low agreement -> must be filtered
        for img, v in ((a, va), (b, vb)):
            agree = 6 if bad else rnd.choice([8, 9, 10])
            raw = [v] * agree + ["maybe"] * (10 - agree)
            if v == "red":
                raw = ["red"] * (agree - 3) + ["Red"] * 3 + ["maybe"] * (10 - agree)
            qs.append({"question_id": q, "image_id": img, "question": text})
            anns.append({"question_id": q, "image_id": img, "answer_type": kind,
                         "answers": [{"answer": x, "answer_id": k + 1} for k, x in enumerate(raw)]})
            q += 1
        pairs.append([q - 2, q - 1])
    return {"questions": qs}, {"annotations": anns}, pairs


def make_raw(raw: Path, seed: int = 0) -> None:
    rnd = random.Random(seed)
    raw.mkdir(parents=True, exist_ok=True)
    tr, va = list(range(1, 81)), list(range(100001, 100241))
    for split, ids in (("train2014", tr), ("val2014", va)):
        inst, caps = _coco(split, ids, rnd)
        (raw / f"instances_{split}.json").write_text(json.dumps(inst))
        (raw / f"captions_{split}.json").write_text(json.dumps(caps))
        qj, aj, pr = _vqa(split, ids, rnd, 60, 1_000_000 if split == "train2014" else 5_000_000)
        (raw / f"v2_OpenEnded_mscoco_{split}_questions.json").write_text(json.dumps(qj))
        (raw / f"v2_mscoco_{split}_annotations.json").write_text(json.dumps(aj))
        (raw / f"v2_mscoco_{split}_complementary_pairs.json").write_text(json.dumps(pr))


@pytest.fixture(scope="session")
def out_root(tmp_path_factory) -> Path:
    """Synthetic raw annotations -> manifests under a fresh PA3_OUTPUT_ROOT."""
    from pa3vlm.config import load_data_config
    from pa3vlm.data.splits import select_all, write_manifests

    root = tmp_path_factory.mktemp("pa3_t2_out")
    scratch = tmp_path_factory.mktemp("pa3_t2_scratch")
    make_raw(scratch / "raw")
    os.environ["PA3_OUTPUT_ROOT"] = str(root)
    os.environ["PA3_SCRATCH"] = str(scratch)
    os.environ["PA3_INPUT_ROOTS"] = str(scratch / "nonexistent")
    cfg = load_data_config(None, SMALL_DATA)
    sets, rep = select_all(scratch / "raw", cfg)
    write_manifests(sets, rep, root / "data")
    return root


def run_cfg(root: Path, tiny_dir: Path, name: str, **kw) -> Path:
    import yaml

    from pa3vlm.config import load_config

    cfg = load_config("vt196")
    cfg.update({"run_name": name, "model_id": str(tiny_dir), "features": "random", "dtype": "float32",
                "dev_every": 5, "ckpt_every": 2, "calib_images": 8, "bootstrap": 200})
    cfg["stages"][0]["updates"], cfg["stages"][1]["updates"], cfg["stages"][2]["updates"] = 6, 4, 3
    cfg.update(kw)
    p = root / f"cfg_{name}.yaml"
    p.write_text(yaml.safe_dump(cfg))
    return p


def run_module(root: Path, module: str, *args: str, check: bool = True):
    import subprocess

    env = {**os.environ, "PA3_OUTPUT_ROOT": str(root), "PYTHONPATH": os.pathsep.join([str(ROOT / "src"), str(COMMON)]),
           "OMP_NUM_THREADS": "2", "CUDA_VISIBLE_DEVICES": ""}
    r = subprocess.run([sys.executable, "-m", module, *args], env=env, capture_output=True, text=True, timeout=3600)
    if check and r.returncode != 0:
        raise RuntimeError(r.stdout[-3000:] + "\n" + r.stderr[-6000:])
    return r


def pytest_sessionfinish(session, exitstatus):
    shutil.rmtree(ROOT / ".pytest_cache", ignore_errors=True)


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/tests/test_data.py
"""Data pipeline: normalisation/scoring, VQA filtering, seeded selection + isolation, sampler, schedule,
image-control conditions and the pair bootstrap (no model needed)."""
from __future__ import annotations

import json
from collections import Counter

import numpy as np
import pytest


def test_normalization_and_soft_score():
    from pa3vlm.data.normalize import normalize_answer as N, soft_score

    assert N("Two") == "2" and N("  The dog.  ") == "dog" and N("Yes!") == "yes" and N("dont") == "don't"
    assert N("10,000") == "10000" and N("3.5") == "3.5" and N(None) == ""
    refs = ["2"] * 3 + ["3"] * 7
    assert soft_score("2", refs) == pytest.approx(0.9)          # 3 matches: official leave-one-out average
    assert soft_score("2", ["2"] * 4 + ["x"] * 6) == pytest.approx(1.0)
    assert soft_score("2", ["2"] * 2 + ["x"] * 8) == pytest.approx(0.6)
    assert soft_score("2", ["2"] + ["x"] * 9) == pytest.approx(0.3)
    assert soft_score("", ["x"] * 10) == 0.0


def test_vqa_consensus_and_pairs():
    from pa3vlm.data.vqa import candidate_pairs, consensus

    assert consensus(["red"] * 5 + ["Red"] * 3 + ["blue"] * 2, 8) == ("red", "red")
    assert consensus(["Red"] * 5 + ["red"] * 3 + ["blue"] * 2, 8) == ("red", "Red")
    assert consensus(["two"] * 4 + ["2"] * 4 + ["3"] * 2, 8) == ("2", "2")   # tie on raw strings -> lexicographic
    assert consensus(["a"] * 7 + ["b"] * 3, 8) is None
    qs = {1: {"image_id": 10, "question": "Is it red?"}, 2: {"image_id": 11, "question": "Is it  red?"},
          3: {"image_id": 12, "question": "Is it red?"}, 4: {"image_id": 13, "question": "Is it blue?"}}
    an = {1: {"answers": ["yes"] * 10, "answer_type": "yes/no"}, 2: {"answers": ["no"] * 9 + ["yes"], "answer_type": "yes/no"},
          3: {"answers": ["yes"] * 10, "answer_type": "yes/no"}, 4: {"answers": ["no"] * 10, "answer_type": "yes/no"}}
    c = candidate_pairs([[1, 2], [1, 3], [1, 4]], qs, an, "val2014", 8)
    assert len(c) == 1 and [e["canonical"] for e in c[0]["examples"]] == ["yes", "no"]   # same answers / question differ
    assert len(c[0]["examples"][1]["answers"]) == 10


def test_selection_sizes_isolation_and_labels(out_root):
    from pa3vlm.data.coco import NO_COUNTING

    D = out_root / "data"
    rep = json.loads((D / "selection_report.json").read_text())
    assert rep["isolation"]["ok"]
    sz = rep["sizes"]
    assert sz["captions_train"] == 48 and sz["captions_dev"] == 6
    assert sz["vqa_train"] == 10 and sz["vqa_dev"] == 5 and sz["vqa_test"] == 8
    assert rep["kinds"]["grounding_test"] == {"presence": 3, "counting": 3}
    assert rep["kinds"]["vqa_test"] == {"yes/no": 3, "number": 2, "other": 3}
    test = json.loads((D / "grounding_test.json").read_text())
    pres = [p for p in test if p["kind"] == "presence"]
    assert {tuple(sorted(e["target"] for e in p["examples"])) for p in pres} == {("no", "yes")}
    pools = json.loads((D / "presence_pool.json").read_text()) + json.loads((D / "counting_pool.json").read_text())
    first = Counter(p["examples"][0]["target"] for p in pools if p["kind"] == "presence")
    assert first["yes"] > 0 and first["no"] > 0, "slot order must be randomised"
    for p in pools + test:
        a, b = p["examples"]
        assert a["image"] != b["image"] and a["target"] != b["target"]
        if p["kind"] == "counting":
            assert p["category"] not in NO_COUNTING and {a["target"], b["target"]} <= {"1", "2", "3", "4"}
            assert p["question"].startswith("How many ")
    vt = json.loads((D / "vqa_train.json").read_text())
    for p in vt:
        for e in p["examples"]:
            assert len(e["answers"]) == 10 and e["target"] in e["answers"]
    ids = json.loads((D / "selected_ids.json").read_text())
    assert len(ids["vqa_test"]["pair_ids"]) == 8 and ids["vqa_test"]["source_question_pairs"][0] is not None


def test_selection_is_deterministic(out_root, tmp_path):
    from pa3vlm.config import load_data_config
    from pa3vlm.data.splits import select_all
    from conftest import SMALL_DATA
    import os
    from pathlib import Path

    sets, _ = select_all(Path(os.environ["PA3_SCRATCH"]) / "raw", load_data_config(None, SMALL_DATA))
    for name in ("vqa_test", "grounding_test", "captions_train", "presence_pool"):
        assert sets[name] == json.loads((out_root / "data" / f"{name}.json").read_text()), name


def test_sampler_stage1_and_mixture():
    from pa3vlm.data.sampler import CurriculumSampler, StageSpec

    s = CurriculumSampler(6304, 48, {"presence": 30, "counting": 20, "vqa": 10}, 8, 2, passes=2)
    assert s.stage1_updates() == 6
    seen = Counter()
    for u in range(6):
        mbs, c = s.update(StageSpec(1, 6, None), u)
        assert [len(m) for m in mbs] == [8, 8] and c == {"caption": 16}
        seen.update(i for m in mbs for _, i in m)
    assert set(seen) == set(range(48)) and set(seen.values()) == {2}
    # each pass is a permutation; passes differ
    p0 = [i for u in range(3) for m in s.update(StageSpec(1, 6, None), u)[0] for _, i in m]
    p1 = [i for u in range(3, 6) for m in s.update(StageSpec(1, 6, None), u)[0] for _, i in m]
    assert sorted(p0) == sorted(p1) == list(range(48)) and p0 != p1
    mix = {"presence": 0.5, "counting": 0.25, "caption": 0.25}
    tot = Counter()
    for u in range(400):
        mbs, c = s.update(StageSpec(2, 400, mix), u)
        tot.update(c)
        for m in mbs:       # pairs complete and adjacent inside a microbatch
            for k in range(0, 8, 2):
                if m[k][0] != "caption":
                    assert m[k][:2] == m[k + 1][:2] and (m[k][2], m[k + 1][2]) == (0, 1)
        assert s.update(StageSpec(2, 400, mix), u) == (mbs, c)      # stateless
    n = sum(tot.values())
    assert n == 400 * 16
    for k, p in mix.items():
        assert abs(tot[k] / n - p) < 0.03, (k, tot[k] / n)


def test_lr_schedule():
    from pa3vlm.train.schedule import lr_at, warmup_updates

    assert [warmup_updates(T, 0.05) for T in (2250, 1500, 750)] == [113, 75, 38]
    assert lr_at(1, 2250, 1e-3) == pytest.approx(1e-3 / 113)
    assert lr_at(113, 2250, 1e-3) == pytest.approx(1e-3)
    assert lr_at(2250, 2250, 1e-3) == pytest.approx(1e-4)
    assert lr_at(750, 750, 1e-5) == pytest.approx(1e-6)
    v = [lr_at(t, 750, 1.0) for t in range(38, 751)]
    assert all(a >= b for a, b in zip(v, v[1:]))


def test_conditions_mapping():
    from pa3vlm.eval.conditions import images_for, mismatch_mapping

    pairs = [{"pair_id": f"p{i}", "examples": [{"ex_id": f"p{i}/0", "image": f"v:{2 * i}"},
                                               {"ex_id": f"p{i}/1", "image": f"v:{2 * i + 1}"}]} for i in range(9)]
    mm = mismatch_mapping(pairs, 6304, 1)
    assert mm == mismatch_mapping(pairs, 6304, 1)
    assert len(set(mm.values())) == 18
    for p in pairs:
        own = {e["image"] for e in p["examples"]}
        assert all(mm[e["ex_id"]] not in own for e in p["examples"])
    sw = images_for(pairs, "swap")
    assert sw[:2] == ["v:1", "v:0"] and images_for(pairs, "none") == [None] * 18
    assert images_for(pairs, "mismatched", mm) == [mm[e["ex_id"]] for p in pairs for e in p["examples"]]


def test_bootstrap_paired():
    from pa3vlm.eval.bootstrap import ci, diff_ci, resample_indices

    rng = np.random.default_rng(0)
    a = rng.random(50)
    b = a - 0.1
    idx = resample_indices(50, 4000, [6304, 1])
    assert (idx == resample_indices(50, 4000, [6304, 1])).all()
    p, lo, hi = diff_ci(a, b, idx)
    assert p == pytest.approx(0.1) and lo == pytest.approx(0.1) and hi == pytest.approx(0.1)  # paired -> no spread
    p, lo, hi = ci(a, idx)
    assert lo < p < hi


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/tests/test_features.py
"""SigLIP feature cache on CPU with a 1-layer random SigLIP vision tower (real 224/16 geometry, width 768):
final LayerNorm applied exactly once, FP16 memmap layout, deterministic preprocessing, 2x2 pooling."""
from __future__ import annotations

import json

import numpy as np
import torch

from conftest import run_module


def test_feature_cache(tmp_path, monkeypatch):
    from PIL import Image
    from transformers import SiglipConfig, SiglipModel, SiglipVisionModel

    from pa3vlm.features.siglip_cache import load_preproc, preprocess
    from pa3vlm.features.store import FeatureStore, pool2x2

    vdir = tmp_path / "siglip_tiny"
    torch.manual_seed(0)
    # full SigLIP checkpoint (vision_model.* + text_model.* keys), like the hub one
    full = SiglipModel(SiglipConfig(
        vision_config=dict(image_size=224, patch_size=16, hidden_size=768, intermediate_size=64, num_hidden_layers=1,
                           num_attention_heads=2),
        text_config=dict(hidden_size=32, intermediate_size=37, num_hidden_layers=1, num_attention_heads=2,
                         vocab_size=99, bos_token_id=1, eos_token_id=2, pad_token_id=1)))
    with torch.no_grad():   # non-trivial final LayerNorm so applying it twice would be detected
        full.vision_model.post_layernorm.weight.uniform_(0.5, 2.0)
        full.vision_model.post_layernorm.bias.uniform_(-1, 1)
    full.save_pretrained(vdir)
    m = SiglipVisionModel.from_pretrained(vdir).eval()
    ln = getattr(m, "vision_model", m).post_layernorm
    assert torch.equal(ln.weight, full.vision_model.post_layernorm.weight)
    (vdir / "preprocessor_config.json").write_text(json.dumps({
        "do_resize": True, "size": {"height": 224, "width": 224}, "resample": 3, "do_rescale": True,
        "rescale_factor": 1 / 255, "do_normalize": True, "image_mean": [0.5] * 3, "image_std": [0.5] * 3}))
    root, scratch = tmp_path / "out", tmp_path / "scratch"
    keys = ["train2014:1", "train2014:2", "val2014:3"]
    rng = np.random.default_rng(0)
    for k in keys:
        split, i = k.split(":")
        p = scratch / "images" / split / f"COCO_{split}_{int(i):012d}.jpg"
        p.parent.mkdir(parents=True, exist_ok=True)
        mode = "L" if i == "3" else "RGB"       # grayscale COCO images exist: must be converted to RGB
        arr = rng.integers(0, 255, (300, 410, 3) if mode == "RGB" else (300, 410), dtype=np.uint8)
        Image.fromarray(arr, mode).save(p)
    (root / "data").mkdir(parents=True)
    (root / "data" / "images.json").write_text(json.dumps({"train": keys[:2], "val": keys[2:]}))
    monkeypatch.setenv("PA3_SCRATCH", str(scratch))
    for s in (0, 1):
        run_module(root, "pa3vlm.features.siglip_cache", "--shard", str(s), "--nshards", "2", "--vision-id",
                   str(vdir), "--workers", "0")
    st = FeatureStore(scratch / "features", 1)
    f = st.get(keys)
    assert f.shape == (3, 196, 768)
    pp = load_preproc(str(vdir))
    x = torch.from_numpy(np.stack([preprocess(scratch / "images" / "train2014" / f"COCO_train2014_{1:012d}.jpg", pp)]))
    assert x.shape == (1, 3, 224, 224)
    with torch.no_grad():
        out = m(pixel_values=x, output_hidden_states=True)
        once = ln(out.hidden_states[-1])
    assert torch.allclose(once, out.last_hidden_state, atol=1e-5)
    assert torch.allclose(f[0], once[0].half().float(), atol=1e-6)
    twice = ln(once)
    assert not torch.allclose(f[0], twice[0], atol=1e-2)
    meta = json.loads((root / "features" / "cache_shard0.json").read_text())
    assert meta["nonfinite_fp16"] == 0 and meta["ln_check_max_diff"] < 1e-4
    assert torch.allclose(FeatureStore(scratch / "features", 2).get(keys[:1]), pool2x2(f[:1]))


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/tests/test_interface.py
"""Visual interface: chat-template split, collate (masks/positions/labels), embedding splice, loss masking and
normalisation against a full-logits reference, padding invariance, no-image position rebuild, gradient flow,
LoRA boundary, parameter counts, and the KV-cache greedy decoder against HF generate."""
from __future__ import annotations

import pytest
import torch
import torch.nn.functional as F

from pa3vlm.data.collate import collate
from pa3vlm.data.prompts import ANSWER_INSTRUCTION, CAPTION_PROMPT, ChatFormat
from pa3vlm.features.store import RandomFeatures, pool2x2

CAPS = {"question": 80, "caption": 64, "answer": 16}


@pytest.fixture(scope="module")
def fmt(tok):
    return ChatFormat(tok, CAPS)


@pytest.fixture(scope="module")
def vlm(tiny_dir):
    from pa3vlm.models.connector import make_connector
    from pa3vlm.models.vlm import VLM, load_lm

    lm = load_lm(str(tiny_dir), torch.float32, "sdpa", grad_ckpt=True)
    v = VLM(lm, make_connector(6304, d_out=lm.config.hidden_size))
    v.connector.scale.fill_(0.37)
    return v.eval()


def _encs(fmt):
    return [fmt.encode("a", "train2014:1", "How many dogs can you see?", "2"),
            fmt.encode("b", "train2014:2", None, "A brown dog runs across a very green park near the river."),
            fmt.encode("c", "train2014:3", "Does this image contain a cat?", "no"),
            fmt.encode("d", "train2014:4", "What color is the bus?", "red")]


def test_chat_format(fmt, tok):
    e = fmt.encode("x", "k", "How many dogs can you see?", "two")
    assert tok.decode(e.prefix) == "<|im_start|>user\n"
    assert tok.decode(e.suffix) == ("\nHow many dogs can you see?\n" + ANSWER_INSTRUCTION +
                                    "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n")
    assert e.target[-1] == fmt.im_end and tok.decode(e.target[:-1]) == "two"
    ref = tok.apply_chat_template([{"role": "user", "content": "\nHow many dogs can you see?\n" + ANSWER_INSTRUCTION}],
                                  tokenize=False, add_generation_prompt=True, enable_thinking=False)
    assert tok.decode(e.prefix + e.suffix) == ref           # text-only rendering identical, image slot between
    c = fmt.encode("y", "k", None, "word " * 100)
    assert len(c.target) == 64 + 1 and fmt.stats["target_truncated"] >= 1
    assert CAPTION_PROMPT in tok.decode(c.suffix)
    q = fmt.encode("z", "k", "why " * 200, "yes")
    assert len(tok(tok.decode(q.suffix)).input_ids) < 80 + 40


@pytest.mark.parametrize("side", ["right", "left"])
def test_collate_masks_positions_labels(fmt, side):
    encs = _encs(fmt)
    st = RandomFeatures(1)
    imgs = [e.image for e in encs[:3]] + [None]
    b = collate(encs, imgs, st.get, 196, fmt.pad_id, side, True)
    for r, e in enumerate(encs):
        n_real = len(e.prefix) + (196 if imgs[r] else 0) + len(e.suffix) + len(e.target)
        assert int(b.attn[r].sum()) == n_real and int(b.vis[r].sum()) == (196 if imgs[r] else 0)
        real = b.attn[r].bool()
        assert b.pos[r][real].tolist() == list(range(n_real))
        lab = b.labels[r][b.labels[r] != -100].tolist()
        assert lab == e.target and int(b.term[r].sum()) == 1 and b.labels[r][b.term[r]].item() == fmt.im_end
        vis_idx = b.vis[r].nonzero().flatten().tolist()
        if vis_idx:
            off = real.nonzero().flatten()[0].item()
            assert vis_idx == list(range(off + len(e.prefix), off + len(e.prefix) + 196))
        assert not bool((b.vis[r] & (b.labels[r] != -100)).any())
    assert b.feats.shape == (3, 196, 768)
    # no-image row: identical text tokens, visual slots removed
    b2 = collate(encs[:1], [None], st.get, 196, fmt.pad_id, side, True)
    full = b.ids[0][b.attn[0].bool() & ~b.vis[0]].tolist()
    assert b2.ids[0][b2.attn[0].bool()].tolist() == full


def test_pool2x2():
    x = torch.arange(196, dtype=torch.float32).view(1, 196, 1).repeat(1, 1, 3)
    y = pool2x2(x)
    assert y.shape == (1, 49, 3)
    assert y[0, 0, 0].item() == pytest.approx((0 + 1 + 14 + 15) / 4)
    assert y[0, 8, 0].item() == pytest.approx((30 + 31 + 44 + 45) / 4)   # block (1,1)


def test_splice(fmt, vlm):
    encs = _encs(fmt)[:2]
    st = RandomFeatures(1)
    b = collate(encs, [e.image for e in encs], st.get, 196, fmt.pad_id, "right", True)
    with torch.no_grad():
        emb = vlm.embed(b)
        E = vlm.lm.get_input_embeddings()(b.ids)
        V = vlm.connector(b.feats)
    assert torch.equal(emb[b.vis], V.reshape(-1, V.shape[-1]))
    assert torch.equal(emb[~b.vis], E[~b.vis])
    assert torch.allclose(V, 0.37 * vlm.connector.raw(b.feats))


def _reference_loss(vlm, b):
    """Full logits from the HF causal LM, standard shift, per-example token mean, then batch mean."""
    with torch.no_grad():
        logits = vlm.lm(inputs_embeds=vlm.embed(b), attention_mask=b.attn, position_ids=b.pos).logits.float()
    per = []
    for r in range(b.ids.shape[0]):
        lab = b.labels[r, 1:]
        m = lab != -100
        per.append(F.cross_entropy(logits[r, :-1][m], lab[m]))
    return torch.stack(per)


def test_loss_matches_reference_and_padding_invariance(fmt, vlm):
    from pa3vlm.models.vlm import per_example_mean

    encs = _encs(fmt)
    st = RandomFeatures(1)
    imgs = [e.image for e in encs[:3]] + [None]
    b = collate(encs, imgs, st.get, 196, fmt.pad_id, "right", True)
    with torch.no_grad():
        nll, rows, term = vlm.supervised_nll(b)
    per = per_example_mean(nll, rows, 4)
    assert torch.allclose(per, _reference_loss(vlm, b), atol=1e-5)
    # example weighting: each example counts equally regardless of its number of target tokens
    assert torch.allclose(per.mean(), _reference_loss(vlm, b).mean(), atol=1e-6)
    ans = per_example_mean(nll, rows, 4, ~term)
    for r, e in enumerate(encs):          # alone (no padding) == inside a padded batch
        b1 = collate([e], [imgs[r]], st.get, 196, fmt.pad_id, "right", True)
        with torch.no_grad():
            n1, r1, t1 = vlm.supervised_nll(b1)
        assert torch.allclose(per_example_mean(n1, r1, 1), per[r:r + 1], atol=1e-5)
        assert torch.allclose(per_example_mean(n1, r1, 1, ~t1), ans[r:r + 1], atol=1e-5)


def test_no_image_positions_rebuilt(fmt, vlm):
    e = _encs(fmt)[0]
    b = collate([e], [None], RandomFeatures(1).get, 196, fmt.pad_id, "right", True)
    ids = torch.tensor([e.prefix + e.suffix + e.target])
    with torch.no_grad():
        ref = vlm.lm(input_ids=ids).logits            # HF default positions 0..n-1
        got = vlm.lm(inputs_embeds=vlm.embed(b), attention_mask=b.attn, position_ids=b.pos).logits
    assert torch.allclose(ref, got, atol=1e-5)


def test_gradients_reach_connector_only(fmt, vlm):
    encs = _encs(fmt)[:2]
    b = collate(encs, [e.image for e in encs], RandomFeatures(1).get, 196, fmt.pad_id, "right", True)
    vlm.train()
    try:
        assert vlm.lm.model.gradient_checkpointing
        nll, rows, _ = vlm.supervised_nll(b)
        nll.mean().backward()
        assert all(p.grad is not None and p.grad.abs().sum() > 0 for p in vlm.connector.parameters())
        assert all(p.grad is None and not p.requires_grad for p in vlm.lm.parameters())
    finally:
        vlm.zero_grad(set_to_none=True)
        vlm.eval()


def test_lora_boundary_and_param_counts(fmt, tiny_dir):
    from transformers import Qwen3Config, Qwen3ForCausalLM

    from pa3common.lora import inject_lora, lora_modules
    from pa3common.utils import count_params
    from pa3vlm.models.connector import Connector, make_connector
    from pa3vlm.models.vlm import VLM, load_lm

    lm = load_lm(str(tiny_dir), torch.float32, "sdpa", grad_ckpt=False)
    v = VLM(lm, make_connector(6304, d_out=64)).eval()
    encs = _encs(fmt)
    b = collate(encs, [e.image for e in encs], RandomFeatures(1).get, 196, fmt.pad_id, "right", True)
    with torch.no_grad():
        before = v.lm(inputs_embeds=v.embed(b), attention_mask=b.attn, position_ids=b.pos).logits
    inject_lora(v.lm, 16, 32, 0.05, 6304, ("q_proj", "k_proj", "v_proj", "o_proj"))
    assert len(lora_modules(v.lm)) == 4 * 28
    with torch.no_grad():
        after = v.lm(inputs_embeds=v.embed(b), attention_mask=b.attn, position_ids=b.pos).logits
    assert torch.equal(before, after), "B = 0 at injection: function unchanged at the stage boundary"
    names = {n for n, p in v.named_parameters() if p.requires_grad}
    assert all(n.startswith("connector.") or n.endswith(("lora_A", "lora_B")) for n in names)
    # real Qwen3-0.6B dimensions (meta device, no download)
    assert count_params(Connector(), True) == 1_837_056
    cfg = Qwen3Config(vocab_size=151936, hidden_size=1024, intermediate_size=3072, num_hidden_layers=28,
                      num_attention_heads=16, num_key_value_heads=8, head_dim=128, tie_word_embeddings=True)
    with torch.device("meta"):
        big = Qwen3ForCausalLM(cfg)
    for p in big.parameters():
        p.requires_grad_(False)
    inject_lora(big, 16, 32, 0.05, 6304, ("q_proj", "k_proj", "v_proj", "o_proj"))
    assert count_params(big, True) == 4_587_520


def test_greedy_matches_hf_generate(fmt, vlm):
    from pa3vlm.eval.generate import greedy

    encs = _encs(fmt)
    st = RandomFeatures(1)
    imgs = [e.image for e in encs[:3]] + [None]
    b = collate(encs, imgs, st.get, 196, fmt.pad_id, "left", False)
    seqs, stopped = greedy(vlm, b, 16, fmt.eos_ids, fmt.pad_id)
    for r, e in enumerate(encs):
        b1 = collate([e], [imgs[r]], st.get, 196, fmt.pad_id, "left", False)
        with torch.no_grad():
            emb = vlm.embed(b1)
            out = vlm.lm.generate(inputs_embeds=emb, attention_mask=b1.attn, max_new_tokens=16, do_sample=False,
                                  eos_token_id=fmt.eos_ids, pad_token_id=fmt.pad_id, use_cache=True)
        ref = [t for t in out[0].tolist()]
        cut = next((i for i, t in enumerate(ref) if t in fmt.eos_ids), len(ref))
        assert seqs[r] == ref[:cut], (r, seqs[r], ref)
        assert stopped[r] == (cut < len(ref))
        s1, _ = greedy(vlm, b1, 16, fmt.eos_ids, fmt.pad_id)
        assert s1[0] == seqs[r]


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/tests/test_training.py
"""End-to-end CPU run of the 3-stage curriculum (tiny Qwen3, random features), kill-and-resume equality,
final evaluation under the four image conditions, and report assets."""
from __future__ import annotations

import json
import os
import subprocess
import sys

import pytest
import torch

from conftest import ROOT, run_cfg, run_module


def _jl(p):
    return [json.loads(l) for l in p.read_text().splitlines() if l.strip()]


@pytest.fixture(scope="module")
def full_run(out_root, tiny_dir):
    cfg = run_cfg(out_root, tiny_dir, "full")
    run_module(out_root, "pa3vlm.train.loop", "--config", str(cfg))
    return out_root / "runs" / "full", cfg


def test_full_run_logs(full_run):
    run, _ = full_run
    tr = _jl(run / "train_log.jsonl")
    assert [r["update"] for r in tr] == list(range(1, 14))
    assert [r["stage"] for r in tr] == [1] * 6 + [2] * 4 + [3] * 3
    assert all(set(r["lr"]) == {"connector_w", "connector_b"} for r in tr[:6])
    assert all(set(r["lr"]) == {"connector_w", "connector_b", "lora"} for r in tr[6:])
    # per-stage warmup restarts: first update of each stage = peak / ceil(5% T) = peak (T small -> w = 1)
    assert tr[0]["lr"]["connector_w"] == pytest.approx(1e-3) and tr[6]["lr"]["lora"] == pytest.approx(2e-5)
    assert tr[5]["lr"]["connector_w"] == pytest.approx(1e-4) and tr[12]["lr"]["lora"] == pytest.approx(1e-6)
    assert all(sum(r["mix"].values()) == 16 for r in tr)
    assert all(set(r["mix"]) <= {"presence", "counting", "caption"} for r in tr[6:10])
    assert all(torch.isfinite(torch.tensor(r["loss"])) for r in tr)
    dv = _jl(run / "dev_metrics.jsonl")
    assert [r["update"] for r in dv] == [0, 5, 6, 10, 13]
    for k in ("caption_loss", "caption_loss_noimage", "grounding/all/delta_nll_swap", "vqa/all/nll_none",
              "grounding/presence/acc_greedy", "vqa/yes/no/nll_correct", "connector_norm_median"):
        assert k in dv[0], k
    starts = [e for e in _jl(run / "events.jsonl") if e["event"] == "stage_start"]
    assert starts[0]["lora_params"] == 0 and starts[0]["connector_params"] == 768 * 1024 + 1024 + 1024 * 64 + 64
    assert starts[1]["lora_params"] == 28 * 16 * ((64 + 64) + (64 + 32) + (64 + 32) + (64 + 64))
    for s in (1, 2, 3):
        assert (run / "weights" / f"stage{s}.pt").exists()
    cal = json.loads((run / "calibration.json").read_text())
    assert cal["scale"] > 0 and all(k.startswith("train2014:") for k in cal["image_keys"])
    assert len(list((run / "checkpoints").glob("resume_*.pt"))) == 2


def test_stage1_trains_connector_only(full_run):
    run, _ = full_run
    w1 = torch.load(run / "weights" / "stage1.pt", weights_only=False)
    w3 = torch.load(run / "weights" / "stage3.pt", weights_only=False)
    assert w1["lora"] == {} and len(w3["lora"]) == 2 * 4 * 28
    assert any(v.abs().sum() > 0 for k, v in w3["lora"].items() if k.endswith("lora_B"))


def test_resume_is_exact(full_run, out_root, tiny_dir):
    run, _ = full_run
    cfg = run_cfg(out_root, tiny_dir, "resumed", stop_after_update=7)
    r = run_module(out_root, "pa3vlm.train.loop", "--config", str(cfg), check=False)
    assert r.returncode == 3, r.stderr[-2000:]
    cfg = run_cfg(out_root, tiny_dir, "resumed", stop_after_update=None)
    run_module(out_root, "pa3vlm.train.loop", "--config", str(cfg))
    res = out_root / "runs" / "resumed"
    a, b = _jl(run / "train_log.jsonl"), _jl(res / "train_log.jsonl")
    assert [x["loss"] for x in a] == [x["loss"] for x in b]
    wa = torch.load(run / "weights" / "final.pt", weights_only=False)
    wb = torch.load(res / "weights" / "final.pt", weights_only=False)
    for part in ("connector", "lora"):
        for k in wa[part]:
            assert torch.equal(wa[part][k], wb[part][k]), (part, k)
    assert [r["update"] for r in _jl(res / "dev_metrics.jsonl")] == [0, 5, 6, 10, 13]


def test_final_eval_and_report(full_run, out_root):
    run, cfg = full_run
    run_module(out_root, "pa3vlm.eval.final", "--config", str(cfg))
    fe = run / "final_eval"
    for suite, n in (("grounding_test", 12), ("vqa_test", 16)):
        for cond in ("correct", "swap", "mismatched", "none"):
            recs = _jl(fe / f"{suite}__{cond}.jsonl")
            assert len(recs) == n
            if cond == "correct":
                assert all(r["image_used"] == r["own_image"] for r in recs)
            if cond == "swap":
                assert all(r["image_used"] == r["partner_image"] for r in recs)
            if cond == "none":
                assert all(r["image_used"] is None for r in recs)
            if cond == "mismatched":
                assert all(r["image_used"] not in (r["own_image"], r["partner_image"]) for r in recs)
    S = json.loads((fe / "summary.json").read_text())
    g = S["suites"]["grounding"]
    assert set(g) == {"all", "counting", "presence"} and g["all"]["n_pairs"] == 6
    lo, p, hi = g["all"]["conditions"]["correct"]["acc"][1], g["all"]["conditions"]["correct"]["acc"][0], \
        g["all"]["conditions"]["correct"]["acc"][2]
    assert lo <= p <= hi
    assert set(S["suites"]["vqa"]) == {"all", "yes/no", "number", "other"}
    assert "swap" in S["suites"]["vqa"]["all"]["advantages"]
    env = {**os.environ, "PA3_OUTPUT_ROOT": str(out_root)}
    os.makedirs(out_root / "runs" / "vt196", exist_ok=True)
    # report script expects run names vt196/vt49: link the test run as vt196
    import shutil
    shutil.copytree(run, out_root / "runs" / "vt196", dirs_exist_ok=True)
    r = subprocess.run([sys.executable, str(ROOT / "scripts" / "make_report_assets.py")], env=env,
                       capture_output=True, text=True, timeout=600)
    assert r.returncode == 0, r.stderr[-3000:]
    A = out_root / "report_assets"
    assert (A / "figures" / "vt196_learning_curves.png").exists()
    assert (A / "tables" / "vt196_final_results.md").exists() and (A / "tables" / "resources.md").exists()
    q = json.loads((A / "tables" / "qualitative_examples.json").read_text())
    assert "rule" in q


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_2/README.md
# PA3 Task 2 — Vision-language model and visual grounding

SigLIP (`google/siglip-base-patch16-224`, frozen, 196 cached patch features) → 2-layer MLP connector
(768→1024→1024, trainable, fixed calibration scale *s*) → `Qwen/Qwen3-0.6B` (frozen FP16; LoRA r16 on
q/k/v/o in Stages 2–3). Three-stage curriculum (2,250 + 1,500 + 750 updates, effective batch 16) and a
grounding evaluation under four image conditions. Approved design: project doc `claude/task2_design_decisions.md`.

## Run on Kaggle
Open `notebooks/pa3_task2_kaggle.ipynb` (self-contained: it writes every file of this folder and `common/`
with `%%writefile` cells). Settings: **GPU T4 ×2**, **Internet on**, then **Save & Run All (Commit)**.
GPU0 trains the 196-token model and GPU1 the optional 49-token model at the same time.
If `status.json` says `paused`, make the output a dataset, attach it and run again (work resumes exactly).
If a download host is unreachable, attach a dataset with the COCO 2014 / VQAv2 JSONs (and optionally the
`train2014`/`val2014` image folders); they are found automatically under `/kaggle/input`.

Regenerate the notebook after code changes: `python task_2/scripts/build_notebook.py`.

## Layout
```
configs/        base.yaml (shared recipe), vt196.yaml, vt49.yaml, data.yaml (selection sizes)
src/pa3vlm/
  data/         download (attached-or-download), coco (category wording, label filters), vqa (pair filter),
                grounding (presence/counting pairs), splits (seeded selection + isolation checks),
                normalize (VQA normalisation + soft score), prompts (chat template with image slot),
                collate (masks/positions/labels), sampler (stateless curriculum), bundle (lazy tokenisation)
  features/     siglip_cache (FP16 memmap, LN applied once), store (+2x2 pooling)
  models/       connector, calibrate, vlm (embedding splice, supervised-only LM head), build
  train/        loop (3 stages, AMP, checkpoints, dev eval, pause/resume), schedule, checkpoint
  eval/         dev, generate (KV-cache greedy), conditions, metrics, bootstrap, final, qualitative
  orchestrate.py  Kaggle pipeline (prefetch → data → audit → features → preflight → train → eval → report)
scripts/        build_notebook.py, label_audit.py, make_report_assets.py
tests/          CPU tests (tiny 28-layer Qwen3, synthetic COCO/VQA): 23 tests
```

## Outputs (`/kaggle/working/pa3_task2_outputs`)
`data/` manifests, selected IDs, selection report, label audit · `features/` caching cost ·
`runs/<vt196|vt49>/` config, calibration, train_log, dev_metrics, events, checkpoints log, stage weights,
`final_eval/` (per-condition generations, summary.json/.md) · `report_assets/` figures and tables ·
`results_task2.zip` (everything except resumable checkpoints).

## CPU tests
`PA3_TEST_TOKENIZER=<dir with the Qwen3 tokenizer> python -m pytest task_2/tests -q` (defaults to the hub id).


In [ ]:
# ============================ INTEGRITY CHECK ============================
import hashlib, json
EXPECTED = json.loads("{\"common/pa3common/__init__.py\": \"e66d72d492a8da8c2fed466f233fc02a88fdc69a922463a0cef6fc5a65403a90\", \"common/pa3common/utils.py\": \"3560127c545ba862ac5247f4287fc8d97f2c261f8ade20a8c297e9a61787a8c5\", \"common/pa3common/kaggle.py\": \"dafc543c9a557ccebab701927c45a75a53992d49d08966145f1a9249243db520\", \"common/pa3common/lora.py\": \"1856575a963e846dc1ea2210f17b9db07849be1750a38e1c42f37a4d535a0af1\", \"task_2/configs/base.yaml\": \"41315685c297c1232ad4d803fc11f485078eabd1270423fa8d6f0250b3213d88\", \"task_2/configs/data.yaml\": \"1e1d1fc4207dbb986dcdcd4675711525dc48d0b6874d7a47f978f24d1b0d7be2\", \"task_2/configs/vt196.yaml\": \"0f2a7d427807626ba871fb0a24b1924cdf079f3d123a7267c46193762b548870\", \"task_2/configs/vt49.yaml\": \"3378e9a61fcf15876d406c30dba6ab2a3916dbae5733ea8b71595170cea1a807\", \"task_2/src/pa3vlm/__init__.py\": \"eb44148a51faeb1e19a5025dda4ca06c416c4529b63962084b2cf915336a9b31\", \"task_2/src/pa3vlm/config.py\": \"17013ebbee7815f3269b37f8f54a53c99aaaa741be690c0e785a8360860b8bc3\", \"task_2/src/pa3vlm/data/__init__.py\": \"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\", \"task_2/src/pa3vlm/data/bundle.py\": \"9a189294547886eb8308a813cbb97260049578cb4273c7a23a0ad9e4b3209b89\", \"task_2/src/pa3vlm/data/coco.py\": \"84fdb461a0a8c7c733b427fa93ae12ed1e4ba74db8caa6eb01bac18c823b65a1\", \"task_2/src/pa3vlm/data/collate.py\": \"a1282ae7c014416a921a0227aab8fbe2c82bfe7cf9c2d9b0dcbcdd35f64fd95e\", \"task_2/src/pa3vlm/data/download.py\": \"a3f89f1fe774772377226b26d8e9dd67fd0b9856635fc1c02e461d0c2f9450a1\", \"task_2/src/pa3vlm/data/grounding.py\": \"7adb3d908ed9002890f05b6b9194988b432d09d72dcb27a876d39925d27ef6cd\", \"task_2/src/pa3vlm/data/normalize.py\": \"288cf22b90aa008a105055e75742eaa05834f679e467d6789b677e70568be651\", \"task_2/src/pa3vlm/data/prompts.py\": \"2444b033e3c3836760205df10f75400065faf9e51967c0b6aa1c88a80fb59208\", \"task_2/src/pa3vlm/data/sampler.py\": \"1dfa6cb1730d4cb5c5d0f321d33d5373cf532d0750f5960f694489b406937a8a\", \"task_2/src/pa3vlm/data/splits.py\": \"6d161f7e8af2c25d8886c338b4031e5c5ce006d8060c4bb05b3539cc3342585a\", \"task_2/src/pa3vlm/data/vqa.py\": \"6b7e3370590df93cbd80d12de115416bec1e189c4896c6c823a2e4c3d7b33569\", \"task_2/src/pa3vlm/eval/__init__.py\": \"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\", \"task_2/src/pa3vlm/eval/bootstrap.py\": \"8988ee6eba608cfe9a06ecb195f8626eea71ec144ba62976d5978d8337b67040\", \"task_2/src/pa3vlm/eval/conditions.py\": \"4ed44a8cd9f4675fe5ab1a392d7b2386f054f2006b2bd0fa69efd94f26dba79d\", \"task_2/src/pa3vlm/eval/dev.py\": \"f835490b8e47ebc86199b985aaa5b661ed79fc476ded9c191ea2ecca6df3edb4\", \"task_2/src/pa3vlm/eval/final.py\": \"2b0e8331b4fd10cae92ba3bf3ce4808e1c4923406f32780cc5d51b6e0752175e\", \"task_2/src/pa3vlm/eval/generate.py\": \"4bf55e9ed30b89110a64fc83399e06f69e4ddf1a30504ccda74094487fc4d06f\", \"task_2/src/pa3vlm/eval/metrics.py\": \"dde32f689002bbf062bf0e8c011a2abdf7f794caf6fbcc4f8a3d94ca147e21cf\", \"task_2/src/pa3vlm/eval/qualitative.py\": \"05414b9d2aea890ba17c9ddbe1ed6bb560e4c9999ebdf83492a64c9b2c402de7\", \"task_2/src/pa3vlm/features/__init__.py\": \"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\", \"task_2/src/pa3vlm/features/siglip_cache.py\": \"d3b30c9625c669cc9e51ac65a433b2caa7ef989db5c44b793b796f67d82beb3b\", \"task_2/src/pa3vlm/features/store.py\": \"7d59e7e5ea65ea5262a59959076d2128d93150986ee9cd620766172f6dac8aef\", \"task_2/src/pa3vlm/models/__init__.py\": \"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\", \"task_2/src/pa3vlm/models/build.py\": \"f7e3f79b01f2e20f71b094e6826b446d68f199925abaf4584c890a5b31da7542\", \"task_2/src/pa3vlm/models/calibrate.py\": \"9b1c48198cbfa850acdd3dd4dd4893b0acaa4e6e9eb965070de89b42f8a56070\", \"task_2/src/pa3vlm/models/connector.py\": \"f22a6bc6f78cb013ae0b01c3f98104716be145bb76fdf797262739fad75367e7\", \"task_2/src/pa3vlm/models/vlm.py\": \"95799beda1abd0c42d04d405996f7d44d61d8171f4f40ca580dca75003591048\", \"task_2/src/pa3vlm/orchestrate.py\": \"ebe3b54603afa0c0a419e2fff770288d5527df922f9ca6a636970e7a1b1912a8\", \"task_2/src/pa3vlm/prefetch.py\": \"0052e1e7e8d690ad9e56bc427463396f455b3d0ace16ad4e03ef90e76e492011\", \"task_2/src/pa3vlm/train/__init__.py\": \"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\", \"task_2/src/pa3vlm/train/checkpoint.py\": \"882c74d75b434be1a255241c66577e37c105f4b50c614c851f29fe08c4f52eec\", \"task_2/src/pa3vlm/train/loop.py\": \"8611a6c00aea357c60c263712b9926f5ac517c37a4d844dc7a60cb9a2eab3aaa\", \"task_2/src/pa3vlm/train/schedule.py\": \"b07a89bd7e1e252e0b5f236a73d109c49de508d01bfe8f8a60de60d107182269\", \"task_2/scripts/build_notebook.py\": \"b6db50f77803c1471a32705eb41e2d285e23f02fd61cd237cb557d6b4a95b406\", \"task_2/scripts/label_audit.py\": \"705b23dd2454e4d9c6311e0b90a2095e2b03affdbc770c0360f2485e4e747e37\", \"task_2/scripts/make_report_assets.py\": \"39cbc0a93d0d1a4c0a1c878c85fcc7bc7d5d97cbe41ba0aaf12ec212e976ab19\", \"task_2/tests/conftest.py\": \"d8ce3696fa12fda1321db08e46a7cf64690ef45b1581df6bdc89df94503c0c48\", \"task_2/tests/test_data.py\": \"09ee55ba03a8281625f50a6be0010f1d431ef7942473be652c83ed21400a817c\", \"task_2/tests/test_features.py\": \"7e327d5098ecbe4042937e3f8872af1618e5367aaf0e17927d8b4a6522b4eae9\", \"task_2/tests/test_interface.py\": \"934e7836ff5172d5e90add39c3c7fb11f63c6e8d87569adf26aad5d3870893dc\", \"task_2/tests/test_training.py\": \"78657dd25baf594ff689f85e58ba99e17371b3f59eaec16176179c8e0987917e\", \"task_2/README.md\": \"fb59843827c4a3094a02475a3ff697b7ed27823b6fd9dcdfee53ef3b434b2ccb\"}")
bad = [r for r, h in EXPECTED.items()
       if hashlib.sha256(open('/kaggle/working/ATML-PA3/' + r, 'rb').read().rstrip(b'\n')).hexdigest() != h]
assert not bad, f'files differ from the embedded sources: {bad}'
print(f'{len(EXPECTED)} files written and verified')

In [ ]:
# ============================ DEPENDENCIES ============================
import sys, subprocess
sys.path[:0] = [f"{CODE_DIR}/common", f"{CODE_DIR}/task_2/src"]
from pa3common.kaggle import pip_install
import torch
torch_before = torch.__version__
pip_install(["transformers==5.21.0", "safetensors", "matplotlib", "pytest", "pyyaml"])
out = subprocess.run([sys.executable, "-c", "import torch, transformers; print(torch.__version__, transformers.__version__)"],
                     capture_output=True, text=True)
print("kernel torch:", torch_before, "| subprocess torch/transformers:", out.stdout.strip(), out.stderr[-400:])
assert out.stdout.split()[0] == torch_before, "pip changed torch — restart the kernel and rerun"
gpus = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print("GPUs:", gpus)
if len(gpus) < 2:
    print("!! WARNING: fewer than 2 GPUs visible — the two runs will execute one after the other")

In [ ]:
# ============================ RUN ============================
os.chdir(f"{CODE_DIR}/task_2")
from pa3vlm import orchestrate as O
info = O.setup_env(OUTPUT_ROOT, budget_hours=BUDGET_HOURS)
result = O.stage_auto(run_cpu_tests=RUN_CPU_TESTS, run_49=RUN_49_TOKENS)
print(result)

In [ ]:
# ============================ RESULTS ============================
import json
from pathlib import Path
from IPython.display import Markdown, Image, display
R = Path(OUTPUT_ROOT)
if (R / "status.json").exists():
    print(json.dumps(json.loads((R / "status.json").read_text()), indent=1))
T, F = R / "report_assets/tables", R / "report_assets/figures"
for t in sorted(T.glob("*.md")) if T.exists() else []:
    display(Markdown(f"**{t.name}**\n\n" + t.read_text()))
for f in sorted(F.glob("*.png")) if F.exists() else []:
    display(Image(filename=str(f)))